In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q -U huggingface_hub safetensors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 771.9/771.9 kB 13.4 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import snapshot_download
import os

MODEL_ID = "microsoft/bitnet-b1.58-2B-4T-bf16"

SAVE_DIR = "/content/drive/MyDrive/BitNetModels/bitnet-b1.58-2B-4T-bf16"

os.makedirs(SAVE_DIR, exist_ok=True)

model_path = snapshot_download(
    repo_id=MODEL_ID,
    local_dir=SAVE_DIR,
)

print("Model downloaded successfully!")
print("Saved at:", model_path)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Model downloaded successfully!
Saved at: /content/drive/MyDrive/BitNetModels/bitnet-b1.58-2B-4T-bf16


In [ ]:
import os

for root, dirs, files in os.walk(SAVE_DIR):
    for file in files:
        path = os.path.join(root, file)
        size_gb = os.path.getsize(path) / (1024 ** 3)

        print(
            file,
            f"{size_gb:.3f} GB"
        )

data_summary_card.md 0.000 GB
config.json 0.000 GB
model.safetensors 4.494 GB
special_tokens_map.json 0.000 GB
.gitattributes 0.000 GB
generation_config.json 0.000 GB
LICENSE 0.000 GB
README.md 0.000 GB
tokenizer.json 0.008 GB
tokenizer_config.json 0.000 GB
CACHEDIR.TAG 0.000 GB
.gitignore 0.000 GB
276681394656abdadb8e80e5b2c3db5e5d7fcaff.json 0.000 GB
data_summary_card.md.lock 0.000 GB
config.json.lock 0.000 GB
model.safetensors.lock 0.000 GB
README.md.lock 0.000 GB
special_tokens_map.json.lock 0.000 GB
generation_config.json.lock 0.000 GB
LICENSE.lock 0.000 GB
.gitattributes.lock 0.000 GB
data_summary_card.md.metadata 0.000 GB
tokenizer.json.lock 0.000 GB
config.json.metadata 0.000 GB
tokenizer_config.json.lock 0.000 GB
special_tokens_map.json.metadata 0.000 GB
generation_config.json.metadata 0.000 GB
.gitattributes.metadata 0.000 GB
README.md.metadata 0.000 GB
LICENSE.metadata 0.000 GB
tokenizer.json.metadata 0.000 GB
tokenizer_config.json.metadata 0.000 GB
model.safetensors.metadat

In [ ]:
from pathlib import Path

safetensor_files = sorted(
    Path(SAVE_DIR).glob("*.safetensors")
)

print(
    "Number of safetensors files:",
    len(safetensor_files)
)

for file in safetensor_files:
    size_gb = file.stat().st_size / (1024 ** 3)

    print(
        file.name,
        f"{size_gb:.3f} GB"
    )

Number of safetensors files: 1
model.safetensors 4.494 GB


In [ ]:
from safetensors import safe_open
import os

MODEL_PATH = "/content/drive/MyDrive/BitNetModels/bitnet-b1.58-2B-4T-bf16/model.safetensors"

total_params = 0
tensor_info = []

with safe_open(MODEL_PATH, framework="pt", device="cpu") as f:
    keys = list(f.keys())

    print("Total tensors:", len(keys))
    print("=" * 80)

    for name in keys:
        tensor = f.get_tensor(name)

        params = tensor.numel()
        bytes_size = params * tensor.element_size()

        total_params += params

        tensor_info.append({
            "name": name,
            "shape": tuple(tensor.shape),
            "dtype": str(tensor.dtype),
            "parameters": params,
            "size_mb": bytes_size / (1024**2)
        })

        del tensor

print(f"\nTotal parameters: {total_params:,}")
print(f"Total parameters (billions): {total_params / 1e9:.3f} B")
print(f"File size: {os.path.getsize(MODEL_PATH) / 1024**3:.3f} GB")


Total tensors: 332

Total parameters: 2,412,820,480
Total parameters (billions): 2.413 B
File size: 4.494 GB


In [ ]:
import pandas as pd

df = pd.DataFrame(tensor_info)

display(df.head(30))

,name,shape,dtype,parameters,size_mb
0,model.embed_tokens.weight,"(128256, 2560)",torch.bfloat16,328335360,626.250000
1,model.layers.0.input_layernorm.weight,"(2560,)",torch.bfloat16,2560,0.004883
2,model.layers.0.mlp.down_proj.weight,"(2560, 6912)",torch.bfloat16,17694720,33.750000
3,model.layers.0.mlp.ffn_sub_norm.weight,"(6912,)",torch.bfloat16,6912,0.013184
4,model.layers.0.mlp.gate_proj.weight,"(6912, 2560)",torch.bfloat16,17694720,33.750000
5,model.layers.0.mlp.up_proj.weight,"(6912, 2560)",torch.bfloat16,17694720,33.750000
6,model.layers.0.post_attention_layernorm.weight,"(2560,)",torch.bfloat16,2560,0.004883
7,model.layers.0.self_attn.attn_sub_norm.weight,"(2560,)",torch.bfloat16,2560,0.004883
8,model.layers.0.self_attn.k_proj.weight,"(640, 2560)",torch.bfloat16,1638400,3.125000
9,model.layers.0.self_attn.o_proj.weight,"(2560, 2560)",torch.bfloat16,6553600,12.500000


In [ ]:
linear_weights = df[
    (df["name"].str.contains("weight")) &
    (df["shape"].apply(lambda x: len(x) == 2))
].copy()

print("Number of 2D weight tensors:", len(linear_weights))
print(
    "Parameters in 2D weight tensors:",
    f"{linear_weights['parameters'].sum():,}"
)
print(
    "Parameters in 2D weight tensors (B):",
    f"{linear_weights['parameters'].sum() / 1e9:.3f}"
)

display(linear_weights.head(30))

Number of 2D weight tensors: 211
Parameters in 2D weight tensors: 2,412,380,160
Parameters in 2D weight tensors (B): 2.412


,name,shape,dtype,parameters,size_mb
0,model.embed_tokens.weight,"(128256, 2560)",torch.bfloat16,328335360,626.250
2,model.layers.0.mlp.down_proj.weight,"(2560, 6912)",torch.bfloat16,17694720,33.750
4,model.layers.0.mlp.gate_proj.weight,"(6912, 2560)",torch.bfloat16,17694720,33.750
5,model.layers.0.mlp.up_proj.weight,"(6912, 2560)",torch.bfloat16,17694720,33.750
8,model.layers.0.self_attn.k_proj.weight,"(640, 2560)",torch.bfloat16,1638400,3.125
9,model.layers.0.self_attn.o_proj.weight,"(2560, 2560)",torch.bfloat16,6553600,12.500
10,model.layers.0.self_attn.q_proj.weight,"(2560, 2560)",torch.bfloat16,6553600,12.500
11,model.layers.0.self_attn.v_proj.weight,"(640, 2560)",torch.bfloat16,1638400,3.125
13,model.layers.1.mlp.down_proj.weight,"(2560, 6912)",torch.bfloat16,17694720,33.750
15,model.layers.1.mlp.gate_proj.weight,"(6912, 2560)",torch.bfloat16,17694720,33.750


In [ ]:
from safetensors import safe_open
import pandas as pd

MODEL_PATH = "/content/drive/MyDrive/BitNetModels/bitnet-b1.58-2B-4T-bf16/model.safetensors"

TARGET_MODULES = (
    "q_proj.weight",
    "k_proj.weight",
    "v_proj.weight",
    "o_proj.weight",
    "gate_proj.weight",
    "up_proj.weight",
    "down_proj.weight",
)

target_info = []

with safe_open(MODEL_PATH, framework="pt", device="cpu") as f:

    for name in f.keys():

        if not name.endswith(TARGET_MODULES):
            continue

        tensor = f.get_tensor(name)

        target_info.append({
            "name": name,
            "shape": tuple(tensor.shape),
            "parameters": tensor.numel(),
            "dtype": str(tensor.dtype),
            "bf16_bytes": tensor.numel() * tensor.element_size()
        })

        del tensor


target_df = pd.DataFrame(target_info)

display(target_df.head(20))

total_target_params = target_df["parameters"].sum()
total_bf16_bytes = target_df["bf16_bytes"].sum()

print("====================================")
print("TARGET TENSOR SUMMARY")
print("====================================")

print(
    "Number of target tensors:",
    len(target_df)
)

print(
    "Target parameters:",
    f"{total_target_params:,}"
)

print(
    "Target parameters (B):",
    f"{total_target_params / 1e9:.3f}"
)

print(
    "Target BF16 size (GB):",
    f"{total_bf16_bytes / 1024**3:.3f}"
)

print(
    "Fraction of total model:",
    f"{100 * total_target_params / 2_412_820_480:.2f}%"
)

,name,shape,parameters,dtype,bf16_bytes
0,model.layers.0.mlp.down_proj.weight,"(2560, 6912)",17694720,torch.bfloat16,35389440
1,model.layers.0.mlp.gate_proj.weight,"(6912, 2560)",17694720,torch.bfloat16,35389440
2,model.layers.0.mlp.up_proj.weight,"(6912, 2560)",17694720,torch.bfloat16,35389440
3,model.layers.0.self_attn.k_proj.weight,"(640, 2560)",1638400,torch.bfloat16,3276800
4,model.layers.0.self_attn.o_proj.weight,"(2560, 2560)",6553600,torch.bfloat16,13107200
5,model.layers.0.self_attn.q_proj.weight,"(2560, 2560)",6553600,torch.bfloat16,13107200
6,model.layers.0.self_attn.v_proj.weight,"(640, 2560)",1638400,torch.bfloat16,3276800
7,model.layers.1.mlp.down_proj.weight,"(2560, 6912)",17694720,torch.bfloat16,35389440
8,model.layers.1.mlp.gate_proj.weight,"(6912, 2560)",17694720,torch.bfloat16,35389440
9,model.layers.1.mlp.up_proj.weight,"(6912, 2560)",17694720,torch.bfloat16,35389440


TARGET TENSOR SUMMARY
Number of target tensors: 210
Target parameters: 2,084,044,800
Target parameters (B): 2.084
Target BF16 size (GB): 3.882
Fraction of total model: 86.37%


In [ ]:
import torch
import numpy as np
from safetensors import safe_open

TEST_LAYER = "model.layers.0.self_attn.q_proj.weight"

with safe_open(
    MODEL_PATH,
    framework="pt",
    device="cpu"
) as f:

    weight = f.get_tensor(TEST_LAYER).float()


print("Layer:", TEST_LAYER)
print("Shape:", weight.shape)
print("Parameters:", weight.numel())

print("\nOriginal statistics:")
print("Min:", weight.min().item())
print("Max:", weight.max().item())
print("Mean:", weight.mean().item())
print("Std:", weight.std().item())

Layer: model.layers.0.self_attn.q_proj.weight
Shape: torch.Size([2560, 2560])
Parameters: 6553600

Original statistics:
Min: -39.75
Max: 37.0
Mean: 0.0013550525764003396
Std: 2.073319435119629


In [ ]:
def absmean_ternarize(weight):

    # Mean absolute magnitude
    scale = weight.abs().mean()

    # Scale weights
    scaled = weight / (scale + 1e-8)

    # Round and clamp to ternary values
    ternary = torch.round(scaled).clamp(-1, 1)

    return ternary.to(torch.int8), scale


ternary, scale = absmean_ternarize(weight)

print("\nAbsmean scale:")
print(scale.item())

print("\nUnique ternary values:")
print(torch.unique(ternary))

values, counts = torch.unique(
    ternary,
    return_counts=True
)

print("\nDistribution:")

for value, count in zip(values, counts):

    percentage = (
        count.item()
        / ternary.numel()
        * 100
    )

    print(
        f"{value.item():2d}: "
        f"{count.item():,} "
        f"({percentage:.2f}%)"
    )


Absmean scale:
1.2188506126403809

Unique ternary values:
tensor([-1,  0,  1], dtype=torch.int8)

Distribution:
-1: 1,609,634 (24.56%)
 0: 3,331,434 (50.83%)
 1: 1,612,532 (24.61%)


In [ ]:
def pack_ternary_weights(weights):

    weights = weights.cpu().numpy().astype(np.int8)

    rows, original_cols = weights.shape

    padded_cols = (
        (original_cols + 31) // 32
    ) * 32

    if padded_cols != original_cols:

        padded = np.zeros(
            (rows, padded_cols),
            dtype=np.int8
        )

        padded[:, :original_cols] = weights

        weights = padded


    words_per_row = padded_cols // 32

    pos = np.zeros(
        (rows, words_per_row),
        dtype=np.uint32
    )

    neg = np.zeros(
        (rows, words_per_row),
        dtype=np.uint32
    )


    for bit in range(32):

        indices = np.arange(
            bit,
            padded_cols,
            32
        )

        values = weights[:, indices]

        pos |= (
            (values == 1).astype(np.uint32)
            << bit
        )

        neg |= (
            (values == -1).astype(np.uint32)
            << bit
        )


    return (
        pos,
        neg,
        original_cols
    )


pos, neg, original_cols = \
    pack_ternary_weights(ternary)


print("Positive plane shape:", pos.shape)
print("Negative plane shape:", neg.shape)

print(
    "Packed bytes:",
    pos.nbytes + neg.nbytes
)

print(
    "Packed MB:",
    (pos.nbytes + neg.nbytes)
    / 1024**2
)

Positive plane shape: (2560, 80)
Negative plane shape: (2560, 80)
Packed bytes: 1638400
Packed MB: 1.5625


In [ ]:
def unpack_ternary_weights(
    pos,
    neg,
    original_cols
):

    rows, words = pos.shape

    padded_cols = words * 32

    output = np.zeros(
        (rows, padded_cols),
        dtype=np.int8
    )


    for bit in range(32):

        indices = np.arange(
            bit,
            padded_cols,
            32
        )

        positive = (
            (pos >> bit) & 1
        ).astype(np.int8)

        negative = (
            (neg >> bit) & 1
        ).astype(np.int8)

        output[:, indices] = (
            positive - negative
        )


    return output[:, :original_cols]


reconstructed = unpack_ternary_weights(
    pos,
    neg,
    original_cols
)


original = ternary.cpu().numpy()


mismatches = np.count_nonzero(
    reconstructed != original
)


print("Mismatches:", mismatches)


if mismatches == 0:

    print(
        "PASS: 100% round-trip integrity"
    )

else:

    print(
        "FAIL: Packing implementation error"
    )

Mismatches: 0
PASS: 100% round-trip integrity


In [ ]:
import os
import gc
import struct
import numpy as np
import pandas as pd
import torch

from safetensors import safe_open


MODEL_PATH = (
    "/content/drive/MyDrive/BitNetModels/"
    "bitnet-b1.58-2B-4T-bf16/model.safetensors"
)

OUTPUT_DIR = (
    "/content/drive/MyDrive/"
    "BitNetExperiments"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)


TARGET_MODULES = (
    "q_proj.weight",
    "k_proj.weight",
    "v_proj.weight",
    "o_proj.weight",
    "gate_proj.weight",
    "up_proj.weight",
    "down_proj.weight",
)


# Three representative real layers
EXPORT_LAYERS = {

    "model.layers.0.self_attn.k_proj.weight":
        "k_proj_640x2560.bin",

    "model.layers.0.self_attn.q_proj.weight":
        "q_proj_2560x2560.bin",

    "model.layers.0.mlp.gate_proj.weight":
        "gate_proj_6912x2560.bin",
}

In [ ]:
def absmean_ternarize(weight):

    weight = weight.float()

    scale = weight.abs().mean()

    ternary = torch.round(
        weight / (scale + 1e-8)
    ).clamp(-1, 1)

    return (
        ternary.to(torch.int8),
        scale.item()
    )

In [ ]:
def pack_ternary_weights(ternary):

    weights = (
        ternary.cpu()
        .numpy()
        .astype(np.int8)
    )

    rows, original_cols = weights.shape

    padded_cols = (
        (original_cols + 31) // 32
    ) * 32

    if padded_cols != original_cols:

        padded = np.zeros(
            (rows, padded_cols),
            dtype=np.int8
        )

        padded[:, :original_cols] = weights

        weights = padded


    words_per_row = padded_cols // 32


    pos = np.zeros(
        (rows, words_per_row),
        dtype=np.uint32
    )

    neg = np.zeros(
        (rows, words_per_row),
        dtype=np.uint32
    )


    for bit in range(32):

        indices = np.arange(
            bit,
            padded_cols,
            32
        )

        values = weights[:, indices]

        pos |= (
            (values == 1)
            .astype(np.uint32)
            << bit
        )

        neg |= (
            (values == -1)
            .astype(np.uint32)
            << bit
        )


    return (
        pos,
        neg,
        original_cols
    )

In [ ]:
def verify_packing(
    ternary,
    pos,
    neg,
    original_cols
):

    original = (
        ternary.cpu()
        .numpy()
        .astype(np.int8)
    )

    rows, words = pos.shape

    padded_cols = words * 32


    reconstructed = np.zeros(
        (rows, padded_cols),
        dtype=np.int8
    )


    for bit in range(32):

        indices = np.arange(
            bit,
            padded_cols,
            32
        )

        p = (
            (pos >> bit) & 1
        ).astype(np.int8)

        n = (
            (neg >> bit) & 1
        ).astype(np.int8)

        reconstructed[:, indices] = (
            p - n
        )


    reconstructed = (
        reconstructed[:, :original_cols]
    )


    mismatches = np.count_nonzero(
        reconstructed != original
    )


    return mismatches

In [ ]:
def export_layer(
    filename,
    pos,
    neg,
    original_cols
):

    rows, words = pos.shape

    with open(filename, "wb") as f:

        # Header:
        # rows
        # words_per_row
        # original_cols

        f.write(
            struct.pack(
                "<III",
                rows,
                words,
                original_cols
            )
        )

        f.write(
            pos.astype("<u4").tobytes()
        )

        f.write(
            neg.astype("<u4").tobytes()
        )

In [ ]:
results = []


with safe_open(
    MODEL_PATH,
    framework="pt",
    device="cpu"
) as f:

    target_names = [

        name

        for name in f.keys()

        if name.endswith(
            TARGET_MODULES
        )
    ]


    print(
        "Target tensors:",
        len(target_names)
    )


    for index, name in enumerate(
        target_names,
        start=1
    ):

        print(
            f"[{index}/{len(target_names)}]",
            name
        )


        # --------------------
        # Load BF16 tensor
        # --------------------

        weight = f.get_tensor(name)

        rows, cols = weight.shape

        params = weight.numel()

        bf16_bytes = (
            params *
            weight.element_size()
        )


        # --------------------
        # Absmean ternarization
        # --------------------

        ternary, scale = (
            absmean_ternarize(
                weight
            )
        )


        # --------------------
        # Distribution
        # --------------------

        neg_count = (
            ternary == -1
        ).sum().item()

        zero_count = (
            ternary == 0
        ).sum().item()

        pos_count = (
            ternary == 1
        ).sum().item()


        # --------------------
        # Pack
        # --------------------

        pos, neg, original_cols = (
            pack_ternary_weights(
                ternary
            )
        )


        packed_bytes = (
            pos.nbytes +
            neg.nbytes
        )


        # --------------------
        # Verify
        # --------------------

        mismatches = (
            verify_packing(
                ternary,
                pos,
                neg,
                original_cols
            )
        )


        # --------------------
        # Save benchmark layers
        # --------------------

        if name in EXPORT_LAYERS:

            output_path = os.path.join(
                OUTPUT_DIR,
                EXPORT_LAYERS[name]
            )

            export_layer(
                output_path,
                pos,
                neg,
                original_cols
            )

            print(
                "   Exported:",
                output_path
            )


        # --------------------
        # Record statistics
        # --------------------

        results.append({

            "layer":
                name,

            "rows":
                rows,

            "cols":
                cols,

            "parameters":
                params,

            "absmean_scale":
                scale,

            "negative_percent":
                100 * neg_count / params,

            "zero_percent":
                100 * zero_count / params,

            "positive_percent":
                100 * pos_count / params,

            "bf16_bytes":
                bf16_bytes,

            # Hypothetical dense INT8
            # baseline representation
            "int8_bytes":
                params,

            "packed_2bit_bytes":
                packed_bytes,

            "bf16_to_2bit_ratio":
                bf16_bytes /
                packed_bytes,

            "int8_to_2bit_ratio":
                params /
                packed_bytes,

            "mismatches":
                mismatches,

            "verified":
                mismatches == 0,
        })


        # --------------------
        # Free memory
        # --------------------

        del weight
        del ternary
        del pos
        del neg

        gc.collect()

Target tensors: 210
[1/210] model.layers.0.mlp.down_proj.weight
[2/210] model.layers.0.mlp.gate_proj.weight
   Exported: /content/drive/MyDrive/BitNetExperiments/gate_proj_6912x2560.bin
[3/210] model.layers.0.mlp.up_proj.weight
[4/210] model.layers.0.self_attn.k_proj.weight
   Exported: /content/drive/MyDrive/BitNetExperiments/k_proj_640x2560.bin
[5/210] model.layers.0.self_attn.o_proj.weight
[6/210] model.layers.0.self_attn.q_proj.weight
   Exported: /content/drive/MyDrive/BitNetExperiments/q_proj_2560x2560.bin
[7/210] model.layers.0.self_attn.v_proj.weight
[8/210] model.layers.1.mlp.down_proj.weight
[9/210] model.layers.1.mlp.gate_proj.weight
[10/210] model.layers.1.mlp.up_proj.weight
[11/210] model.layers.1.self_attn.k_proj.weight
[12/210] model.layers.1.self_attn.o_proj.weight
[13/210] model.layers.1.self_attn.q_proj.weight
[14/210] model.layers.1.self_attn.v_proj.weight
[15/210] model.layers.10.mlp.down_proj.weight
[16/210] model.layers.10.mlp.gate_proj.weight
[17/210] model.layer

In [ ]:
results_df = pd.DataFrame(
    results
)

CSV_PATH = os.path.join(
    OUTPUT_DIR,
    "whole_target_compression_results.csv"
)

results_df.to_csv(
    CSV_PATH,
    index=False
)

print(
    "Saved results:",
    CSV_PATH
)

Saved results: /content/drive/MyDrive/BitNetExperiments/whole_target_compression_results.csv


In [ ]:
total_params = (
    results_df[
        "parameters"
    ].sum()
)

total_bf16 = (
    results_df[
        "bf16_bytes"
    ].sum()
)

total_int8 = (
    results_df[
        "int8_bytes"
    ].sum()
)

total_packed = (
    results_df[
        "packed_2bit_bytes"
    ].sum()
)


weighted_neg = (
    (
        results_df[
            "negative_percent"
        ]
        *
        results_df[
            "parameters"
        ]
    ).sum()
    /
    total_params
)

weighted_zero = (
    (
        results_df[
            "zero_percent"
        ]
        *
        results_df[
            "parameters"
        ]
    ).sum()
    /
    total_params
)

weighted_pos = (
    (
        results_df[
            "positive_percent"
        ]
        *
        results_df[
            "parameters"
        ]
    ).sum()
    /
    total_params
)


print(
    "======================================"
)

print(
    "WHOLE TARGET EXPERIMENT SUMMARY"
)

print(
    "======================================"
)


print(
    "Tensors processed:",
    len(results_df)
)

print(
    "Parameters:",
    f"{total_params:,}"
)

print(
    "Parameters (B):",
    f"{total_params / 1e9:.3f}"
)


print(
    "\nBF16 size (GiB):",
    f"{total_bf16 / 1024**3:.3f}"
)

print(
    "Dense INT8 size (GiB):",
    f"{total_int8 / 1024**3:.3f}"
)

print(
    "Dual-plane 2-bit size (GiB):",
    f"{total_packed / 1024**3:.3f}"
)


print(
    "\nBF16 -> 2-bit ratio:",
    f"{total_bf16 / total_packed:.2f}x"
)

print(
    "INT8 -> 2-bit ratio:",
    f"{total_int8 / total_packed:.2f}x"
)


print(
    "\nOverall ternary distribution:"
)

print(
    "-1:",
    f"{weighted_neg:.2f}%"
)

print(
    " 0:",
    f"{weighted_zero:.2f}%"
)

print(
    "+1:",
    f"{weighted_pos:.2f}%"
)


print(
    "\nAll packing verified:",
    results_df[
        "verified"
    ].all()
)

print(
    "Total mismatches:",
    results_df[
        "mismatches"
    ].sum()
)

WHOLE TARGET EXPERIMENT SUMMARY
Tensors processed: 210
Parameters: 2,084,044,800
Parameters (B): 2.084

BF16 size (GiB): 3.882
Dense INT8 size (GiB): 1.941
Dual-plane 2-bit size (GiB): 0.485

BF16 -> 2-bit ratio: 8.00x
INT8 -> 2-bit ratio: 4.00x

Overall ternary distribution:
-1: 28.90%
 0: 42.19%
+1: 28.91%

All packing verified: True
Total mismatches: 0


In [ ]:
import os
import struct

OUTPUT_DIR = "/content/drive/MyDrive/BitNetExperiments"

EXPECTED = {
    "k_proj_640x2560.bin": {
        "rows": 640,
        "cols": 2560,
    },
    "q_proj_2560x2560.bin": {
        "rows": 2560,
        "cols": 2560,
    },
    "gate_proj_6912x2560.bin": {
        "rows": 6912,
        "cols": 2560,
    },
}

all_valid = True

for filename, expected in EXPECTED.items():

    path = os.path.join(OUTPUT_DIR, filename)

    print("=" * 60)
    print("FILE:", filename)

    if not os.path.exists(path):
        print("FAIL: File does not exist")
        all_valid = False
        continue

    actual_size = os.path.getsize(path)

    with open(path, "rb") as f:
        header = f.read(12)

        if len(header) != 12:
            print("FAIL: Invalid/incomplete header")
            all_valid = False
            continue

        rows, words_per_row, original_cols = struct.unpack(
            "<III",
            header
        )

    expected_rows = expected["rows"]
    expected_cols = expected["cols"]

    expected_words = (
        expected_cols + 31
    ) // 32

    # Header = 12 bytes
    #
    # Positive plane:
    # rows * words * 4 bytes
    #
    # Negative plane:
    # rows * words * 4 bytes

    expected_size = (
        12
        + expected_rows
        * expected_words
        * 4
        * 2
    )

    print("Header:")
    print("  rows           :", rows)
    print("  words_per_row  :", words_per_row)
    print("  original_cols  :", original_cols)

    print("\nExpected:")
    print("  rows           :", expected_rows)
    print("  words_per_row  :", expected_words)
    print("  original_cols  :", expected_cols)

    print("\nFile size:")
    print("  actual         :", actual_size, "bytes")
    print("  expected       :", expected_size, "bytes")

    header_ok = (
        rows == expected_rows
        and words_per_row == expected_words
        and original_cols == expected_cols
    )

    size_ok = (
        actual_size == expected_size
    )

    if header_ok and size_ok:
        print("\nPASS")
    else:
        print("\nFAIL")
        all_valid = False


print("\n" + "=" * 60)

if all_valid:
    print("ALL THREE BENCHMARK BINARIES VERIFIED")
else:
    print("ONE OR MORE BINARIES FAILED VERIFICATION")

FILE: k_proj_640x2560.bin
Header:
  rows           : 640
  words_per_row  : 80
  original_cols  : 2560

Expected:
  rows           : 640
  words_per_row  : 80
  original_cols  : 2560

File size:
  actual         : 409612 bytes
  expected       : 409612 bytes

PASS
FILE: q_proj_2560x2560.bin
Header:
  rows           : 2560
  words_per_row  : 80
  original_cols  : 2560

Expected:
  rows           : 2560
  words_per_row  : 80
  original_cols  : 2560

File size:
  actual         : 1638412 bytes
  expected       : 1638412 bytes

PASS
FILE: gate_proj_6912x2560.bin
Header:
  rows           : 6912
  words_per_row  : 80
  original_cols  : 2560

Expected:
  rows           : 6912
  words_per_row  : 80
  original_cols  : 2560

File size:
  actual         : 4423692 bytes
  expected       : 4423692 bytes

PASS

ALL THREE BENCHMARK BINARIES VERIFIED


In [ ]:
%%writefile /content/benchmark_final.cpp

#include <algorithm>
#include <chrono>
#include <cmath>
#include <cstdint>
#include <fstream>
#include <iomanip>
#include <iostream>
#include <random>
#include <stdexcept>
#include <string>
#include <vector>

#if defined(_MSC_VER)
#include <intrin.h>
#define POPCOUNT32 __popcnt
#else
#define POPCOUNT32 __builtin_popcount
#endif

// Prevent benchmark work from being removed.
volatile int64_t benchmark_sink = 0;

struct PackedLayer {
    uint32_t rows;
    uint32_t words_per_row;
    uint32_t original_cols;

    std::vector<uint32_t> pos;
    std::vector<uint32_t> neg;
};

// ------------------------------------------------------------
// Load packed binary
// ------------------------------------------------------------

PackedLayer load_layer(const std::string& filename) {

    std::ifstream file(filename, std::ios::binary);

    if (!file) {
        throw std::runtime_error(
            "Could not open: " + filename
        );
    }

    PackedLayer layer;

    file.read(
        reinterpret_cast<char*>(&layer.rows),
        sizeof(uint32_t)
    );

    file.read(
        reinterpret_cast<char*>(&layer.words_per_row),
        sizeof(uint32_t)
    );

    file.read(
        reinterpret_cast<char*>(&layer.original_cols),
        sizeof(uint32_t)
    );

    size_t total_words =
        static_cast<size_t>(layer.rows)
        * layer.words_per_row;

    layer.pos.resize(total_words);
    layer.neg.resize(total_words);

    file.read(
        reinterpret_cast<char*>(layer.pos.data()),
        total_words * sizeof(uint32_t)
    );

    file.read(
        reinterpret_cast<char*>(layer.neg.data()),
        total_words * sizeof(uint32_t)
    );

    if (!file) {
        throw std::runtime_error(
            "Error reading packed data."
        );
    }

    return layer;
}


// ------------------------------------------------------------
// Reconstruct dense INT8 weights
//
// IMPORTANT:
// This is performed ONCE before timing.
// Baseline A therefore represents a model whose ternary
// weights are already stored densely as INT8.
// ------------------------------------------------------------

std::vector<int8_t> reconstruct_dense_weights(
    const PackedLayer& layer
) {

    size_t total_elements =
        static_cast<size_t>(layer.rows)
        * layer.original_cols;

    std::vector<int8_t> dense(total_elements);

    for (uint32_t r = 0; r < layer.rows; ++r) {

        size_t packed_offset =
            static_cast<size_t>(r)
            * layer.words_per_row;

        size_t dense_offset =
            static_cast<size_t>(r)
            * layer.original_cols;

        for (uint32_t c = 0;
             c < layer.original_cols;
             ++c) {

            uint32_t word = c / 32;
            uint32_t bit  = c % 32;

            uint32_t p =
                layer.pos[
                    packed_offset + word
                ];

            uint32_t n =
                layer.neg[
                    packed_offset + word
                ];

            dense[dense_offset + c] =
                static_cast<int8_t>(
                    ((p >> bit) & 1U)
                    -
                    ((n >> bit) & 1U)
                );
        }
    }

    return dense;
}


// ------------------------------------------------------------
// BASELINE A
//
// Honest dense INT8 ternary-weight dot product.
// No packed-weight decoding inside the timed kernel.
// ------------------------------------------------------------

int64_t baseline_a_dense(
    const std::vector<int8_t>& weights,
    const std::vector<int8_t>& activations,
    uint32_t rows,
    uint32_t cols
) {

    int64_t checksum = 0;

    for (uint32_t r = 0; r < rows; ++r) {

        int32_t acc = 0;

        size_t offset =
            static_cast<size_t>(r) * cols;

        for (uint32_t c = 0; c < cols; ++c) {

            acc +=
                static_cast<int32_t>(
                    weights[offset + c]
                )
                *
                static_cast<int32_t>(
                    activations[c]
                );
        }

        checksum += acc;
    }

    return checksum;
}


// ------------------------------------------------------------
// BASELINE B
//
// Packed weights are decoded bit-by-bit inside the kernel,
// then conventional arithmetic is performed.
//
// This is the "unpacking tax" ablation.
// ------------------------------------------------------------

int64_t baseline_b_unpack(
    const PackedLayer& layer,
    const std::vector<int8_t>& activations
) {

    int64_t checksum = 0;

    for (uint32_t r = 0;
         r < layer.rows;
         ++r) {

        int32_t acc = 0;

        size_t offset =
            static_cast<size_t>(r)
            * layer.words_per_row;

        for (uint32_t c = 0;
             c < layer.original_cols;
             ++c) {

            uint32_t word = c / 32;
            uint32_t bit  = c % 32;

            uint32_t p =
                layer.pos[offset + word];

            uint32_t n =
                layer.neg[offset + word];

            int32_t weight =
                static_cast<int32_t>(
                    (p >> bit) & 1U
                )
                -
                static_cast<int32_t>(
                    (n >> bit) & 1U
                );

            acc +=
                weight
                *
                static_cast<int32_t>(
                    activations[c]
                );
        }

        checksum += acc;
    }

    return checksum;
}


// ------------------------------------------------------------
// Activation packing
//
// Same INT8 activation vector used by A/B/C.
// Only Proposed requires this conversion.
// ------------------------------------------------------------

void pack_activations(
    const std::vector<int8_t>& activations,
    std::vector<uint32_t> planes[8],
    uint32_t words_per_row
) {

    for (int b = 0; b < 8; ++b) {
        planes[b].assign(
            words_per_row,
            0U
        );
    }

    for (uint32_t c = 0;
         c < activations.size();
         ++c) {

        uint8_t value =
            static_cast<uint8_t>(
                activations[c]
            );

        uint32_t word = c / 32;
        uint32_t bit  = c % 32;

        for (int b = 0; b < 8; ++b) {

            if ((value >> b) & 1U) {

                planes[b][word] |=
                    (1U << bit);
            }
        }
    }
}


// ------------------------------------------------------------
// PROPOSED
//
// Packed ternary weights + packed activation bitplanes.
// ------------------------------------------------------------

int64_t proposed_bitserial(
    const PackedLayer& layer,
    const std::vector<uint32_t> planes[8]
) {

    int64_t checksum = 0;

    for (uint32_t r = 0;
         r < layer.rows;
         ++r) {

        int32_t acc = 0;

        size_t offset =
            static_cast<size_t>(r)
            * layer.words_per_row;

        for (int b = 0; b < 8; ++b) {

            int32_t plane_acc = 0;

            int32_t scale =
                (b == 7)
                ? -128
                : (1 << b);

            for (uint32_t w = 0;
                 w < layer.words_per_row;
                 ++w) {

                uint32_t a =
                    planes[b][w];

                plane_acc +=
                    POPCOUNT32(
                        layer.pos[offset + w]
                        & a
                    );

                plane_acc -=
                    POPCOUNT32(
                        layer.neg[offset + w]
                        & a
                    );
            }

            acc += plane_acc * scale;
        }

        checksum += acc;
    }

    return checksum;
}


// ------------------------------------------------------------
// Timing helper
// ------------------------------------------------------------

template <typename Func>
double time_function_us(Func&& function) {

    auto start =
        std::chrono::steady_clock::now();

    int64_t result = function();

    auto end =
        std::chrono::steady_clock::now();

    benchmark_sink += result;

    return std::chrono::duration<
        double,
        std::micro
    >(end - start).count();
}


// ------------------------------------------------------------
// Main benchmark
// ------------------------------------------------------------

int main(
    int argc,
    char* argv[]
) {

    if (argc != 4) {

        std::cerr
            << "Usage:\n"
            << "./benchmark_final "
            << "<layer.bin> "
            << "<layer_name> "
            << "<output.csv>\n";

        return 1;
    }


    std::string binary_path = argv[1];
    std::string layer_name  = argv[2];
    std::string csv_path    = argv[3];


    PackedLayer layer =
        load_layer(binary_path);


    std::cout
        << "\n====================================\n"
        << "Layer: " << layer_name << "\n"
        << "Matrix: "
        << layer.rows
        << " x "
        << layer.original_cols
        << "\n"
        << "Words/row: "
        << layer.words_per_row
        << "\n"
        << "====================================\n";


    // --------------------------------------------------------
    // Dense representation for honest Baseline A
    // --------------------------------------------------------

    std::cout
        << "[1] Reconstructing dense INT8 weights...\n";

    std::vector<int8_t> dense_weights =
        reconstruct_dense_weights(layer);


    // --------------------------------------------------------
    // ONE shared activation vector
    // --------------------------------------------------------

    std::cout
        << "[2] Generating shared activation vector...\n";

    std::vector<int8_t> activations(
        layer.original_cols
    );

    std::mt19937 rng(42);

    std::uniform_int_distribution<int>
        distribution(-128, 127);

    for (auto& value : activations) {

        value =
            static_cast<int8_t>(
                distribution(rng)
            );
    }


    // --------------------------------------------------------
    // Pack activations for Proposed
    // --------------------------------------------------------

    std::vector<uint32_t>
        activation_planes[8];

    pack_activations(
        activations,
        activation_planes,
        layer.words_per_row
    );


    // --------------------------------------------------------
    // THREE-WAY CORRECTNESS CHECK
    // --------------------------------------------------------

    std::cout
        << "[3] Three-way correctness check...\n";

    int64_t output_a =
        baseline_a_dense(
            dense_weights,
            activations,
            layer.rows,
            layer.original_cols
        );

    int64_t output_b =
        baseline_b_unpack(
            layer,
            activations
        );

    int64_t output_c =
        proposed_bitserial(
            layer,
            activation_planes
        );


    std::cout
        << "    A = "
        << output_a
        << "\n";

    std::cout
        << "    B = "
        << output_b
        << "\n";

    std::cout
        << "    C = "
        << output_c
        << "\n";


    if (
        output_a != output_b
        ||
        output_a != output_c
    ) {

        std::cerr
            << "\nFAIL: Three-way correctness "
            << "check failed.\n";

        return 2;
    }


    std::cout
        << "    PASS: A == B == C\n";


    // --------------------------------------------------------
    // Warm-up
    // --------------------------------------------------------

    const int WARMUP = 5;

    std::cout
        << "[4] Warm-up: "
        << WARMUP
        << " rounds...\n";


    for (int i = 0;
         i < WARMUP;
         ++i) {

        benchmark_sink +=
            baseline_a_dense(
                dense_weights,
                activations,
                layer.rows,
                layer.original_cols
            );

        benchmark_sink +=
            baseline_b_unpack(
                layer,
                activations
            );

        benchmark_sink +=
            proposed_bitserial(
                layer,
                activation_planes
            );
    }


    // --------------------------------------------------------
    // CSV
    // --------------------------------------------------------

    bool csv_exists =
        std::ifstream(csv_path).good();


    std::ofstream csv(
        csv_path,
        std::ios::app
    );


    if (!csv_exists) {

        csv
            << "layer,"
            << "trial,"
            << "order,"
            << "baseline_a_us,"
            << "baseline_b_us,"
            << "activation_pack_us,"
            << "proposed_kernel_us\n";
    }


    // --------------------------------------------------------
    // Measured outer trials
    // --------------------------------------------------------

    const int TRIALS = 30;

    std::cout
        << "[5] Running "
        << TRIALS
        << " measured trials...\n";


    for (int trial = 0;
         trial < TRIALS;
         ++trial) {


        double time_a = 0.0;
        double time_b = 0.0;
        double time_c = 0.0;


        // ----------------------------------------------
        // Activation packing measured independently
        // ----------------------------------------------

        std::vector<uint32_t>
            trial_planes[8];


        auto pack_start =
            std::chrono::steady_clock::now();


        pack_activations(
            activations,
            trial_planes,
            layer.words_per_row
        );


        auto pack_end =
            std::chrono::steady_clock::now();


        double pack_time =
            std::chrono::duration<
                double,
                std::micro
            >(
                pack_end - pack_start
            ).count();


        // ----------------------------------------------
        // Rotate A/B/C execution order
        // ----------------------------------------------

        int order =
            trial % 3;


        std::string order_name;


        if (order == 0) {

            order_name = "A-B-C";

            time_a =
                time_function_us(
                    [&]() {
                        return baseline_a_dense(
                            dense_weights,
                            activations,
                            layer.rows,
                            layer.original_cols
                        );
                    }
                );

            time_b =
                time_function_us(
                    [&]() {
                        return baseline_b_unpack(
                            layer,
                            activations
                        );
                    }
                );

            time_c =
                time_function_us(
                    [&]() {
                        return proposed_bitserial(
                            layer,
                            trial_planes
                        );
                    }
                );
        }


        else if (order == 1) {

            order_name = "B-C-A";

            time_b =
                time_function_us(
                    [&]() {
                        return baseline_b_unpack(
                            layer,
                            activations
                        );
                    }
                );

            time_c =
                time_function_us(
                    [&]() {
                        return proposed_bitserial(
                            layer,
                            trial_planes
                        );
                    }
                );

            time_a =
                time_function_us(
                    [&]() {
                        return baseline_a_dense(
                            dense_weights,
                            activations,
                            layer.rows,
                            layer.original_cols
                        );
                    }
                );
        }


        else {

            order_name = "C-A-B";

            time_c =
                time_function_us(
                    [&]() {
                        return proposed_bitserial(
                            layer,
                            trial_planes
                        );
                    }
                );

            time_a =
                time_function_us(
                    [&]() {
                        return baseline_a_dense(
                            dense_weights,
                            activations,
                            layer.rows,
                            layer.original_cols
                        );
                    }
                );

            time_b =
                time_function_us(
                    [&]() {
                        return baseline_b_unpack(
                            layer,
                            activations
                        );
                    }
                );
        }


        csv
            << layer_name << ","
            << trial << ","
            << order_name << ","
            << std::fixed
            << std::setprecision(3)
            << time_a << ","
            << time_b << ","
            << pack_time << ","
            << time_c
            << "\n";


        std::cout
            << "Trial "
            << std::setw(2)
            << trial
            << " ["
            << order_name
            << "]"
            << " A="
            << time_a
            << " us"
            << " B="
            << time_b
            << " us"
            << " C="
            << time_c
            << " us"
            << " Pack="
            << pack_time
            << " us\n";
    }


    csv.close();


    std::cout
        << "\nBenchmark complete.\n";

    std::cout
        << "Checksum sink: "
        << benchmark_sink
        << "\n";

    std::cout
        << "Raw results: "
        << csv_path
        << "\n";


    return 0;
}

Writing /content/benchmark_final.cpp


In [ ]:
!g++ \
    -O3 \
    -march=native \
    -mpopcnt \
    -std=c++17 \
    /content/benchmark_final.cpp \
    -o /content/benchmark_final

In [ ]:
import os

CSV_PATH = "/content/benchmark_raw.csv"

if os.path.exists(CSV_PATH):
    os.remove(CSV_PATH)

print("Ready.")

Ready.


In [ ]:
!/content/benchmark_final \
"/content/drive/MyDrive/BitNetExperiments/k_proj_640x2560.bin" \
"k_proj_640x2560" \
"/content/benchmark_raw.csv"


Layer: k_proj_640x2560
Matrix: 640 x 2560
Words/row: 80
[1] Reconstructing dense INT8 weights...
[2] Generating shared activation vector...
[3] Three-way correctness check...
    A = -33730
    B = -33730
    C = -33730
    PASS: A == B == C
[4] Warm-up: 5 rounds...
[5] Running 30 measured trials...
Trial  0 [A-B-C] A=283.54 us B=3577.77 us C=496.41 us Pack=91.608 us
Trial  1 [B-C-A] A=314.36 us B=2439.01 us C=495.305 us Pack=90.789 us
Trial  2 [C-A-B] A=255.66 us B=3526.71 us C=498.862 us Pack=87.207 us
Trial  3 [A-B-C] A=247.681 us B=3741.03 us C=491.877 us Pack=85.798 us
Trial  4 [B-C-A] A=314.974 us B=2441.03 us C=485.077 us Pack=91.82 us
Trial  5 [C-A-B] A=247.583 us B=3619.05 us C=487.231 us Pack=97.449 us
Trial  6 [A-B-C] A=246.224 us B=3702.05 us C=527.959 us Pack=87.197 us
Trial  7 [B-C-A] A=309.314 us B=2448.04 us C=496.615 us Pack=91.812 us
Trial  8 [C-A-B] A=254.386 us B=3526.35 us C=508.244 us Pack=90.125 us
Trial  9 [A-B-C] A=252.801 us B=3597.68 us C=496.069 us Pack=100

In [ ]:
!/content/benchmark_final \
"/content/drive/MyDrive/BitNetExperiments/gate_proj_6912x2560.bin" \
"gate_proj_6912x2560" \
"/content/benchmark_raw.csv"


Layer: gate_proj_6912x2560
Matrix: 6912 x 2560
Words/row: 80
[1] Reconstructing dense INT8 weights...
[2] Generating shared activation vector...
[3] Three-way correctness check...
    A = -2994401
    B = -2994401
    C = -2994401
    PASS: A == B == C
[4] Warm-up: 5 rounds...
[5] Running 30 measured trials...
Trial  0 [A-B-C] A=3406.01 us B=71438.1 us C=8571.07 us Pack=102.71 us
Trial  1 [B-C-A] A=7403.03 us B=54477.9 us C=16230.1 us Pack=99.434 us
Trial  2 [C-A-B] A=7416.71 us B=150728 us C=14414.2 us Pack=101.892 us
Trial  3 [A-B-C] A=7502.19 us B=89507 us C=8594.39 us Pack=102.82 us
Trial  4 [B-C-A] A=11505.7 us B=101446 us C=24629.9 us Pack=102.169 us
Trial  5 [C-A-B] A=12341.1 us B=191488 us C=22616.4 us Pack=106.844 us
Trial  6 [A-B-C] A=3145.09 us B=39252.1 us C=5585.64 us Pack=91.15 us
Trial  7 [B-C-A] A=3124.66 us B=39974.8 us C=6300.22 us Pack=95.761 us
Trial  8 [C-A-B] A=3205.31 us B=39052 us C=5474.69 us Pack=92.426 us
Trial  9 [A-B-C] A=3156.3 us B=38835 us C=5464.46 us 

In [ ]:
!/content/benchmark_final \
"/content/drive/MyDrive/BitNetExperiments/q_proj_2560x2560.bin" \
"q_proj_2560x2560" \
"/content/benchmark_raw.csv"


Layer: q_proj_2560x2560
Matrix: 2560 x 2560
Words/row: 80
[1] Reconstructing dense INT8 weights...
[2] Generating shared activation vector...
[3] Three-way correctness check...
    A = -94125
    B = -94125
    C = -94125
    PASS: A == B == C
[4] Warm-up: 5 rounds...
[5] Running 30 measured trials...
Trial  0 [A-B-C] A=2195.66 us B=73002.6 us C=10284 us Pack=113.331 us
Trial  1 [B-C-A] A=1221.41 us B=45988.5 us C=19234.2 us Pack=107.429 us
Trial  2 [C-A-B] A=1182.16 us B=119741 us C=19211 us Pack=103.48 us
Trial  3 [A-B-C] A=9010.17 us B=126817 us C=17349 us Pack=102.698 us
Trial  4 [B-C-A] A=1271.48 us B=79110.7 us C=7033.31 us Pack=98.449 us
Trial  5 [C-A-B] A=1228.17 us B=71564.3 us C=8068.84 us Pack=103.975 us
Trial  6 [A-B-C] A=1258.72 us B=83171.4 us C=3507.52 us Pack=105.998 us
Trial  7 [B-C-A] A=1246.61 us B=18467.8 us C=3509 us Pack=105.499 us
Trial  8 [C-A-B] A=1238.06 us B=29673.5 us C=3325.55 us Pack=105.165 us
Trial  9 [A-B-C] A=1232.99 us B=23545.5 us C=3229.06 us Pack=

In [ ]:
import pandas as pd

df = pd.read_csv(
    "/content/benchmark_raw.csv"
)

print("Rows:", len(df))

display(df.head())

print("\nTrials per layer:")
print(
    df.groupby("layer")
      .size()
)

Rows: 90


,layer,trial,order,baseline_a_us,baseline_b_us,activation_pack_us,proposed_kernel_us
0,k_proj_640x2560,0,A-B-C,283.540,3577.772,91.608,496.410
1,k_proj_640x2560,1,B-C-A,314.360,2439.007,90.789,495.305
2,k_proj_640x2560,2,C-A-B,255.660,3526.707,87.207,498.862
3,k_proj_640x2560,3,A-B-C,247.681,3741.032,85.798,491.877
4,k_proj_640x2560,4,B-C-A,314.974,2441.033,91.820,485.077



Trials per layer:
layer
gate_proj_6912x2560    30
k_proj_640x2560        30
q_proj_2560x2560       30
dtype: int64


In [ ]:
import pandas as pd
import numpy as np

CSV_PATH = "/content/benchmark_raw.csv"

df = pd.read_csv(CSV_PATH)

# Proposed total = activation packing + bit-serial kernel
df["proposed_total_us"] = (
    df["activation_pack_us"]
    + df["proposed_kernel_us"]
)

# Speedup > 1 means Proposed is faster
# Speedup < 1 means Proposed is slower

df["kernel_speedup_vs_A"] = (
    df["baseline_a_us"]
    / df["proposed_kernel_us"]
)

df["total_speedup_vs_A"] = (
    df["baseline_a_us"]
    / df["proposed_total_us"]
)

df["kernel_speedup_vs_B"] = (
    df["baseline_b_us"]
    / df["proposed_kernel_us"]
)

df["total_speedup_vs_B"] = (
    df["baseline_b_us"]
    / df["proposed_total_us"]
)

print("Rows:", len(df))

display(df.head())

Rows: 90


,layer,trial,order,baseline_a_us,baseline_b_us,activation_pack_us,proposed_kernel_us,proposed_total_us,kernel_speedup_vs_A,total_speedup_vs_A,kernel_speedup_vs_B,total_speedup_vs_B
0,k_proj_640x2560,0,A-B-C,283.540,3577.772,91.608,496.410,588.018,0.571181,0.482196,7.207292,6.084460
1,k_proj_640x2560,1,B-C-A,314.360,2439.007,90.789,495.305,586.094,0.634680,0.536364,4.924253,4.161460
2,k_proj_640x2560,2,C-A-B,255.660,3526.707,87.207,498.862,586.069,0.512486,0.436228,7.069504,6.017563
3,k_proj_640x2560,3,A-B-C,247.681,3741.032,85.798,491.877,577.675,0.503543,0.428755,7.605625,6.476015
4,k_proj_640x2560,4,B-C-A,314.974,2441.033,91.820,485.077,576.897,0.649328,0.545980,5.032259,4.231315


In [ ]:
METRICS = [
    "baseline_a_us",
    "baseline_b_us",
    "activation_pack_us",
    "proposed_kernel_us",
    "proposed_total_us"
]

summary_rows = []

for layer, group in df.groupby("layer"):

    for metric in METRICS:

        values = group[metric]

        mean = values.mean()
        median = values.median()
        std = values.std(ddof=1)
        minimum = values.min()
        maximum = values.max()

        cv = (
            std / mean * 100
            if mean != 0
            else np.nan
        )

        summary_rows.append({
            "layer": layer,
            "metric": metric,
            "mean_us": mean,
            "median_us": median,
            "std_us": std,
            "cv_percent": cv,
            "min_us": minimum,
            "max_us": maximum
        })


summary = pd.DataFrame(summary_rows)

display(
    summary.round(3)
)

,layer,metric,mean_us,median_us,std_us,cv_percent,min_us,max_us
0,gate_proj_6912x2560,baseline_a_us,4232.242,3212.061,2455.926,58.029,3124.656,12341.077
1,gate_proj_6912x2560,baseline_b_us,50615.951,39059.386,37342.402,73.776,26452.753,191488.488
2,gate_proj_6912x2560,activation_pack_us,96.127,93.727,4.872,5.069,90.643,106.844
3,gate_proj_6912x2560,proposed_kernel_us,7623.152,5512.093,5047.679,66.215,5395.626,24629.931
4,gate_proj_6912x2560,proposed_total_us,7719.279,5607.097,5050.560,65.428,5498.286,24732.100
5,k_proj_640x2560,baseline_a_us,275.301,258.108,30.981,11.254,241.936,358.366
6,k_proj_640x2560,baseline_b_us,3368.090,3580.966,703.121,20.876,2431.482,5242.938
7,k_proj_640x2560,activation_pack_us,92.628,89.682,9.339,10.082,85.045,129.947
8,k_proj_640x2560,proposed_kernel_us,502.055,496.240,26.978,5.373,485.077,635.603
9,k_proj_640x2560,proposed_total_us,594.683,587.970,27.715,4.661,570.799,724.540


In [ ]:
speedup_rows = []

for layer, group in df.groupby("layer"):

    A_mean = group["baseline_a_us"].mean()
    B_mean = group["baseline_b_us"].mean()
    C_mean = group["proposed_kernel_us"].mean()
    T_mean = group["proposed_total_us"].mean()

    A_median = group["baseline_a_us"].median()
    B_median = group["baseline_b_us"].median()
    C_median = group["proposed_kernel_us"].median()
    T_median = group["proposed_total_us"].median()

    speedup_rows.append({

        "layer": layer,

        "kernel_vs_A_mean":
            A_mean / C_mean,

        "total_vs_A_mean":
            A_mean / T_mean,

        "kernel_vs_B_mean":
            B_mean / C_mean,

        "total_vs_B_mean":
            B_mean / T_mean,

        "kernel_vs_A_median":
            A_median / C_median,

        "total_vs_A_median":
            A_median / T_median,

        "kernel_vs_B_median":
            B_median / C_median,

        "total_vs_B_median":
            B_median / T_median,
    })


speedups = pd.DataFrame(speedup_rows)

display(
    speedups.round(3)
)

,layer,kernel_vs_A_mean,total_vs_A_mean,kernel_vs_B_mean,total_vs_B_mean,kernel_vs_A_median,total_vs_A_median,kernel_vs_B_median,total_vs_B_median
0,gate_proj_6912x2560,0.555,0.548,6.640,6.557,0.583,0.573,7.086,6.966
1,k_proj_640x2560,0.548,0.463,6.709,5.664,0.520,0.439,7.216,6.090
2,q_proj_2560x2560,0.283,0.278,6.977,6.843,0.353,0.342,7.735,7.493


In [ ]:
order_analysis = (
    df
    .groupby(
        ["layer", "order"]
    )
    [
        [
            "baseline_a_us",
            "baseline_b_us",
            "proposed_kernel_us",
            "activation_pack_us"
        ]
    ]
    .agg(
        ["mean", "median", "std"]
    )
)

display(
    order_analysis.round(3)
)

baseline_a_us                     baseline_b_us  \
                                   mean    median       std          mean   
layer               order                                                   
gate_proj_6912x2560 A-B-C      3627.228  3178.781  1363.630     47411.274   
                    B-C-A      4453.706  3216.624  2807.605     38594.832   
                    C-A-B      4615.792  3218.372  3011.358     65841.747   
k_proj_640x2560     A-B-C       260.744   251.680    21.191      3788.608   
                    B-C-A       307.839   311.837    23.620      2618.429   
                    C-A-B       257.320   254.020    17.744      3697.233   
q_proj_2560x2560    A-B-C      2116.750  1249.250  2440.534     47361.924   
                    B-C-A      1283.106  1243.374   100.166     27822.920   
                    C-A-B      1324.418  1239.759   199.677     41217.663   

                                                proposed_kernel_us            \
                              median        std               mean    median   
layer               order                                                      
gate_proj_6912x2560 A-B-C  39109.929  17943.159           6186.077  5499.400   
                    B-C-A  26671.034  23887.717           8545.336  5508.475   
                    C-A-B  39191.503  56312.574           8138.043  5551.331   
k_proj_640x2560     A-B-C   3609.348    513.990            496.908   494.890   
                    B-C-A   2444.538    512.810            494.880   495.146   
                    C-A-B   3601.603    352.911            514.378   502.258   
q_proj_2560x2560    A-B-C  28321.032  35139.114           5474.034  3512.377   
                    B-C-A  18485.109  19948.979           5664.122  3590.103   
                    C-A-B  27369.856  30869.349           5546.056  3520.159   

                                    activation_pack_us                   
                                std               mean   median     std  
layer               order                                                
gate_proj_6912x2560 A-B-C  1293.698             94.363   92.274   4.510  
                    B-C-A  6574.427             96.179   95.723   4.186  
                    C-A-B  5802.441             97.840   97.362   5.649  
k_proj_640x2560     A-B-C    11.767             95.324   92.313  13.354  
                    B-C-A     6.755             91.023   91.300   4.316  
                    C-A-B    43.654             91.537   88.564   8.474  
q_proj_2560x2560    A-B-C  4704.766            106.460  105.518   4.108  
                    B-C-A  4918.388            105.704  105.489   4.272  
                    C-A-B  5015.004            113.647  104.570  26.079

In [ ]:
order_means = (
    df
    .groupby(
        ["layer", "order"]
    )
    [
        [
            "baseline_a_us",
            "baseline_b_us",
            "proposed_kernel_us"
        ]
    ]
    .mean()
    .reset_index()
)

display(
    order_means.round(3)
)

,layer,order,baseline_a_us,baseline_b_us,proposed_kernel_us
0,gate_proj_6912x2560,A-B-C,3627.228,47411.274,6186.077
1,gate_proj_6912x2560,B-C-A,4453.706,38594.832,8545.336
2,gate_proj_6912x2560,C-A-B,4615.792,65841.747,8138.043
3,k_proj_640x2560,A-B-C,260.744,3788.608,496.908
4,k_proj_640x2560,B-C-A,307.839,2618.429,494.880
5,k_proj_640x2560,C-A-B,257.320,3697.233,514.378
6,q_proj_2560x2560,A-B-C,2116.750,47361.924,5474.034
7,q_proj_2560x2560,B-C-A,1283.106,27822.920,5664.122
8,q_proj_2560x2560,C-A-B,1324.418,41217.663,5546.056


In [ ]:
for layer in df["layer"].unique():

    print("\n====================================")
    print(layer)
    print("====================================")

    display(
        df[
            df["layer"] == layer
        ][
            [
                "trial",
                "order",
                "baseline_a_us",
                "baseline_b_us",
                "proposed_kernel_us",
                "activation_pack_us"
            ]
        ]
    )


k_proj_640x2560


,trial,order,baseline_a_us,baseline_b_us,proposed_kernel_us,activation_pack_us
0,0,A-B-C,283.540,3577.772,496.410,91.608
1,1,B-C-A,314.360,2439.007,495.305,90.789
2,2,C-A-B,255.660,3526.707,498.862,87.207
3,3,A-B-C,247.681,3741.032,491.877,85.798
4,4,B-C-A,314.974,2441.033,485.077,91.820
5,5,C-A-B,247.583,3619.047,487.231,97.449
6,6,A-B-C,246.224,3702.051,527.959,87.197
7,7,B-C-A,309.314,2448.044,496.615,91.812
8,8,C-A-B,254.386,3526.350,508.244,90.125
9,9,A-B-C,252.801,3597.679,496.069,100.967



gate_proj_6912x2560


,trial,order,baseline_a_us,baseline_b_us,proposed_kernel_us,activation_pack_us
30,0,A-B-C,3406.006,71438.068,8571.071,102.710
31,1,B-C-A,7403.027,54477.897,16230.102,99.434
32,2,C-A-B,7416.714,150727.605,14414.153,101.892
33,3,A-B-C,7502.191,89507.006,8594.393,102.820
34,4,B-C-A,11505.651,101445.521,24629.931,102.169
35,5,C-A-B,12341.077,191488.488,22616.437,106.844
36,6,A-B-C,3145.087,39252.084,5585.643,91.150
37,7,B-C-A,3124.656,39974.799,6300.220,95.761
38,8,C-A-B,3205.312,39051.974,5474.694,92.426
39,9,A-B-C,3156.300,38834.995,5464.464,93.871



q_proj_2560x2560


,trial,order,baseline_a_us,baseline_b_us,proposed_kernel_us,activation_pack_us
60,0,A-B-C,2195.657,73002.624,10284.048,113.331
61,1,B-C-A,1221.413,45988.461,19234.205,107.429
62,2,C-A-B,1182.164,119741.108,19211.006,103.480
63,3,A-B-C,9010.166,126816.963,17348.970,102.698
64,4,B-C-A,1271.482,79110.749,7033.312,98.449
65,5,C-A-B,1228.171,71564.270,8068.843,103.975
66,6,A-B-C,1258.720,83171.432,3507.516,105.998
67,7,B-C-A,1246.606,18467.770,3508.997,105.499
68,8,C-A-B,1238.061,29673.514,3325.546,105.165
69,9,A-B-C,1232.994,23545.450,3229.056,104.476


In [ ]:
SUMMARY_PATH = (
    "/content/drive/MyDrive/"
    "BitNetExperiments/"
    "benchmark_summary.csv"
)

SPEEDUP_PATH = (
    "/content/drive/MyDrive/"
    "BitNetExperiments/"
    "benchmark_speedups.csv"
)

RAW_BACKUP_PATH = (
    "/content/drive/MyDrive/"
    "BitNetExperiments/"
    "benchmark_raw_x86.csv"
)

summary.to_csv(
    SUMMARY_PATH,
    index=False
)

speedups.to_csv(
    SPEEDUP_PATH,
    index=False
)

df.to_csv(
    RAW_BACKUP_PATH,
    index=False
)

print("Saved:")
print(SUMMARY_PATH)
print(SPEEDUP_PATH)
print(RAW_BACKUP_PATH)

Saved:
/content/drive/MyDrive/BitNetExperiments/benchmark_summary.csv
/content/drive/MyDrive/BitNetExperiments/benchmark_speedups.csv
/content/drive/MyDrive/BitNetExperiments/benchmark_raw_x86.csv


In [ ]:
%%writefile /content/benchmark_v2.cpp

#include <algorithm>
#include <array>
#include <chrono>
#include <cstdint>
#include <fstream>
#include <iomanip>
#include <iostream>
#include <random>
#include <stdexcept>
#include <string>
#include <vector>

#if defined(_MSC_VER)
#include <intrin.h>
#define POPCOUNT32 __popcnt
#else
#define POPCOUNT32 __builtin_popcount
#endif

// Prevent compiler from eliminating benchmark computations.
volatile int64_t benchmark_sink = 0;

struct PackedLayer {
    uint32_t rows;
    uint32_t words_per_row;
    uint32_t original_cols;

    std::vector<uint32_t> pos;
    std::vector<uint32_t> neg;
};

// ============================================================
// Load packed layer
// ============================================================

PackedLayer load_layer(const std::string& filename) {

    std::ifstream file(filename, std::ios::binary);

    if (!file) {
        throw std::runtime_error(
            "Could not open: " + filename
        );
    }

    PackedLayer layer;

    file.read(
        reinterpret_cast<char*>(&layer.rows),
        sizeof(uint32_t)
    );

    file.read(
        reinterpret_cast<char*>(&layer.words_per_row),
        sizeof(uint32_t)
    );

    file.read(
        reinterpret_cast<char*>(&layer.original_cols),
        sizeof(uint32_t)
    );

    if (!file) {
        throw std::runtime_error(
            "Failed to read header."
        );
    }

    size_t total_words =
        static_cast<size_t>(layer.rows)
        * layer.words_per_row;

    layer.pos.resize(total_words);
    layer.neg.resize(total_words);

    file.read(
        reinterpret_cast<char*>(layer.pos.data()),
        total_words * sizeof(uint32_t)
    );

    file.read(
        reinterpret_cast<char*>(layer.neg.data()),
        total_words * sizeof(uint32_t)
    );

    if (!file) {
        throw std::runtime_error(
            "Failed to read packed weight data."
        );
    }

    return layer;
}

// ============================================================
// Reconstruct dense INT8 representation
//
// This happens ONCE before benchmarking.
// Baseline A does NOT pay reconstruction cost.
// ============================================================

std::vector<int8_t> reconstruct_dense_weights(
    const PackedLayer& layer
) {

    size_t total_elements =
        static_cast<size_t>(layer.rows)
        * layer.original_cols;

    std::vector<int8_t> dense(total_elements);

    for (uint32_t r = 0; r < layer.rows; ++r) {

        size_t packed_offset =
            static_cast<size_t>(r)
            * layer.words_per_row;

        size_t dense_offset =
            static_cast<size_t>(r)
            * layer.original_cols;

        for (uint32_t c = 0;
             c < layer.original_cols;
             ++c) {

            uint32_t word = c / 32;
            uint32_t bit = c % 32;

            uint32_t p =
                layer.pos[
                    packed_offset + word
                ];

            uint32_t n =
                layer.neg[
                    packed_offset + word
                ];

            int32_t weight =
                static_cast<int32_t>(
                    (p >> bit) & 1U
                )
                -
                static_cast<int32_t>(
                    (n >> bit) & 1U
                );

            dense[dense_offset + c] =
                static_cast<int8_t>(weight);
        }
    }

    return dense;
}

// ============================================================
// BASELINE A
//
// Dense INT8 ternary weights.
// Conventional multiply-accumulate.
// ============================================================

int64_t baseline_a_dense(
    const std::vector<int8_t>& weights,
    const std::vector<int8_t>& activations,
    uint32_t rows,
    uint32_t cols
) {

    int64_t checksum = 0;

    for (uint32_t r = 0; r < rows; ++r) {

        int32_t acc = 0;

        size_t offset =
            static_cast<size_t>(r)
            * cols;

        for (uint32_t c = 0; c < cols; ++c) {

            acc +=
                static_cast<int32_t>(
                    weights[offset + c]
                )
                *
                static_cast<int32_t>(
                    activations[c]
                );
        }

        checksum += acc;
    }

    return checksum;
}

// ============================================================
// BASELINE B
//
// Packed ternary weights.
// Decode each weight inside the kernel,
// then conventional arithmetic.
//
// Measures unpacking tax.
// ============================================================

int64_t baseline_b_unpack(
    const PackedLayer& layer,
    const std::vector<int8_t>& activations
) {

    int64_t checksum = 0;

    for (uint32_t r = 0;
         r < layer.rows;
         ++r) {

        int32_t acc = 0;

        size_t offset =
            static_cast<size_t>(r)
            * layer.words_per_row;

        for (uint32_t c = 0;
             c < layer.original_cols;
             ++c) {

            uint32_t word = c / 32;
            uint32_t bit = c % 32;

            uint32_t p =
                layer.pos[offset + word];

            uint32_t n =
                layer.neg[offset + word];

            int32_t weight =
                static_cast<int32_t>(
                    (p >> bit) & 1U
                )
                -
                static_cast<int32_t>(
                    (n >> bit) & 1U
                );

            acc +=
                weight
                *
                static_cast<int32_t>(
                    activations[c]
                );
        }

        checksum += acc;
    }

    return checksum;
}

// ============================================================
// Activation packing
//
// INT8 activations -> 8 bitplanes
// ============================================================

void pack_activations(
    const std::vector<int8_t>& activations,
    std::vector<uint32_t> planes[8],
    uint32_t words_per_row
) {

    for (int b = 0; b < 8; ++b) {

        planes[b].assign(
            words_per_row,
            0U
        );
    }

    for (uint32_t c = 0;
         c < activations.size();
         ++c) {

        uint8_t value =
            static_cast<uint8_t>(
                activations[c]
            );

        uint32_t word = c / 32;
        uint32_t bit = c % 32;

        for (int b = 0; b < 8; ++b) {

            if ((value >> b) & 1U) {

                planes[b][word] |=
                    (1U << bit);
            }
        }
    }
}

// ============================================================
// PROPOSED BIT-SERIAL KERNEL
// ============================================================

int64_t proposed_bitserial(
    const PackedLayer& layer,
    const std::vector<uint32_t> planes[8]
) {

    int64_t checksum = 0;

    for (uint32_t r = 0;
         r < layer.rows;
         ++r) {

        int32_t acc = 0;

        size_t offset =
            static_cast<size_t>(r)
            * layer.words_per_row;

        for (int b = 0; b < 8; ++b) {

            int32_t plane_acc = 0;

            // Two's-complement INT8:
            // bits 0-6 positive significance
            // bit 7 negative significance
            int32_t scale =
                (b == 7)
                ? -128
                : (1 << b);

            for (uint32_t w = 0;
                 w < layer.words_per_row;
                 ++w) {

                uint32_t activation_bits =
                    planes[b][w];

                plane_acc +=
                    POPCOUNT32(
                        layer.pos[offset + w]
                        & activation_bits
                    );

                plane_acc -=
                    POPCOUNT32(
                        layer.neg[offset + w]
                        & activation_bits
                    );
            }

            acc +=
                plane_acc
                * scale;
        }

        checksum += acc;
    }

    return checksum;
}

// ============================================================
// Timing helper
//
// Runs kernel multiple times inside timed region,
// then returns average latency per invocation.
// ============================================================

template <typename Func>
double time_repeated_us(
    Func&& function,
    int repetitions
) {

    int64_t local_checksum = 0;

    auto start =
        std::chrono::steady_clock::now();

    for (int i = 0;
         i < repetitions;
         ++i) {

        local_checksum +=
            function();
    }

    auto end =
        std::chrono::steady_clock::now();

    benchmark_sink +=
        local_checksum;

    double total_us =
        std::chrono::duration<
            double,
            std::micro
        >(end - start).count();

    return total_us
        / repetitions;
}

// ============================================================
// Main
// ============================================================

int main(
    int argc,
    char* argv[]
) {

    if (argc != 4) {

        std::cerr
            << "Usage:\n"
            << "./benchmark_v2 "
            << "<layer.bin> "
            << "<layer_name> "
            << "<output.csv>\n";

        return 1;
    }

    const std::string binary_path =
        argv[1];

    const std::string layer_name =
        argv[2];

    const std::string csv_path =
        argv[3];

    // --------------------------------------------------------
    // Benchmark configuration
    // --------------------------------------------------------

    constexpr int WARMUP_ROUNDS = 20;

    constexpr int TRIALS = 60;

    constexpr int INNER_REPS = 5;

    // All 6 permutations.
    // Each occurs exactly 10 times over 60 trials.

    const std::array<std::string, 6>
        ORDERS = {

            "A-B-C",
            "A-C-B",
            "B-A-C",
            "B-C-A",
            "C-A-B",
            "C-B-A"
        };

    // --------------------------------------------------------
    // Load layer
    // --------------------------------------------------------

    PackedLayer layer =
        load_layer(binary_path);

    std::cout
        << "\n========================================\n"
        << "FINAL BENCHMARK V2\n"
        << "========================================\n"
        << "Layer        : "
        << layer_name
        << "\n"
        << "Matrix       : "
        << layer.rows
        << " x "
        << layer.original_cols
        << "\n"
        << "Words/row    : "
        << layer.words_per_row
        << "\n"
        << "Warmups      : "
        << WARMUP_ROUNDS
        << "\n"
        << "Trials       : "
        << TRIALS
        << "\n"
        << "Inner reps   : "
        << INNER_REPS
        << "\n"
        << "========================================\n";

    // --------------------------------------------------------
    // Dense weights for Baseline A
    // --------------------------------------------------------

    std::cout
        << "[1] Reconstructing dense INT8 weights...\n";

    std::vector<int8_t> dense_weights =
        reconstruct_dense_weights(layer);

    // --------------------------------------------------------
    // ONE shared activation vector
    // --------------------------------------------------------

    std::cout
        << "[2] Generating shared INT8 activations...\n";

    std::vector<int8_t> activations(
        layer.original_cols
    );

    std::mt19937 rng(42);

    std::uniform_int_distribution<int>
        distribution(-128, 127);

    for (auto& value : activations) {

        value =
            static_cast<int8_t>(
                distribution(rng)
            );
    }

    // --------------------------------------------------------
    // Initial activation bitplanes
    // --------------------------------------------------------

    std::vector<uint32_t>
        activation_planes[8];

    pack_activations(
        activations,
        activation_planes,
        layer.words_per_row
    );

    // --------------------------------------------------------
    // THREE-WAY CORRECTNESS
    // --------------------------------------------------------

    std::cout
        << "[3] Three-way correctness check...\n";

    int64_t output_a =
        baseline_a_dense(
            dense_weights,
            activations,
            layer.rows,
            layer.original_cols
        );

    int64_t output_b =
        baseline_b_unpack(
            layer,
            activations
        );

    int64_t output_c =
        proposed_bitserial(
            layer,
            activation_planes
        );

    std::cout
        << "    A = "
        << output_a
        << "\n";

    std::cout
        << "    B = "
        << output_b
        << "\n";

    std::cout
        << "    C = "
        << output_c
        << "\n";

    if (
        output_a != output_b
        ||
        output_a != output_c
    ) {

        std::cerr
            << "\nFAIL: "
            << "A != B != C\n";

        return 2;
    }

    std::cout
        << "    PASS: A == B == C\n";

    // --------------------------------------------------------
    // Stronger warm-up
    // --------------------------------------------------------

    std::cout
        << "[4] Running "
        << WARMUP_ROUNDS
        << " warm-up rounds...\n";

    for (int i = 0;
         i < WARMUP_ROUNDS;
         ++i) {

        benchmark_sink +=
            baseline_a_dense(
                dense_weights,
                activations,
                layer.rows,
                layer.original_cols
            );

        benchmark_sink +=
            baseline_b_unpack(
                layer,
                activations
            );

        benchmark_sink +=
            proposed_bitserial(
                layer,
                activation_planes
            );
    }

    // --------------------------------------------------------
    // CSV setup
    // --------------------------------------------------------

    bool csv_exists =
        std::ifstream(csv_path).good();

    std::ofstream csv(
        csv_path,
        std::ios::app
    );

    if (!csv) {

        std::cerr
            << "Could not open CSV: "
            << csv_path
            << "\n";

        return 3;
    }

    if (!csv_exists) {

        csv
            << "layer,"
            << "trial,"
            << "order,"
            << "baseline_a_us,"
            << "baseline_b_us,"
            << "activation_pack_us,"
            << "proposed_kernel_us,"
            << "proposed_e2e_us\n";
    }

    // --------------------------------------------------------
    // 60 measured trials
    // --------------------------------------------------------

    std::cout
        << "[5] Starting measured trials...\n";

    for (int trial = 0;
         trial < TRIALS;
         ++trial) {

        const std::string order =
            ORDERS[trial % ORDERS.size()];

        double time_a = 0.0;
        double time_b = 0.0;
        double time_c = 0.0;

        // ----------------------------------------------------
        // Activation packing measured independently
        //
        // Multiple packing operations are timed and averaged.
        // ----------------------------------------------------

        std::vector<uint32_t>
            trial_planes[8];

        auto pack_start =
            std::chrono::steady_clock::now();

        for (int rep = 0;
             rep < INNER_REPS;
             ++rep) {

            pack_activations(
                activations,
                trial_planes,
                layer.words_per_row
            );
        }

        auto pack_end =
            std::chrono::steady_clock::now();

        double activation_pack_us =
            std::chrono::duration<
                double,
                std::micro
            >(
                pack_end - pack_start
            ).count()
            / INNER_REPS;

        // ----------------------------------------------------
        // Helpers for A/B/C
        // ----------------------------------------------------

        auto run_a = [&]() {

            time_a =
                time_repeated_us(
                    [&]() {

                        return baseline_a_dense(
                            dense_weights,
                            activations,
                            layer.rows,
                            layer.original_cols
                        );
                    },
                    INNER_REPS
                );
        };

        auto run_b = [&]() {

            time_b =
                time_repeated_us(
                    [&]() {

                        return baseline_b_unpack(
                            layer,
                            activations
                        );
                    },
                    INNER_REPS
                );
        };

        auto run_c = [&]() {

            time_c =
                time_repeated_us(
                    [&]() {

                        return proposed_bitserial(
                            layer,
                            trial_planes
                        );
                    },
                    INNER_REPS
                );
        };

        // ----------------------------------------------------
        // Execute all six balanced permutations
        // ----------------------------------------------------

        if (order == "A-B-C") {

            run_a();
            run_b();
            run_c();
        }

        else if (order == "A-C-B") {

            run_a();
            run_c();
            run_b();
        }

        else if (order == "B-A-C") {

            run_b();
            run_a();
            run_c();
        }

        else if (order == "B-C-A") {

            run_b();
            run_c();
            run_a();
        }

        else if (order == "C-A-B") {

            run_c();
            run_a();
            run_b();
        }

        else if (order == "C-B-A") {

            run_c();
            run_b();
            run_a();
        }

        // ----------------------------------------------------
        // Direct Proposed end-to-end timing
        //
        // Pack activation + execute kernel in one timed region.
        // ----------------------------------------------------

        int64_t e2e_checksum = 0;

        auto e2e_start =
            std::chrono::steady_clock::now();

        for (int rep = 0;
             rep < INNER_REPS;
             ++rep) {

            std::vector<uint32_t>
                e2e_planes[8];

            pack_activations(
                activations,
                e2e_planes,
                layer.words_per_row
            );

            e2e_checksum +=
                proposed_bitserial(
                    layer,
                    e2e_planes
                );
        }

        auto e2e_end =
            std::chrono::steady_clock::now();

        benchmark_sink +=
            e2e_checksum;

        double proposed_e2e_us =
            std::chrono::duration<
                double,
                std::micro
            >(
                e2e_end - e2e_start
            ).count()
            / INNER_REPS;

        // ----------------------------------------------------
        // Write raw result
        // ----------------------------------------------------

        csv
            << layer_name
            << ","
            << trial
            << ","
            << order
            << ","
            << std::fixed
            << std::setprecision(3)
            << time_a
            << ","
            << time_b
            << ","
            << activation_pack_us
            << ","
            << time_c
            << ","
            << proposed_e2e_us
            << "\n";

        std::cout
            << "Trial "
            << std::setw(2)
            << trial
            << " ["
            << order
            << "] "
            << "A="
            << std::fixed
            << std::setprecision(2)
            << time_a
            << " us  "
            << "B="
            << time_b
            << " us  "
            << "C="
            << time_c
            << " us  "
            << "Pack="
            << activation_pack_us
            << " us  "
            << "E2E="
            << proposed_e2e_us
            << " us\n";
    }

    csv.close();

    std::cout
        << "\n========================================\n"
        << "BENCHMARK COMPLETE\n"
        << "========================================\n"
        << "Layer: "
        << layer_name
        << "\n"
        << "Checksum sink: "
        << benchmark_sink
        << "\n"
        << "CSV: "
        << csv_path
        << "\n";

    return 0;
}

Writing /content/benchmark_v2.cpp


In [ ]:
!g++ \
    -O3 \
    -march=native \
    -mpopcnt \
    -std=c++17 \
    /content/benchmark_v2.cpp \
    -o /content/benchmark_v2

In [ ]:
import os
import shutil

OLD_CSV = "/content/benchmark_raw.csv"
V2_CSV = "/content/benchmark_v2_raw.csv"

# Preserve V1 if it exists
if os.path.exists(OLD_CSV):
    backup = (
        "/content/drive/MyDrive/"
        "BitNetExperiments/"
        "benchmark_v1_preliminary.csv"
    )

    shutil.copy2(
        OLD_CSV,
        backup
    )

    print("V1 backed up:", backup)

# Start clean V2 dataset
if os.path.exists(V2_CSV):
    os.remove(V2_CSV)

print("V2 ready:", V2_CSV)

V1 backed up: /content/drive/MyDrive/BitNetExperiments/benchmark_v1_preliminary.csv
V2 ready: /content/benchmark_v2_raw.csv


In [ ]:
!/content/benchmark_v2 \
"/content/drive/MyDrive/BitNetExperiments/k_proj_640x2560.bin" \
"k_proj_640x2560" \
"/content/benchmark_v2_raw.csv"


FINAL BENCHMARK V2
Layer        : k_proj_640x2560
Matrix       : 640 x 2560
Words/row    : 80
Warmups      : 20
Trials       : 60
Inner reps   : 5
[1] Reconstructing dense INT8 weights...
[2] Generating shared INT8 activations...
[3] Three-way correctness check...
    A = -33730
    B = -33730
    C = -33730
    PASS: A == B == C
[4] Running 20 warm-up rounds...
[5] Starting measured trials...
Trial  0 [A-B-C] A=240.87 us  B=2966.89 us  C=229.21 us  Pack=84.35 us  E2E=1243.73 us
Trial  1 [A-C-B] A=289.73 us  B=4250.16 us  C=219.55 us  Pack=104.48 us  E2E=1159.15 us
Trial  2 [B-A-C] A=272.94 us  B=4205.51 us  C=253.99 us  Pack=106.55 us  E2E=1294.98 us
Trial  3 [B-C-A] A=694.33 us  B=4232.11 us  C=199.51 us  Pack=105.90 us  E2E=1204.24 us
Trial  4 [C-A-B] A=321.96 us  B=4185.32 us  C=195.79 us  Pack=102.44 us  E2E=1109.68 us
Trial  5 [C-B-A] A=371.55 us  B=4733.48 us  C=224.09 us  Pack=101.60 us  E2E=1325.18 us
Trial  6 [A-B-C] A=511.41 us  B=4538.50 us  C=233.79 us  Pack=108.79 us  E2

In [ ]:
!/content/benchmark_v2 \
"/content/drive/MyDrive/BitNetExperiments/q_proj_2560x2560.bin" \
"q_proj_2560x2560" \
"/content/benchmark_v2_raw.csv"


FINAL BENCHMARK V2
Layer        : q_proj_2560x2560
Matrix       : 2560 x 2560
Words/row    : 80
Warmups      : 20
Trials       : 60
Inner reps   : 5
[1] Reconstructing dense INT8 weights...
[2] Generating shared INT8 activations...
[3] Three-way correctness check...
    A = -94125
    B = -94125
    C = -94125
    PASS: A == B == C
[4] Running 20 warm-up rounds...
[5] Starting measured trials...
Trial  0 [A-B-C] A=1123.39 us  B=20790.10 us  C=871.30 us  Pack=97.37 us  E2E=4414.48 us
Trial  1 [A-C-B] A=1153.53 us  B=25376.02 us  C=822.85 us  Pack=111.72 us  E2E=4266.34 us
Trial  2 [B-A-C] A=1067.18 us  B=13878.15 us  C=569.06 us  Pack=87.27 us  E2E=2813.06 us
Trial  3 [B-C-A] A=1025.67 us  B=12694.94 us  C=546.11 us  Pack=83.07 us  E2E=2798.97 us
Trial  4 [C-A-B] A=1004.10 us  B=10251.81 us  C=538.28 us  Pack=82.09 us  E2E=3210.08 us
Trial  5 [C-B-A] A=1075.18 us  B=9801.21 us  C=561.50 us  Pack=94.81 us  E2E=2893.78 us
Trial  6 [A-B-C] A=1023.31 us  B=9832.64 us  C=538.23 us  Pack=87.

In [ ]:
!/content/benchmark_v2 \
"/content/drive/MyDrive/BitNetExperiments/gate_proj_6912x2560.bin" \
"gate_proj_6912x2560" \
"/content/benchmark_v2_raw.csv"


FINAL BENCHMARK V2
Layer        : gate_proj_6912x2560
Matrix       : 6912 x 2560
Words/row    : 80
Warmups      : 20
Trials       : 60
Inner reps   : 5
[1] Reconstructing dense INT8 weights...
[2] Generating shared INT8 activations...
[3] Three-way correctness check...
    A = -2994401
    B = -2994401
    C = -2994401
    PASS: A == B == C
[4] Running 20 warm-up rounds...
[5] Starting measured trials...
Trial  0 [A-B-C] A=3178.03 us  B=29646.65 us  C=1467.42 us  Pack=90.66 us  E2E=8577.98 us
Trial  1 [A-C-B] A=3280.65 us  B=26942.94 us  C=1471.14 us  Pack=85.57 us  E2E=7818.09 us
Trial  2 [B-A-C] A=3174.59 us  B=27093.91 us  C=1506.63 us  Pack=85.21 us  E2E=7491.29 us
Trial  3 [B-C-A] A=3563.17 us  B=39280.63 us  C=1484.22 us  Pack=82.56 us  E2E=8032.36 us
Trial  4 [C-A-B] A=3162.17 us  B=27679.17 us  C=1491.89 us  Pack=85.12 us  E2E=8424.83 us
Trial  5 [C-B-A] A=3221.98 us  B=31615.86 us  C=1474.73 us  Pack=85.86 us  E2E=7637.39 us
Trial  6 [A-B-C] A=3168.15 us  B=27604.58 us  C=163

In [ ]:
import pandas as pd

df_v2 = pd.read_csv(
    "/content/benchmark_v2_raw.csv"
)

print("Total rows:", len(df_v2))

print("\nTrials per layer:")
print(
    df_v2.groupby("layer").size()
)

print("\nTrials per layer and order:")
print(
    df_v2
    .groupby(["layer", "order"])
    .size()
)

display(
    df_v2.head()
)

Total rows: 180

Trials per layer:
layer
gate_proj_6912x2560    60
k_proj_640x2560        60
q_proj_2560x2560       60
dtype: int64

Trials per layer and order:
layer                order
gate_proj_6912x2560  A-B-C    10
                     A-C-B    10
                     B-A-C    10
                     B-C-A    10
                     C-A-B    10
                     C-B-A    10
k_proj_640x2560      A-B-C    10
                     A-C-B    10
                     B-A-C    10
                     B-C-A    10
                     C-A-B    10
                     C-B-A    10
q_proj_2560x2560     A-B-C    10
                     A-C-B    10
                     B-A-C    10
                     B-C-A    10
                     C-A-B    10
                     C-B-A    10
dtype: int64


,layer,trial,order,baseline_a_us,baseline_b_us,activation_pack_us,proposed_kernel_us,proposed_e2e_us
0,k_proj_640x2560,0,A-B-C,240.867,2966.886,84.354,229.213,1243.732
1,k_proj_640x2560,1,A-C-B,289.727,4250.161,104.478,219.554,1159.146
2,k_proj_640x2560,2,B-A-C,272.936,4205.511,106.553,253.990,1294.981
3,k_proj_640x2560,3,B-C-A,694.329,4232.113,105.895,199.513,1204.244
4,k_proj_640x2560,4,C-A-B,321.958,4185.322,102.442,195.795,1109.679


In [ ]:
import pandas as pd
import numpy as np

CSV = "/content/benchmark_v2_raw.csv"
df = pd.read_csv(CSV)

# Derived value using independently measured components
df["proposed_total_component_us"] = (
    df["activation_pack_us"] +
    df["proposed_kernel_us"]
)

# Speedups: >1 means Proposed is faster
df["speedup_vs_A_kernel"] = (
    df["baseline_a_us"] /
    df["proposed_kernel_us"]
)

df["speedup_vs_A_total"] = (
    df["baseline_a_us"] /
    df["proposed_total_component_us"]
)

df["speedup_vs_B_kernel"] = (
    df["baseline_b_us"] /
    df["proposed_kernel_us"]
)

df["speedup_vs_B_total"] = (
    df["baseline_b_us"] /
    df["proposed_total_component_us"]
)

metrics = [
    "baseline_a_us",
    "baseline_b_us",
    "activation_pack_us",
    "proposed_kernel_us",
    "proposed_total_component_us",
    "proposed_e2e_us"
]

# --------------------------------------------------
# 1. Overall statistics
# --------------------------------------------------

rows = []

for layer, g in df.groupby("layer"):
    for metric in metrics:
        x = g[metric]

        rows.append({
            "layer": layer,
            "metric": metric,
            "mean_us": x.mean(),
            "median_us": x.median(),
            "std_us": x.std(),
            "cv_percent": 100 * x.std() / x.mean(),
            "min_us": x.min(),
            "p25_us": x.quantile(0.25),
            "p75_us": x.quantile(0.75),
            "p95_us": x.quantile(0.95),
            "max_us": x.max()
        })

stats = pd.DataFrame(rows)

print("\n=== OVERALL LATENCY STATISTICS ===")
display(stats.round(3))


# --------------------------------------------------
# 2. Median-based comparison table
# --------------------------------------------------

summary_rows = []

for layer, g in df.groupby("layer"):

    A = g["baseline_a_us"].median()
    B = g["baseline_b_us"].median()
    C = g["proposed_kernel_us"].median()
    P = g["activation_pack_us"].median()

    # Use median of per-trial sum rather than sum of medians
    total = g["proposed_total_component_us"].median()

    summary_rows.append({
        "layer": layer,
        "baseline_A_median_us": A,
        "baseline_B_median_us": B,
        "proposed_kernel_median_us": C,
        "activation_pack_median_us": P,
        "proposed_total_median_us": total,

        # >1 = proposed faster
        "kernel_speedup_vs_A": A / C,
        "total_speedup_vs_A": A / total,
        "kernel_speedup_vs_B": B / C,
        "total_speedup_vs_B": B / total,

        "packing_overhead_pct_of_total":
            100 * P / total
    })

summary = pd.DataFrame(summary_rows)

print("\n=== MEDIAN-BASED FINAL COMPARISON ===")
display(summary.round(3))


# --------------------------------------------------
# 3. Execution-order effect
# --------------------------------------------------

order_stats = (
    df.groupby(["layer", "order"])
      .agg(
          baseline_A_median=("baseline_a_us", "median"),
          baseline_B_median=("baseline_b_us", "median"),
          proposed_median=("proposed_kernel_us", "median"),
          pack_median=("activation_pack_us", "median"),
          count=("trial", "count")
      )
      .reset_index()
)

print("\n=== EXECUTION ORDER EFFECT ===")
display(order_stats.round(3))


# --------------------------------------------------
# 4. E2E discrepancy check
# --------------------------------------------------

df["e2e_over_component_ratio"] = (
    df["proposed_e2e_us"] /
    df["proposed_total_component_us"]
)

e2e_check = (
    df.groupby("layer")
      .agg(
          component_total_median=(
              "proposed_total_component_us",
              "median"
          ),
          measured_e2e_median=(
              "proposed_e2e_us",
              "median"
          ),
          e2e_ratio_median=(
              "e2e_over_component_ratio",
              "median"
          )
      )
      .reset_index()
)

print("\n=== E2E CONSISTENCY CHECK ===")
display(e2e_check.round(3))


# --------------------------------------------------
# 5. Early vs late trials
# Useful for detecting environment/frequency shifts
# --------------------------------------------------

df["trial_half"] = np.where(
    df["trial"] < 30,
    "first_30",
    "last_30"
)

phase_stats = (
    df.groupby(["layer", "trial_half"])
      .agg(
          A_median=("baseline_a_us", "median"),
          B_median=("baseline_b_us", "median"),
          C_median=("proposed_kernel_us", "median"),
          Pack_median=("activation_pack_us", "median")
      )
      .reset_index()
)

print("\n=== FIRST 30 VS LAST 30 ===")
display(phase_stats.round(3))


=== OVERALL LATENCY STATISTICS ===


,layer,metric,mean_us,median_us,std_us,cv_percent,min_us,p25_us,p75_us,p95_us,max_us
0,gate_proj_6912x2560,baseline_a_us,3343.230,3222.502,340.359,10.181,3026.303,3141.814,3358.164,4107.367,4552.815
1,gate_proj_6912x2560,baseline_b_us,33044.758,27702.656,9008.467,27.261,26601.499,27090.147,39497.714,49489.240,52192.146
2,gate_proj_6912x2560,activation_pack_us,90.278,85.712,9.518,10.542,81.484,83.892,94.125,111.660,113.501
3,gate_proj_6912x2560,proposed_kernel_us,1846.944,1496.187,596.895,32.318,1458.225,1477.166,2272.093,2965.784,3277.324
4,gate_proj_6912x2560,proposed_total_component_us,1937.222,1584.282,605.128,31.237,1540.291,1562.056,2374.873,3078.806,3388.953
5,gate_proj_6912x2560,proposed_e2e_us,9159.018,7706.691,2610.925,28.507,7421.332,7521.820,9623.061,14562.497,15069.783
6,k_proj_640x2560,baseline_a_us,316.209,272.850,121.554,38.441,234.286,254.296,313.347,677.714,737.365
7,k_proj_640x2560,baseline_b_us,3986.083,4201.268,1601.552,40.179,2429.188,2474.899,4648.493,6951.820,9380.619
8,k_proj_640x2560,activation_pack_us,102.134,102.040,48.279,47.270,79.246,83.066,106.752,111.578,458.411
9,k_proj_640x2560,proposed_kernel_us,198.195,199.400,81.855,41.301,131.828,135.462,232.417,254.087,651.614



=== MEDIAN-BASED FINAL COMPARISON ===


,layer,baseline_A_median_us,baseline_B_median_us,proposed_kernel_median_us,activation_pack_median_us,proposed_total_median_us,kernel_speedup_vs_A,total_speedup_vs_A,kernel_speedup_vs_B,total_speedup_vs_B,packing_overhead_pct_of_total
0,gate_proj_6912x2560,3222.502,27702.656,1496.187,85.712,1584.282,2.154,2.034,18.516,17.486,5.410
1,k_proj_640x2560,272.850,4201.268,199.400,102.040,304.389,1.368,0.896,21.070,13.802,33.523
2,q_proj_2560x2560,1133.251,17680.622,902.537,100.098,1009.325,1.256,1.123,19.590,17.517,9.917



=== EXECUTION ORDER EFFECT ===


,layer,order,baseline_A_median,baseline_B_median,proposed_median,pack_median,count
0,gate_proj_6912x2560,A-B-C,3173.092,27401.696,1493.514,84.416,10
1,gate_proj_6912x2560,A-C-B,3172.682,27303.127,1492.672,86.116,10
2,gate_proj_6912x2560,B-A-C,3264.355,28997.488,1510.298,87.390,10
3,gate_proj_6912x2560,B-C-A,3289.582,30120.746,1504.769,84.660,10
4,gate_proj_6912x2560,C-A-B,3249.185,28641.399,1484.110,84.784,10
5,gate_proj_6912x2560,C-B-A,3175.583,27321.558,1521.526,87.616,10
6,k_proj_640x2560,A-B-C,257.706,3424.025,162.083,93.092,10
7,k_proj_640x2560,A-C-B,273.371,4187.460,215.486,100.796,10
8,k_proj_640x2560,B-A-C,272.541,3388.357,166.432,103.218,10
9,k_proj_640x2560,B-C-A,301.632,4233.042,201.805,104.306,10



=== E2E CONSISTENCY CHECK ===


,layer,component_total_median,measured_e2e_median,e2e_ratio_median
0,gate_proj_6912x2560,1584.282,7706.691,4.806
1,k_proj_640x2560,304.389,1161.238,3.603
2,q_proj_2560x2560,1009.325,4643.084,4.538



=== FIRST 30 VS LAST 30 ===


,layer,trial_half,A_median,B_median,C_median,Pack_median
0,gate_proj_6912x2560,first_30,3204.652,27489.506,1486.787,85.020
1,gate_proj_6912x2560,last_30,3258.138,35181.656,2389.399,93.794
2,k_proj_640x2560,first_30,293.342,4527.642,230.983,106.153
3,k_proj_640x2560,last_30,254.528,2472.692,135.446,82.986
4,q_proj_2560x2560,first_30,1181.156,17735.192,866.159,99.388
5,q_proj_2560x2560,last_30,1123.818,17647.122,930.437,101.392


In [ ]:
%%writefile /content/benchmark_v3.cpp

#include <algorithm>
#include <array>
#include <chrono>
#include <cstdint>
#include <fstream>
#include <iomanip>
#include <iostream>
#include <random>
#include <stdexcept>
#include <string>
#include <vector>

#if defined(_MSC_VER)
#include <intrin.h>
#define POPCOUNT32 __popcnt
#else
#define POPCOUNT32 __builtin_popcount
#endif

// Prevent compiler from eliminating benchmark computations.
volatile int64_t benchmark_sink = 0;

struct PackedLayer {
    uint32_t rows;
    uint32_t words_per_row;
    uint32_t original_cols;

    std::vector<uint32_t> pos;
    std::vector<uint32_t> neg;
};

// ============================================================
// Load packed layer
// ============================================================

PackedLayer load_layer(const std::string& filename) {

    std::ifstream file(filename, std::ios::binary);

    if (!file) {
        throw std::runtime_error(
            "Could not open: " + filename
        );
    }

    PackedLayer layer;

    file.read(
        reinterpret_cast<char*>(&layer.rows),
        sizeof(uint32_t)
    );

    file.read(
        reinterpret_cast<char*>(&layer.words_per_row),
        sizeof(uint32_t)
    );

    file.read(
        reinterpret_cast<char*>(&layer.original_cols),
        sizeof(uint32_t)
    );

    if (!file) {
        throw std::runtime_error(
            "Failed to read header."
        );
    }

    size_t total_words =
        static_cast<size_t>(layer.rows)
        * layer.words_per_row;

    layer.pos.resize(total_words);
    layer.neg.resize(total_words);

    file.read(
        reinterpret_cast<char*>(layer.pos.data()),
        total_words * sizeof(uint32_t)
    );

    file.read(
        reinterpret_cast<char*>(layer.neg.data()),
        total_words * sizeof(uint32_t)
    );

    if (!file) {
        throw std::runtime_error(
            "Failed to read packed weight data."
        );
    }

    return layer;
}

// ============================================================
// Reconstruct dense INT8 representation
//
// Happens ONCE before benchmarking.
// Baseline A does NOT pay reconstruction cost.
// ============================================================

std::vector<int8_t> reconstruct_dense_weights(
    const PackedLayer& layer
) {

    size_t total_elements =
        static_cast<size_t>(layer.rows)
        * layer.original_cols;

    std::vector<int8_t> dense(total_elements);

    for (uint32_t r = 0;
         r < layer.rows;
         ++r) {

        size_t packed_offset =
            static_cast<size_t>(r)
            * layer.words_per_row;

        size_t dense_offset =
            static_cast<size_t>(r)
            * layer.original_cols;

        for (uint32_t c = 0;
             c < layer.original_cols;
             ++c) {

            uint32_t word = c / 32;
            uint32_t bit = c % 32;

            uint32_t p =
                layer.pos[
                    packed_offset + word
                ];

            uint32_t n =
                layer.neg[
                    packed_offset + word
                ];

            int32_t weight =
                static_cast<int32_t>(
                    (p >> bit) & 1U
                )
                -
                static_cast<int32_t>(
                    (n >> bit) & 1U
                );

            dense[dense_offset + c] =
                static_cast<int8_t>(weight);
        }
    }

    return dense;
}

// ============================================================
// BASELINE A
//
// Dense INT8 ternary weights.
// Conventional multiply-accumulate.
// ============================================================

int64_t baseline_a_dense(
    const std::vector<int8_t>& weights,
    const std::vector<int8_t>& activations,
    uint32_t rows,
    uint32_t cols
) {

    int64_t checksum = 0;

    for (uint32_t r = 0;
         r < rows;
         ++r) {

        int32_t acc = 0;

        size_t offset =
            static_cast<size_t>(r)
            * cols;

        for (uint32_t c = 0;
             c < cols;
             ++c) {

            acc +=
                static_cast<int32_t>(
                    weights[offset + c]
                )
                *
                static_cast<int32_t>(
                    activations[c]
                );
        }

        checksum += acc;
    }

    return checksum;
}

// ============================================================
// BASELINE B
//
// Packed ternary weights.
// Decode each weight inside kernel,
// then conventional arithmetic.
//
// Measures unpacking tax.
// ============================================================

int64_t baseline_b_unpack(
    const PackedLayer& layer,
    const std::vector<int8_t>& activations
) {

    int64_t checksum = 0;

    for (uint32_t r = 0;
         r < layer.rows;
         ++r) {

        int32_t acc = 0;

        size_t offset =
            static_cast<size_t>(r)
            * layer.words_per_row;

        for (uint32_t c = 0;
             c < layer.original_cols;
             ++c) {

            uint32_t word = c / 32;
            uint32_t bit = c % 32;

            uint32_t p =
                layer.pos[offset + word];

            uint32_t n =
                layer.neg[offset + word];

            int32_t weight =
                static_cast<int32_t>(
                    (p >> bit) & 1U
                )
                -
                static_cast<int32_t>(
                    (n >> bit) & 1U
                );

            acc +=
                weight
                *
                static_cast<int32_t>(
                    activations[c]
                );
        }

        checksum += acc;
    }

    return checksum;
}

// ============================================================
// Activation packing
//
// INT8 activations -> 8 bitplanes
//
// IMPORTANT V3 CHANGE:
// Buffers are preallocated.
// This function performs NO vector allocation.
// ============================================================

void pack_activations(
    const std::vector<int8_t>& activations,
    std::vector<uint32_t> planes[8],
    uint32_t words_per_row
) {

    // Clear existing preallocated buffers.
    for (int b = 0; b < 8; ++b) {

        std::fill(
            planes[b].begin(),
            planes[b].end(),
            0U
        );
    }

    for (uint32_t c = 0;
         c < activations.size();
         ++c) {

        uint8_t value =
            static_cast<uint8_t>(
                activations[c]
            );

        uint32_t word = c / 32;
        uint32_t bit = c % 32;

        for (int b = 0; b < 8; ++b) {

            if ((value >> b) & 1U) {

                planes[b][word] |=
                    (1U << bit);
            }
        }
    }
}

// ============================================================
// PROPOSED BIT-SERIAL KERNEL
// ============================================================

int64_t proposed_bitserial(
    const PackedLayer& layer,
    const std::vector<uint32_t> planes[8]
) {

    int64_t checksum = 0;

    for (uint32_t r = 0;
         r < layer.rows;
         ++r) {

        int32_t acc = 0;

        size_t offset =
            static_cast<size_t>(r)
            * layer.words_per_row;

        for (int b = 0;
             b < 8;
             ++b) {

            int32_t plane_acc = 0;

            // Two's-complement INT8:
            // bits 0-6 positive significance
            // bit 7 negative significance.
            int32_t scale =
                (b == 7)
                ? -128
                : (1 << b);

            for (uint32_t w = 0;
                 w < layer.words_per_row;
                 ++w) {

                uint32_t activation_bits =
                    planes[b][w];

                plane_acc +=
                    POPCOUNT32(
                        layer.pos[offset + w]
                        & activation_bits
                    );

                plane_acc -=
                    POPCOUNT32(
                        layer.neg[offset + w]
                        & activation_bits
                    );
            }

            acc +=
                plane_acc
                * scale;
        }

        checksum += acc;
    }

    return checksum;
}

// ============================================================
// Timing helper
//
// Runs kernel multiple times inside timed region,
// then returns average latency per invocation.
// ============================================================

template <typename Func>
double time_repeated_us(
    Func&& function,
    int repetitions
) {

    int64_t local_checksum = 0;

    auto start =
        std::chrono::steady_clock::now();

    for (int i = 0;
         i < repetitions;
         ++i) {

        local_checksum +=
            function();
    }

    auto end =
        std::chrono::steady_clock::now();

    benchmark_sink +=
        local_checksum;

    double total_us =
        std::chrono::duration<
            double,
            std::micro
        >(end - start).count();

    return total_us
        / repetitions;
}

// ============================================================
// Main
// ============================================================

int main(
    int argc,
    char* argv[]
) {

    if (argc != 4) {

        std::cerr
            << "Usage:\n"
            << "./benchmark_v3 "
            << "<layer.bin> "
            << "<layer_name> "
            << "<output.csv>\n";

        return 1;
    }

    const std::string binary_path =
        argv[1];

    const std::string layer_name =
        argv[2];

    const std::string csv_path =
        argv[3];

    // --------------------------------------------------------
    // Benchmark configuration
    // --------------------------------------------------------

    constexpr int WARMUP_ROUNDS = 20;
    constexpr int TRIALS = 60;
    constexpr int INNER_REPS = 5;

    // All 6 permutations.
    // Each occurs exactly 10 times over 60 trials.
    const std::array<std::string, 6>
        ORDERS = {

            "A-B-C",
            "A-C-B",
            "B-A-C",
            "B-C-A",
            "C-A-B",
            "C-B-A"
        };

    // --------------------------------------------------------
    // Load layer
    // --------------------------------------------------------

    PackedLayer layer =
        load_layer(binary_path);

    std::cout
        << "\n========================================\n"
        << "FINAL BENCHMARK V3\n"
        << "========================================\n"
        << "Layer        : "
        << layer_name
        << "\n"
        << "Matrix       : "
        << layer.rows
        << " x "
        << layer.original_cols
        << "\n"
        << "Words/row    : "
        << layer.words_per_row
        << "\n"
        << "Warmups      : "
        << WARMUP_ROUNDS
        << "\n"
        << "Trials       : "
        << TRIALS
        << "\n"
        << "Inner reps   : "
        << INNER_REPS
        << "\n"
        << "========================================\n";

    // --------------------------------------------------------
    // Dense weights for Baseline A
    // --------------------------------------------------------

    std::cout
        << "[1] Reconstructing dense INT8 weights...\n";

    std::vector<int8_t> dense_weights =
        reconstruct_dense_weights(layer);

    // --------------------------------------------------------
    // ONE shared activation vector
    // --------------------------------------------------------

    std::cout
        << "[2] Generating shared INT8 activations...\n";

    std::vector<int8_t> activations(
        layer.original_cols
    );

    std::mt19937 rng(42);

    std::uniform_int_distribution<int>
        distribution(-128, 127);

    for (auto& value : activations) {

        value =
            static_cast<int8_t>(
                distribution(rng)
            );
    }

    // --------------------------------------------------------
    // Preallocate initial activation bitplanes
    // --------------------------------------------------------

    std::vector<uint32_t>
        activation_planes[8];

    for (int b = 0;
         b < 8;
         ++b) {

        activation_planes[b].resize(
            layer.words_per_row
        );
    }

    pack_activations(
        activations,
        activation_planes,
        layer.words_per_row
    );

    // --------------------------------------------------------
    // THREE-WAY CORRECTNESS
    // --------------------------------------------------------

    std::cout
        << "[3] Three-way correctness check...\n";

    int64_t output_a =
        baseline_a_dense(
            dense_weights,
            activations,
            layer.rows,
            layer.original_cols
        );

    int64_t output_b =
        baseline_b_unpack(
            layer,
            activations
        );

    int64_t output_c =
        proposed_bitserial(
            layer,
            activation_planes
        );

    std::cout
        << "    A = "
        << output_a
        << "\n";

    std::cout
        << "    B = "
        << output_b
        << "\n";

    std::cout
        << "    C = "
        << output_c
        << "\n";

    if (
        output_a != output_b
        ||
        output_a != output_c
    ) {

        std::cerr
            << "\nFAIL: "
            << "A, B and C outputs do not match.\n";

        return 2;
    }

    std::cout
        << "    PASS: A == B == C\n";

    // --------------------------------------------------------
    // Stronger warm-up
    // --------------------------------------------------------

    std::cout
        << "[4] Running "
        << WARMUP_ROUNDS
        << " warm-up rounds...\n";

    for (int i = 0;
         i < WARMUP_ROUNDS;
         ++i) {

        benchmark_sink +=
            baseline_a_dense(
                dense_weights,
                activations,
                layer.rows,
                layer.original_cols
            );

        benchmark_sink +=
            baseline_b_unpack(
                layer,
                activations
            );

        benchmark_sink +=
            proposed_bitserial(
                layer,
                activation_planes
            );
    }

    // --------------------------------------------------------
    // CSV setup
    // --------------------------------------------------------

    bool csv_exists =
        std::ifstream(csv_path).good();

    std::ofstream csv(
        csv_path,
        std::ios::app
    );

    if (!csv) {

        std::cerr
            << "Could not open CSV: "
            << csv_path
            << "\n";

        return 3;
    }

    if (!csv_exists) {

        csv
            << "layer,"
            << "trial,"
            << "order,"
            << "baseline_a_us,"
            << "baseline_b_us,"
            << "activation_pack_us,"
            << "proposed_kernel_us,"
            << "proposed_e2e_us\n";
    }

    // --------------------------------------------------------
    // V3: Preallocate benchmark activation-plane buffers
    //
    // These buffers are allocated ONCE before measured trials.
    // Neither standalone packing nor E2E timing pays vector
    // allocation/deallocation costs.
    // --------------------------------------------------------

    std::vector<uint32_t>
        trial_planes[8];

    std::vector<uint32_t>
        e2e_planes[8];

    for (int b = 0;
         b < 8;
         ++b) {

        trial_planes[b].resize(
            layer.words_per_row
        );

        e2e_planes[b].resize(
            layer.words_per_row
        );
    }

    // --------------------------------------------------------
    // 60 measured trials
    // --------------------------------------------------------

    std::cout
        << "[5] Starting measured trials...\n";

    for (int trial = 0;
         trial < TRIALS;
         ++trial) {

        const std::string order =
            ORDERS[
                trial % ORDERS.size()
            ];

        double time_a = 0.0;
        double time_b = 0.0;
        double time_c = 0.0;

        // ----------------------------------------------------
        // Activation packing measured independently
        //
        // V3:
        // Uses preallocated trial_planes.
        // No vector allocation occurs in timed region.
        // ----------------------------------------------------

        auto pack_start =
            std::chrono::steady_clock::now();

        for (int rep = 0;
             rep < INNER_REPS;
             ++rep) {

            pack_activations(
                activations,
                trial_planes,
                layer.words_per_row
            );
        }

        auto pack_end =
            std::chrono::steady_clock::now();

        double activation_pack_us =
            std::chrono::duration<
                double,
                std::micro
            >(
                pack_end - pack_start
            ).count()
            / INNER_REPS;

        // ----------------------------------------------------
        // Helpers for A/B/C
        // ----------------------------------------------------

        auto run_a = [&]() {

            time_a =
                time_repeated_us(
                    [&]() {

                        return baseline_a_dense(
                            dense_weights,
                            activations,
                            layer.rows,
                            layer.original_cols
                        );
                    },
                    INNER_REPS
                );
        };

        auto run_b = [&]() {

            time_b =
                time_repeated_us(
                    [&]() {

                        return baseline_b_unpack(
                            layer,
                            activations
                        );
                    },
                    INNER_REPS
                );
        };

        auto run_c = [&]() {

            time_c =
                time_repeated_us(
                    [&]() {

                        return proposed_bitserial(
                            layer,
                            trial_planes
                        );
                    },
                    INNER_REPS
                );
        };

        // ----------------------------------------------------
        // Execute all six balanced permutations
        // ----------------------------------------------------

        if (order == "A-B-C") {

            run_a();
            run_b();
            run_c();
        }

        else if (order == "A-C-B") {

            run_a();
            run_c();
            run_b();
        }

        else if (order == "B-A-C") {

            run_b();
            run_a();
            run_c();
        }

        else if (order == "B-C-A") {

            run_b();
            run_c();
            run_a();
        }

        else if (order == "C-A-B") {

            run_c();
            run_a();
            run_b();
        }

        else if (order == "C-B-A") {

            run_c();
            run_b();
            run_a();
        }

        // ----------------------------------------------------
        // Direct Proposed end-to-end timing
        //
        // V3:
        // Measures ONLY:
        //
        //     activation packing
        //            +
        //     proposed bit-serial kernel
        //
        // e2e_planes are preallocated outside the timed region.
        // ----------------------------------------------------

        int64_t e2e_checksum = 0;

        auto e2e_start =
            std::chrono::steady_clock::now();

        for (int rep = 0;
             rep < INNER_REPS;
             ++rep) {

            pack_activations(
                activations,
                e2e_planes,
                layer.words_per_row
            );

            e2e_checksum +=
                proposed_bitserial(
                    layer,
                    e2e_planes
                );
        }

        auto e2e_end =
            std::chrono::steady_clock::now();

        benchmark_sink +=
            e2e_checksum;

        double proposed_e2e_us =
            std::chrono::duration<
                double,
                std::micro
            >(
                e2e_end - e2e_start
            ).count()
            / INNER_REPS;

        // ----------------------------------------------------
        // Write raw result
        // ----------------------------------------------------

        csv
            << layer_name
            << ","
            << trial
            << ","
            << order
            << ","
            << std::fixed
            << std::setprecision(3)
            << time_a
            << ","
            << time_b
            << ","
            << activation_pack_us
            << ","
            << time_c
            << ","
            << proposed_e2e_us
            << "\n";

        std::cout
            << "Trial "
            << std::setw(2)
            << trial
            << " ["
            << order
            << "] "
            << "A="
            << std::fixed
            << std::setprecision(2)
            << time_a
            << " us  "
            << "B="
            << time_b
            << " us  "
            << "C="
            << time_c
            << " us  "
            << "Pack="
            << activation_pack_us
            << " us  "
            << "E2E="
            << proposed_e2e_us
            << " us\n";
    }

    csv.close();

    std::cout
        << "\n========================================\n"
        << "BENCHMARK V3 COMPLETE\n"
        << "========================================\n"
        << "Layer: "
        << layer_name
        << "\n"
        << "Checksum sink: "
        << benchmark_sink
        << "\n"
        << "CSV: "
        << csv_path
        << "\n";

    return 0;
}

Writing /content/benchmark_v3.cpp


In [ ]:
!g++ -O3 -std=c++17 -march=native /content/benchmark_v3.cpp -o /content/benchmark_v3

In [ ]:
!rm -f /content/benchmark_v3_raw.csv

In [ ]:
!find /content -type f -name "*.bin"

/content/drive/MyDrive/BitNetExperiments/gate_proj_6912x2560.bin
/content/drive/MyDrive/BitNetExperiments/k_proj_640x2560.bin
/content/drive/MyDrive/BitNetExperiments/q_proj_2560x2560.bin


In [ ]:
!/content/benchmark_v3 /content/drive/MyDrive/BitNetExperiments/k_proj_640x2560.bin k_proj_640x2560 /content/benchmark_v3_raw.csv

!/content/benchmark_v3 /content/drive/MyDrive/BitNetExperiments/q_proj_2560x2560.bin q_proj_2560x2560 /content/benchmark_v3_raw.csv

!/content/benchmark_v3 /content/drive/MyDrive/BitNetExperiments/gate_proj_6912x2560.bin gate_proj_6912x2560 /content/benchmark_v3_raw.csv


FINAL BENCHMARK V3
Layer        : k_proj_640x2560
Matrix       : 640 x 2560
Words/row    : 80
Warmups      : 20
Trials       : 60
Inner reps   : 5
[1] Reconstructing dense INT8 weights...
[2] Generating shared INT8 activations...
[3] Three-way correctness check...
    A = -33730
    B = -33730
    C = -33730
    PASS: A == B == C
[4] Running 20 warm-up rounds...
[5] Starting measured trials...
Trial  0 [A-B-C] A=284.47 us  B=4105.97 us  C=193.74 us  Pack=434.93 us  E2E=1866.63 us
Trial  1 [A-C-B] A=282.40 us  B=5175.39 us  C=621.95 us  Pack=504.67 us  E2E=1221.77 us
Trial  2 [B-A-C] A=280.48 us  B=4799.37 us  C=237.35 us  Pack=101.11 us  E2E=1220.67 us
Trial  3 [B-C-A] A=715.34 us  B=5074.47 us  C=626.14 us  Pack=92.39 us  E2E=2265.98 us
Trial  4 [C-A-B] A=741.16 us  B=12917.93 us  C=697.36 us  Pack=93.92 us  E2E=2397.23 us
Trial  5 [C-B-A] A=307.45 us  B=6414.49 us  C=204.06 us  Pack=101.44 us  E2E=1106.67 us
Trial  6 [A-B-C] A=277.85 us  B=4366.72 us  C=234.78 us  Pack=95.12 us  E2E

In [ ]:
import pandas as pd

df = pd.read_csv("/content/benchmark_v3_raw.csv")

print("Total rows:", len(df))
print("\nTrials per layer:")
print(df.groupby("layer").size())

Total rows: 180

Trials per layer:
layer
gate_proj_6912x2560    60
k_proj_640x2560        60
q_proj_2560x2560       60
dtype: int64


In [ ]:
import pandas as pd

df = pd.read_csv("/content/benchmark_v3_raw.csv")

print("Total rows:", len(df))

print("\nTrials per layer:")
print(df.groupby("layer").size())

print("\nTrials per layer and order:")
print(df.groupby(["layer", "order"]).size())

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst rows:")
print(df.head())

Total rows: 180

Trials per layer:
layer
gate_proj_6912x2560    60
k_proj_640x2560        60
q_proj_2560x2560       60
dtype: int64

Trials per layer and order:
layer                order
gate_proj_6912x2560  A-B-C    10
                     A-C-B    10
                     B-A-C    10
                     B-C-A    10
                     C-A-B    10
                     C-B-A    10
k_proj_640x2560      A-B-C    10
                     A-C-B    10
                     B-A-C    10
                     B-C-A    10
                     C-A-B    10
                     C-B-A    10
q_proj_2560x2560     A-B-C    10
                     A-C-B    10
                     B-A-C    10
                     B-C-A    10
                     C-A-B    10
                     C-B-A    10
dtype: int64

Columns:
['layer', 'trial', 'order', 'baseline_a_us', 'baseline_b_us', 'activation_pack_us', 'proposed_kernel_us', 'proposed_e2e_us']

First rows:
             layer  trial  order  baseline_a_us  baseline

In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_csv("/content/benchmark_v3_raw.csv")

# Component total = independently measured pack + kernel
df["proposed_component_total_us"] = (
    df["activation_pack_us"] + df["proposed_kernel_us"]
)

# Per-trial ratios
df["e2e_to_component_ratio"] = (
    df["proposed_e2e_us"] / df["proposed_component_total_us"]
)

# ============================================================
# 1. OVERALL LATENCY STATISTICS
# ============================================================

metrics = [
    "baseline_a_us",
    "baseline_b_us",
    "activation_pack_us",
    "proposed_kernel_us",
    "proposed_component_total_us",
    "proposed_e2e_us",
]

rows = []

for layer, group in df.groupby("layer"):
    for metric in metrics:
        x = group[metric]

        rows.append({
            "layer": layer,
            "metric": metric,
            "mean_us": x.mean(),
            "median_us": x.median(),
            "std_us": x.std(),
            "cv_percent": 100 * x.std() / x.mean(),
            "min_us": x.min(),
            "p25_us": x.quantile(0.25),
            "p75_us": x.quantile(0.75),
            "p95_us": x.quantile(0.95),
            "max_us": x.max(),
        })

stats = pd.DataFrame(rows)

print("\n=== OVERALL LATENCY STATISTICS ===")
display(stats.round(3))


# ============================================================
# 2. MEDIAN-BASED FINAL COMPARISON
# ============================================================

comparison_rows = []

for layer, group in df.groupby("layer"):

    A = group["baseline_a_us"].median()
    B = group["baseline_b_us"].median()
    C = group["proposed_kernel_us"].median()
    P = group["activation_pack_us"].median()

    # Median of per-trial component total, rather than
    # simply assuming median(P) + median(C)
    T = group["proposed_component_total_us"].median()

    E = group["proposed_e2e_us"].median()

    comparison_rows.append({
        "layer": layer,

        "baseline_A_median_us": A,
        "baseline_B_median_us": B,

        "proposed_kernel_median_us": C,
        "activation_pack_median_us": P,
        "component_total_median_us": T,
        "measured_e2e_median_us": E,

        "kernel_speedup_vs_A": A / C,
        "component_speedup_vs_A": A / T,
        "measured_e2e_speedup_vs_A": A / E,

        "kernel_speedup_vs_B": B / C,
        "component_speedup_vs_B": B / T,
        "measured_e2e_speedup_vs_B": B / E,

        "packing_overhead_pct_of_component": (
            100 * P / T
        ),

        "e2e_to_component_ratio": E / T,
    })

comparison = pd.DataFrame(comparison_rows)

print("\n=== MEDIAN-BASED FINAL COMPARISON ===")
display(comparison.round(3))


# ============================================================
# 3. EXECUTION ORDER EFFECT
# ============================================================

order_effect = (
    df.groupby(["layer", "order"])
    .agg(
        baseline_A_median=("baseline_a_us", "median"),
        baseline_B_median=("baseline_b_us", "median"),
        proposed_median=("proposed_kernel_us", "median"),
        pack_median=("activation_pack_us", "median"),
        e2e_median=("proposed_e2e_us", "median"),
        count=("trial", "count"),
    )
    .reset_index()
)

print("\n=== EXECUTION ORDER EFFECT ===")
display(order_effect.round(3))


# ============================================================
# 4. TEMPORAL DRIFT: FIRST 30 VS LAST 30
# ============================================================

df["trial_half"] = np.where(
    df["trial"] < 30,
    "first_30",
    "last_30"
)

half_effect = (
    df.groupby(["layer", "trial_half"])
    .agg(
        A_median=("baseline_a_us", "median"),
        B_median=("baseline_b_us", "median"),
        C_median=("proposed_kernel_us", "median"),
        Pack_median=("activation_pack_us", "median"),
        E2E_median=("proposed_e2e_us", "median"),
    )
    .reset_index()
)

print("\n=== FIRST 30 VS LAST 30 ===")
display(half_effect.round(3))


# ============================================================
# 5. QUARTER-OF-RUN TEMPORAL ANALYSIS
#
# More detailed view of performance regime changes.
# ============================================================

df["trial_block"] = pd.cut(
    df["trial"],
    bins=[-1, 14, 29, 44, 59],
    labels=[
        "trials_0_14",
        "trials_15_29",
        "trials_30_44",
        "trials_45_59"
    ]
)

block_effect = (
    df.groupby(
        ["layer", "trial_block"],
        observed=True
    )
    .agg(
        A_median=("baseline_a_us", "median"),
        B_median=("baseline_b_us", "median"),
        C_median=("proposed_kernel_us", "median"),
        Pack_median=("activation_pack_us", "median"),
        E2E_median=("proposed_e2e_us", "median"),
    )
    .reset_index()
)

print("\n=== TEMPORAL BLOCK ANALYSIS ===")
display(block_effect.round(3))


# ============================================================
# 6. E2E CONSISTENCY CHECK
# ============================================================

e2e_rows = []

for layer, group in df.groupby("layer"):

    component = group[
        "proposed_component_total_us"
    ].median()

    measured = group[
        "proposed_e2e_us"
    ].median()

    ratios = group[
        "e2e_to_component_ratio"
    ]

    e2e_rows.append({
        "layer": layer,

        "component_total_median_us":
            component,

        "measured_e2e_median_us":
            measured,

        "ratio_of_medians":
            measured / component,

        "median_per_trial_e2e_ratio":
            ratios.median(),

        "min_per_trial_ratio":
            ratios.min(),

        "max_per_trial_ratio":
            ratios.max(),
    })

e2e_check = pd.DataFrame(e2e_rows)

print("\n=== E2E CONSISTENCY CHECK ===")
display(e2e_check.round(3))


# ============================================================
# 7. MEDIAN SPEEDUP SUMMARY
# ============================================================

summary = comparison[[
    "layer",
    "kernel_speedup_vs_A",
    "component_speedup_vs_A",
    "measured_e2e_speedup_vs_A",
    "kernel_speedup_vs_B",
    "component_speedup_vs_B",
    "measured_e2e_speedup_vs_B",
    "packing_overhead_pct_of_component",
    "e2e_to_component_ratio",
]]

print("\n=== FINAL SPEEDUP SUMMARY ===")
display(summary.round(3))


=== OVERALL LATENCY STATISTICS ===


,layer,metric,mean_us,median_us,std_us,cv_percent,min_us,p25_us,p75_us,p95_us,max_us
0,gate_proj_6912x2560,baseline_a_us,3306.983,3111.024,454.266,13.737,2970.239,3068.102,3316.397,4177.351,5370.004
1,gate_proj_6912x2560,baseline_b_us,33379.940,27573.783,9529.072,28.547,26567.902,26886.276,45048.402,49738.701,53324.491
2,gate_proj_6912x2560,activation_pack_us,82.151,76.944,10.485,12.763,73.422,75.181,90.535,103.305,115.270
3,gate_proj_6912x2560,proposed_kernel_us,1835.595,1494.918,612.451,33.365,1453.522,1470.602,1882.707,2940.114,3709.865
4,gate_proj_6912x2560,proposed_component_total_us,1917.746,1570.171,621.625,32.414,1529.343,1547.072,1962.347,3043.325,3811.415
5,gate_proj_6912x2560,proposed_e2e_us,9445.240,7475.229,4147.971,43.916,7358.397,7433.755,10640.772,15172.118,33187.834
6,k_proj_640x2560,baseline_a_us,310.351,260.288,196.380,63.277,234.536,248.449,279.689,716.629,1371.318
7,k_proj_640x2560,baseline_b_us,3339.837,2461.326,1783.667,53.406,2428.755,2441.176,4113.160,6420.594,12917.932
8,k_proj_640x2560,activation_pack_us,94.330,75.391,71.429,75.722,71.892,73.694,94.810,102.955,504.666
9,k_proj_640x2560,proposed_kernel_us,183.456,136.384,117.007,63.779,131.702,134.064,181.346,357.173,697.356



=== MEDIAN-BASED FINAL COMPARISON ===


,layer,baseline_A_median_us,baseline_B_median_us,proposed_kernel_median_us,activation_pack_median_us,component_total_median_us,measured_e2e_median_us,kernel_speedup_vs_A,component_speedup_vs_A,measured_e2e_speedup_vs_A,kernel_speedup_vs_B,component_speedup_vs_B,measured_e2e_speedup_vs_B,packing_overhead_pct_of_component,e2e_to_component_ratio
0,gate_proj_6912x2560,3111.024,27573.783,1494.918,76.944,1570.171,7475.229,2.081,1.981,0.416,18.445,17.561,3.689,4.900,4.761
1,k_proj_640x2560,260.288,2461.326,136.384,75.391,211.110,750.108,1.908,1.233,0.347,18.047,11.659,3.281,35.712,3.553
2,q_proj_2560x2560,1155.454,17089.234,936.652,91.674,1042.709,4649.790,1.234,1.108,0.248,18.245,16.389,3.675,8.792,4.459



=== EXECUTION ORDER EFFECT ===


...

,layer,trial_block,A_median,B_median,C_median,Pack_median,E2E_median
0,gate_proj_6912x2560,trials_0_14,3090.416,27023.698,1480.243,75.441,7414.967
1,gate_proj_6912x2560,trials_15_29,3065.678,27332.891,1482.263,75.663,7473.580
2,gate_proj_6912x2560,trials_30_44,3380.444,48198.947,2610.336,93.159,13236.877
3,gate_proj_6912x2560,trials_45_59,3184.489,27380.262,1496.830,78.080,7572.528
4,k_proj_640x2560,trials_0_14,287.637,4799.368,234.776,98.865,1612.987
5,k_proj_640x2560,trials_15_29,259.837,2456.361,133.808,73.984,746.556
6,k_proj_640x2560,trials_30_44,253.310,2445.044,134.497,74.084,746.283
7,k_proj_640x2560,trials_45_59,255.419,2460.044,136.572,75.352,747.632
8,q_proj_2560x2560,trials_0_14,1220.741,18377.389,1029.677,94.242,4984.297
9,q_proj_2560x2560,trials_15_29,1285.669,18314.987,1020.365,99.996,5020.208



=== E2E CONSISTENCY CHECK ===


,layer,component_total_median_us,measured_e2e_median_us,ratio_of_medians,median_per_trial_e2e_ratio,min_per_trial_ratio,max_per_trial_ratio
0,gate_proj_6912x2560,1570.171,7475.229,4.761,4.789,2.193,21.371
1,k_proj_640x2560,211.110,750.108,3.553,3.582,1.084,8.651
2,q_proj_2560x2560,1042.709,4649.790,4.459,4.570,3.860,6.938



=== FINAL SPEEDUP SUMMARY ===


,layer,kernel_speedup_vs_A,component_speedup_vs_A,measured_e2e_speedup_vs_A,kernel_speedup_vs_B,component_speedup_vs_B,measured_e2e_speedup_vs_B,packing_overhead_pct_of_component,e2e_to_component_ratio
0,gate_proj_6912x2560,2.081,1.981,0.416,18.445,17.561,3.689,4.900,4.761
1,k_proj_640x2560,1.908,1.233,0.347,18.047,11.659,3.281,35.712,3.553
2,q_proj_2560x2560,1.234,1.108,0.248,18.245,16.389,3.675,8.792,4.459


In [ ]:
%%writefile /content/benchmark_v4.cpp

#include <algorithm>
#include <array>
#include <chrono>
#include <cstdint>
#include <fstream>
#include <iomanip>
#include <iostream>
#include <numeric>
#include <random>
#include <stdexcept>
#include <string>
#include <vector>

#if defined(_MSC_VER)
#include <intrin.h>
#define POPCOUNT32 __popcnt
#else
#define POPCOUNT32 __builtin_popcount
#endif

volatile int64_t benchmark_sink = 0;

struct PackedLayer {
    uint32_t rows;
    uint32_t words_per_row;
    uint32_t original_cols;

    std::vector<uint32_t> pos;
    std::vector<uint32_t> neg;
};

// ============================================================
// Load packed layer
// ============================================================

PackedLayer load_layer(const std::string& filename) {

    std::ifstream file(filename, std::ios::binary);

    if (!file) {
        throw std::runtime_error(
            "Could not open: " + filename
        );
    }

    PackedLayer layer;

    file.read(
        reinterpret_cast<char*>(&layer.rows),
        sizeof(uint32_t)
    );

    file.read(
        reinterpret_cast<char*>(&layer.words_per_row),
        sizeof(uint32_t)
    );

    file.read(
        reinterpret_cast<char*>(&layer.original_cols),
        sizeof(uint32_t)
    );

    if (!file) {
        throw std::runtime_error(
            "Failed to read header."
        );
    }

    const size_t total_words =
        static_cast<size_t>(layer.rows)
        * layer.words_per_row;

    layer.pos.resize(total_words);
    layer.neg.resize(total_words);

    file.read(
        reinterpret_cast<char*>(layer.pos.data()),
        total_words * sizeof(uint32_t)
    );

    file.read(
        reinterpret_cast<char*>(layer.neg.data()),
        total_words * sizeof(uint32_t)
    );

    if (!file) {
        throw std::runtime_error(
            "Failed to read packed weight data."
        );
    }

    return layer;
}

// ============================================================
// Reconstruct dense INT8 ternary weights
//
// Done ONCE before benchmarking.
// Baseline A does not pay reconstruction cost.
// ============================================================

std::vector<int8_t> reconstruct_dense_weights(
    const PackedLayer& layer
) {

    const size_t total_elements =
        static_cast<size_t>(layer.rows)
        * layer.original_cols;

    std::vector<int8_t> dense(total_elements);

    for (uint32_t r = 0; r < layer.rows; ++r) {

        const size_t packed_offset =
            static_cast<size_t>(r)
            * layer.words_per_row;

        const size_t dense_offset =
            static_cast<size_t>(r)
            * layer.original_cols;

        for (uint32_t c = 0;
             c < layer.original_cols;
             ++c) {

            const uint32_t word = c / 32;
            const uint32_t bit = c % 32;

            const uint32_t p =
                layer.pos[packed_offset + word];

            const uint32_t n =
                layer.neg[packed_offset + word];

            const int32_t weight =
                static_cast<int32_t>(
                    (p >> bit) & 1U
                )
                -
                static_cast<int32_t>(
                    (n >> bit) & 1U
                );

            dense[dense_offset + c] =
                static_cast<int8_t>(weight);
        }
    }

    return dense;
}

// ============================================================
// BASELINE A
//
// Dense INT8 ternary weights.
// Conventional scalar multiply-accumulate.
//
// Important:
// Dense reconstruction happens outside timed region.
// ============================================================

int64_t baseline_a_dense(
    const std::vector<int8_t>& weights,
    const std::vector<int8_t>& activations,
    uint32_t rows,
    uint32_t cols
) {

    int64_t checksum = 0;

    for (uint32_t r = 0; r < rows; ++r) {

        int32_t acc = 0;

        const size_t offset =
            static_cast<size_t>(r) * cols;

        for (uint32_t c = 0; c < cols; ++c) {

            acc +=
                static_cast<int32_t>(
                    weights[offset + c]
                )
                *
                static_cast<int32_t>(
                    activations[c]
                );
        }

        checksum += acc;
    }

    return checksum;
}

// ============================================================
// BASELINE B
//
// Packed ternary weights with scalar on-the-fly decoding.
//
// This baseline intentionally measures the cost of decoding
// each ternary weight inside the arithmetic loop.
// ============================================================

int64_t baseline_b_unpack(
    const PackedLayer& layer,
    const std::vector<int8_t>& activations
) {

    int64_t checksum = 0;

    for (uint32_t r = 0;
         r < layer.rows;
         ++r) {

        int32_t acc = 0;

        const size_t offset =
            static_cast<size_t>(r)
            * layer.words_per_row;

        for (uint32_t c = 0;
             c < layer.original_cols;
             ++c) {

            const uint32_t word = c / 32;
            const uint32_t bit = c % 32;

            const uint32_t p =
                layer.pos[offset + word];

            const uint32_t n =
                layer.neg[offset + word];

            const int32_t weight =
                static_cast<int32_t>(
                    (p >> bit) & 1U
                )
                -
                static_cast<int32_t>(
                    (n >> bit) & 1U
                );

            acc +=
                weight
                *
                static_cast<int32_t>(
                    activations[c]
                );
        }

        checksum += acc;
    }

    return checksum;
}

// ============================================================
// Preallocate activation bitplanes
//
// This is done OUTSIDE timed regions.
// ============================================================

void allocate_activation_planes(
    std::array<std::vector<uint32_t>, 8>& planes,
    uint32_t words_per_row
) {

    for (auto& plane : planes) {
        plane.resize(words_per_row);
    }
}

// ============================================================
// Activation packing
//
// INT8 activations -> 8 bitplanes.
//
// IMPORTANT V4 CHANGE:
// - No vector assign()
// - No resizing
// - No allocation
//
// Buffers must already be allocated.
// ============================================================

void pack_activations_reuse(
    const std::vector<int8_t>& activations,
    std::array<std::vector<uint32_t>, 8>& planes
) {

    for (auto& plane : planes) {

        std::fill(
            plane.begin(),
            plane.end(),
            0U
        );
    }

    for (uint32_t c = 0;
         c < activations.size();
         ++c) {

        const uint8_t value =
            static_cast<uint8_t>(
                activations[c]
            );

        const uint32_t word = c / 32;
        const uint32_t bit = c % 32;

        const uint32_t mask =
            1U << bit;

        for (int b = 0; b < 8; ++b) {

            if ((value >> b) & 1U) {

                planes[b][word] |= mask;
            }
        }
    }
}

// ============================================================
// PROPOSED BIT-SERIAL KERNEL
//
// Ternary weights represented by:
//   pos mask
//   neg mask
//
// INT8 activations represented by 8 bitplanes.
//
// For each activation bitplane:
//
// popcount(pos & activation_bits)
// -
// popcount(neg & activation_bits)
//
// Two's-complement bit significance:
// bits 0-6 -> +1,+2,+4,...,+64
// bit 7    -> -128
// ============================================================

int64_t proposed_bitserial(
    const PackedLayer& layer,
    const std::array<std::vector<uint32_t>, 8>& planes
) {

    int64_t checksum = 0;

    for (uint32_t r = 0;
         r < layer.rows;
         ++r) {

        int32_t acc = 0;

        const size_t offset =
            static_cast<size_t>(r)
            * layer.words_per_row;

        for (int b = 0; b < 8; ++b) {

            int32_t plane_acc = 0;

            const int32_t scale =
                (b == 7)
                ? -128
                : (1 << b);

            for (uint32_t w = 0;
                 w < layer.words_per_row;
                 ++w) {

                const uint32_t activation_bits =
                    planes[b][w];

                plane_acc +=
                    POPCOUNT32(
                        layer.pos[offset + w]
                        & activation_bits
                    );

                plane_acc -=
                    POPCOUNT32(
                        layer.neg[offset + w]
                        & activation_bits
                    );
            }

            acc +=
                plane_acc * scale;
        }

        checksum += acc;
    }

    return checksum;
}

// ============================================================
// Generic repeated timing helper
//
// Returns average latency per invocation.
// ============================================================

template <typename Func>
double time_repeated_us(
    Func&& function,
    int repetitions
) {

    int64_t local_checksum = 0;

    const auto start =
        std::chrono::steady_clock::now();

    for (int i = 0;
         i < repetitions;
         ++i) {

        local_checksum += function();
    }

    const auto end =
        std::chrono::steady_clock::now();

    benchmark_sink += local_checksum;

    const double total_us =
        std::chrono::duration<
            double,
            std::micro
        >(end - start).count();

    return total_us / repetitions;
}

// ============================================================
// Packing-only timing helper
// ============================================================

double time_pack_repeated_us(
    const std::vector<int8_t>& activations,
    std::array<std::vector<uint32_t>, 8>& planes,
    int repetitions
) {

    const auto start =
        std::chrono::steady_clock::now();

    for (int i = 0;
         i < repetitions;
         ++i) {

        pack_activations_reuse(
            activations,
            planes
        );
    }

    const auto end =
        std::chrono::steady_clock::now();

    const double total_us =
        std::chrono::duration<
            double,
            std::micro
        >(end - start).count();

    // Touch result to discourage elimination.
    benchmark_sink +=
        static_cast<int64_t>(
            planes[0][0]
        );

    return total_us / repetitions;
}

// ============================================================
// Direct proposed E2E timing
//
// IMPORTANT:
//
// Measures ONLY:
//
//     activation packing
//            +
//     proposed bit-serial kernel
//
// All bitplane memory is preallocated before timing.
// No vector construction/destruction/allocation occurs
// inside the repeated timed loop.
// ============================================================

double time_proposed_e2e_us(
    const PackedLayer& layer,
    const std::vector<int8_t>& activations,
    std::array<std::vector<uint32_t>, 8>& planes,
    int repetitions
) {

    int64_t local_checksum = 0;

    const auto start =
        std::chrono::steady_clock::now();

    for (int i = 0;
         i < repetitions;
         ++i) {

        pack_activations_reuse(
            activations,
            planes
        );

        local_checksum +=
            proposed_bitserial(
                layer,
                planes
            );
    }

    const auto end =
        std::chrono::steady_clock::now();

    benchmark_sink += local_checksum;

    const double total_us =
        std::chrono::duration<
            double,
            std::micro
        >(end - start).count();

    return total_us / repetitions;
}

// ============================================================
// Main
// ============================================================

int main(
    int argc,
    char* argv[]
) {

    if (argc != 4) {

        std::cerr
            << "Usage:\n"
            << "./benchmark_v4 "
            << "<layer.bin> "
            << "<layer_name> "
            << "<output.csv>\n";

        return 1;
    }

    const std::string binary_path =
        argv[1];

    const std::string layer_name =
        argv[2];

    const std::string csv_path =
        argv[3];

    // ========================================================
    // Benchmark configuration
    // ========================================================

    constexpr int WARMUP_ROUNDS = 30;

    constexpr int TRIALS = 60;

    // Increased from V3 to reduce timer noise.
    constexpr int INNER_REPS = 10;

    // Six balanced permutations.
    const std::array<std::string, 6>
        BASE_ORDERS = {

            "A-B-C",
            "A-C-B",
            "B-A-C",
            "B-C-A",
            "C-A-B",
            "C-B-A"
        };

    // ========================================================
    // Build balanced randomized trial order
    //
    // Each permutation occurs exactly 10 times.
    // Deterministically shuffled for reproducibility.
    // ========================================================

    std::vector<std::string> trial_orders;

    trial_orders.reserve(TRIALS);

    for (int repeat = 0;
         repeat < TRIALS / 6;
         ++repeat) {

        for (const auto& order :
             BASE_ORDERS) {

            trial_orders.push_back(order);
        }
    }

    std::mt19937 order_rng(2026);

    std::shuffle(
        trial_orders.begin(),
        trial_orders.end(),
        order_rng
    );

    // ========================================================
    // Load layer
    // ========================================================

    PackedLayer layer =
        load_layer(binary_path);

    std::cout
        << "\n========================================\n"
        << "FINAL BENCHMARK V4\n"
        << "========================================\n"
        << "Layer        : "
        << layer_name
        << "\n"
        << "Matrix       : "
        << layer.rows
        << " x "
        << layer.original_cols
        << "\n"
        << "Words/row    : "
        << layer.words_per_row
        << "\n"
        << "Warmups      : "
        << WARMUP_ROUNDS
        << "\n"
        << "Trials       : "
        << TRIALS
        << "\n"
        << "Inner reps   : "
        << INNER_REPS
        << "\n"
        << "========================================\n";

    // ========================================================
    // Dense weights for Baseline A
    // ========================================================

    std::cout
        << "[1] Reconstructing dense INT8 weights...\n";

    std::vector<int8_t> dense_weights =
        reconstruct_dense_weights(layer);

    // ========================================================
    // Shared activation vector
    // ========================================================

    std::cout
        << "[2] Generating shared INT8 activations...\n";

    std::vector<int8_t> activations(
        layer.original_cols
    );

    std::mt19937 activation_rng(42);

    std::uniform_int_distribution<int>
        distribution(-128, 127);

    for (auto& value : activations) {

        value =
            static_cast<int8_t>(
                distribution(
                    activation_rng
                )
            );
    }

    // ========================================================
    // Preallocate ALL activation plane buffers
    // ========================================================

    std::array<
        std::vector<uint32_t>,
        8
    > kernel_planes;

    std::array<
        std::vector<uint32_t>,
        8
    > pack_planes;

    std::array<
        std::vector<uint32_t>,
        8
    > e2e_planes;

    allocate_activation_planes(
        kernel_planes,
        layer.words_per_row
    );

    allocate_activation_planes(
        pack_planes,
        layer.words_per_row
    );

    allocate_activation_planes(
        e2e_planes,
        layer.words_per_row
    );

    pack_activations_reuse(
        activations,
        kernel_planes
    );

    // ========================================================
    // THREE-WAY CORRECTNESS
    // ========================================================

    std::cout
        << "[3] Three-way correctness check...\n";

    const int64_t output_a =
        baseline_a_dense(
            dense_weights,
            activations,
            layer.rows,
            layer.original_cols
        );

    const int64_t output_b =
        baseline_b_unpack(
            layer,
            activations
        );

    const int64_t output_c =
        proposed_bitserial(
            layer,
            kernel_planes
        );

    std::cout
        << "    A = "
        << output_a
        << "\n"
        << "    B = "
        << output_b
        << "\n"
        << "    C = "
        << output_c
        << "\n";

    if (
        output_a != output_b
        ||
        output_a != output_c
    ) {

        std::cerr
            << "\nFAIL: Outputs do not match.\n";

        return 2;
    }

    std::cout
        << "    PASS: A == B == C\n";

    // ========================================================
    // Warm-up
    //
    // Warm all relevant paths including packing and E2E.
    // ========================================================

    std::cout
        << "[4] Running "
        << WARMUP_ROUNDS
        << " warm-up rounds...\n";

    for (int i = 0;
         i < WARMUP_ROUNDS;
         ++i) {

        benchmark_sink +=
            baseline_a_dense(
                dense_weights,
                activations,
                layer.rows,
                layer.original_cols
            );

        benchmark_sink +=
            baseline_b_unpack(
                layer,
                activations
            );

        pack_activations_reuse(
            activations,
            kernel_planes
        );

        benchmark_sink +=
            proposed_bitserial(
                layer,
                kernel_planes
            );

        pack_activations_reuse(
            activations,
            e2e_planes
        );

        benchmark_sink +=
            proposed_bitserial(
                layer,
                e2e_planes
            );
    }

    // ========================================================
    // CSV setup
    // ========================================================

    const bool csv_exists =
        std::ifstream(csv_path).good();

    std::ofstream csv(
        csv_path,
        std::ios::app
    );

    if (!csv) {

        std::cerr
            << "Could not open CSV: "
            << csv_path
            << "\n";

        return 3;
    }

    if (!csv_exists) {

        csv
            << "layer,"
            << "trial,"
            << "order,"
            << "baseline_a_us,"
            << "baseline_b_us,"
            << "activation_pack_us,"
            << "proposed_kernel_us,"
            << "proposed_component_sum_us,"
            << "proposed_e2e_us,"
            << "e2e_to_component_ratio\n";
    }

    // ========================================================
    // Measured trials
    // ========================================================

    std::cout
        << "[5] Starting measured trials...\n";

    for (int trial = 0;
         trial < TRIALS;
         ++trial) {

        const std::string& order =
            trial_orders[trial];

        double time_a = 0.0;
        double time_b = 0.0;
        double time_c = 0.0;

        // ----------------------------------------------------
        // Packing-only measurement
        //
        // Uses preallocated buffer.
        // ----------------------------------------------------

        const double activation_pack_us =
            time_pack_repeated_us(
                activations,
                pack_planes,
                INNER_REPS
            );

        // ----------------------------------------------------
        // A/B/C measurement helpers
        // ----------------------------------------------------

        auto run_a = [&]() {

            time_a =
                time_repeated_us(
                    [&]() {

                        return baseline_a_dense(
                            dense_weights,
                            activations,
                            layer.rows,
                            layer.original_cols
                        );
                    },
                    INNER_REPS
                );
        };

        auto run_b = [&]() {

            time_b =
                time_repeated_us(
                    [&]() {

                        return baseline_b_unpack(
                            layer,
                            activations
                        );
                    },
                    INNER_REPS
                );
        };

        auto run_c = [&]() {

            // Ensure C receives already packed activation
            // planes. Packing is NOT included here.
            pack_activations_reuse(
                activations,
                kernel_planes
            );

            time_c =
                time_repeated_us(
                    [&]() {

                        return proposed_bitserial(
                            layer,
                            kernel_planes
                        );
                    },
                    INNER_REPS
                );
        };

        // ----------------------------------------------------
        // Execute balanced randomized order
        // ----------------------------------------------------

        if (order == "A-B-C") {

            run_a();
            run_b();
            run_c();
        }

        else if (order == "A-C-B") {

            run_a();
            run_c();
            run_b();
        }

        else if (order == "B-A-C") {

            run_b();
            run_a();
            run_c();
        }

        else if (order == "B-C-A") {

            run_b();
            run_c();
            run_a();
        }

        else if (order == "C-A-B") {

            run_c();
            run_a();
            run_b();
        }

        else if (order == "C-B-A") {

            run_c();
            run_b();
            run_a();
        }

        // ----------------------------------------------------
        // Component estimate
        //
        // Separately measured:
        // pack median/sample + kernel sample
        // ----------------------------------------------------

        const double proposed_component_sum_us =
            activation_pack_us
            +
            time_c;

        // ----------------------------------------------------
        // Direct E2E
        //
        // Preallocated buffers.
        //
        // Timed operation:
        //
        // pack activation
        // +
        // proposed kernel
        // ----------------------------------------------------

        const double proposed_e2e_us =
            time_proposed_e2e_us(
                layer,
                activations,
                e2e_planes,
                INNER_REPS
            );

        const double e2e_to_component_ratio =
            proposed_component_sum_us > 0.0
            ? proposed_e2e_us
                / proposed_component_sum_us
            : 0.0;

        // ----------------------------------------------------
        // Write raw result
        // ----------------------------------------------------

        csv
            << layer_name
            << ","
            << trial
            << ","
            << order
            << ","
            << std::fixed
            << std::setprecision(3)
            << time_a
            << ","
            << time_b
            << ","
            << activation_pack_us
            << ","
            << time_c
            << ","
            << proposed_component_sum_us
            << ","
            << proposed_e2e_us
            << ","
            << std::setprecision(6)
            << e2e_to_component_ratio
            << "\n";

        std::cout
            << "Trial "
            << std::setw(2)
            << trial
            << " ["
            << order
            << "] "
            << std::fixed
            << std::setprecision(2)
            << "A="
            << time_a
            << " us  "
            << "B="
            << time_b
            << " us  "
            << "C="
            << time_c
            << " us  "
            << "Pack="
            << activation_pack_us
            << " us  "
            << "Component="
            << proposed_component_sum_us
            << " us  "
            << "E2E="
            << proposed_e2e_us
            << " us  "
            << "Ratio="
            << std::setprecision(3)
            << e2e_to_component_ratio
            << "\n";
    }

    csv.close();

    std::cout
        << "\n========================================\n"
        << "BENCHMARK V4 COMPLETE\n"
        << "========================================\n"
        << "Layer: "
        << layer_name
        << "\n"
        << "Checksum sink: "
        << benchmark_sink
        << "\n"
        << "CSV: "
        << csv_path
        << "\n";

    return 0;
}

Writing /content/benchmark_v4.cpp


In [ ]:
!g++ -O3 -std=c++17 -march=native /content/benchmark_v4.cpp -o /content/benchmark_v4

In [ ]:
!rm -f /content/benchmark_v4_raw.csv

In [ ]:
!/content/benchmark_v4 \
"/content/drive/MyDrive/BitNetExperiments/k_proj_640x2560.bin" \
"k_proj_640x2560" \
"/content/benchmark_v4_raw.csv"

!/content/benchmark_v4 \
"/content/drive/MyDrive/BitNetExperiments/q_proj_2560x2560.bin" \
"q_proj_2560x2560" \
"/content/benchmark_v4_raw.csv"

!/content/benchmark_v4 \
"/content/drive/MyDrive/BitNetExperiments/gate_proj_6912x2560.bin" \
"gate_proj_6912x2560" \
"/content/benchmark_v4_raw.csv"


FINAL BENCHMARK V4
Layer        : k_proj_640x2560
Matrix       : 640 x 2560
Words/row    : 80
Warmups      : 30
Trials       : 60
Inner reps   : 10
[1] Reconstructing dense INT8 weights...
[2] Generating shared INT8 activations...
[3] Three-way correctness check...
    A = -33730
    B = -33730
    C = -33730
    PASS: A == B == C
[4] Running 30 warm-up rounds...
[5] Starting measured trials...
Trial  0 [A-C-B] A=271.96 us  B=6122.40 us  C=82.93 us  Pack=94.38 us  Component=177.30 us  E2E=1890.95 us  Ratio=10.665
Trial  1 [A-B-C] A=470.21 us  B=4473.40 us  C=78.57 us  Pack=91.68 us  Component=170.24 us  E2E=867.30 us  Ratio=5.094
Trial  2 [B-A-C] A=274.97 us  B=6257.65 us  C=83.43 us  Pack=89.20 us  Component=172.63 us  E2E=959.14 us  Ratio=5.556
Trial  3 [A-B-C] A=290.04 us  B=9092.10 us  C=79.30 us  Pack=90.68 us  Component=169.99 us  E2E=1880.86 us  Ratio=11.065
Trial  4 [A-B-C] A=679.91 us  B=12756.74 us  C=78.48 us  Pack=92.93 us  Component=171.41 us  E2E=3471.18 us  Ratio=20.251

In [ ]:
import pandas as pd

df = pd.read_csv("/content/benchmark_v4_raw.csv")

print("Total rows:", len(df))

print("\nTrials per layer:")
print(df.groupby("layer").size())

print("\nTrials per layer and order:")
print(df.groupby(["layer", "order"]).size())

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst rows:")
display(df.head())

print("\nE2E / component ratio:")
display(
    df.groupby("layer")["e2e_to_component_ratio"]
      .agg(["mean", "median", "min", "max"])
)

Total rows: 180

Trials per layer:
layer
gate_proj_6912x2560    60
k_proj_640x2560        60
q_proj_2560x2560       60
dtype: int64

Trials per layer and order:
layer                order
gate_proj_6912x2560  A-B-C    10
                     A-C-B    10
                     B-A-C    10
                     B-C-A    10
                     C-A-B    10
                     C-B-A    10
k_proj_640x2560      A-B-C    10
                     A-C-B    10
                     B-A-C    10
                     B-C-A    10
                     C-A-B    10
                     C-B-A    10
q_proj_2560x2560     A-B-C    10
                     A-C-B    10
                     B-A-C    10
                     B-C-A    10
                     C-A-B    10
                     C-B-A    10
dtype: int64

Columns:
['layer', 'trial', 'order', 'baseline_a_us', 'baseline_b_us', 'activation_pack_us', 'proposed_kernel_us', 'proposed_component_sum_us', 'proposed_e2e_us', 'e2e_to_component_ratio']

First rows:


,layer,trial,order,baseline_a_us,baseline_b_us,activation_pack_us,proposed_kernel_us,proposed_component_sum_us,proposed_e2e_us,e2e_to_component_ratio
0,k_proj_640x2560,0,A-C-B,271.957,6122.397,94.377,82.927,177.305,1890.949,10.664972
1,k_proj_640x2560,1,A-B-C,470.209,4473.402,91.677,78.567,170.244,867.300,5.094459
2,k_proj_640x2560,2,B-A-C,274.975,6257.646,89.203,83.430,172.632,959.144,5.555994
3,k_proj_640x2560,3,A-B-C,290.036,9092.103,90.683,79.303,169.986,1880.859,11.064782
4,k_proj_640x2560,4,A-B-C,679.907,12756.740,92.927,78.484,171.410,3471.182,20.250756



E2E / component ratio:


,mean,median,min,max
layer,,,,
gate_proj_6912x2560,9.035204,8.954920,4.235740,15.748306
k_proj_640x2560,5.366047,4.823379,1.007698,20.250756
q_proj_2560x2560,7.836243,7.700387,4.442628,12.510987


In [ ]:
%%writefile /content/benchmark_v5.cpp

#include <algorithm>
#include <array>
#include <chrono>
#include <cstdint>
#include <fstream>
#include <iomanip>
#include <iostream>
#include <numeric>
#include <random>
#include <stdexcept>
#include <string>
#include <vector>

#if defined(_MSC_VER)
#include <intrin.h>
#define POPCOUNT32 __popcnt
#define NOINLINE __declspec(noinline)
#elif defined(__GNUC__) || defined(__clang__)
#define POPCOUNT32 __builtin_popcount
#define NOINLINE __attribute__((noinline))
#else
#define POPCOUNT32 __builtin_popcount
#define NOINLINE
#endif

volatile int64_t benchmark_sink = 0;

// ============================================================
// Packed ternary layer
// ============================================================

struct PackedLayer {
    uint32_t rows;
    uint32_t words_per_row;
    uint32_t original_cols;

    std::vector<uint32_t> pos;
    std::vector<uint32_t> neg;
};

// ============================================================
// Load packed layer
// ============================================================

PackedLayer load_layer(const std::string& filename) {

    std::ifstream file(filename, std::ios::binary);

    if (!file) {
        throw std::runtime_error(
            "Could not open: " + filename
        );
    }

    PackedLayer layer;

    file.read(
        reinterpret_cast<char*>(&layer.rows),
        sizeof(uint32_t)
    );

    file.read(
        reinterpret_cast<char*>(&layer.words_per_row),
        sizeof(uint32_t)
    );

    file.read(
        reinterpret_cast<char*>(&layer.original_cols),
        sizeof(uint32_t)
    );

    if (!file) {
        throw std::runtime_error(
            "Failed to read header."
        );
    }

    const size_t total_words =
        static_cast<size_t>(layer.rows)
        * layer.words_per_row;

    layer.pos.resize(total_words);
    layer.neg.resize(total_words);

    file.read(
        reinterpret_cast<char*>(layer.pos.data()),
        total_words * sizeof(uint32_t)
    );

    file.read(
        reinterpret_cast<char*>(layer.neg.data()),
        total_words * sizeof(uint32_t)
    );

    if (!file) {
        throw std::runtime_error(
            "Failed to read packed weight data."
        );
    }

    return layer;
}

// ============================================================
// Reconstruct dense INT8 ternary weights
//
// Performed once outside benchmark.
// ============================================================

std::vector<int8_t> reconstruct_dense_weights(
    const PackedLayer& layer
) {

    const size_t total_elements =
        static_cast<size_t>(layer.rows)
        * layer.original_cols;

    std::vector<int8_t> dense(total_elements);

    for (uint32_t r = 0; r < layer.rows; ++r) {

        const size_t packed_offset =
            static_cast<size_t>(r)
            * layer.words_per_row;

        const size_t dense_offset =
            static_cast<size_t>(r)
            * layer.original_cols;

        for (uint32_t c = 0;
             c < layer.original_cols;
             ++c) {

            const uint32_t word = c / 32;
            const uint32_t bit = c % 32;

            const uint32_t p =
                layer.pos[packed_offset + word];

            const uint32_t n =
                layer.neg[packed_offset + word];

            const int32_t weight =
                static_cast<int32_t>(
                    (p >> bit) & 1U
                )
                -
                static_cast<int32_t>(
                    (n >> bit) & 1U
                );

            dense[dense_offset + c] =
                static_cast<int8_t>(weight);
        }
    }

    return dense;
}

// ============================================================
// BASELINE A
//
// Dense INT8 ternary weights.
// Scalar multiply-accumulate.
// ============================================================

NOINLINE int64_t baseline_a_dense(
    const std::vector<int8_t>& weights,
    const std::vector<int8_t>& activations,
    uint32_t rows,
    uint32_t cols
) {

    int64_t checksum = 0;

    for (uint32_t r = 0; r < rows; ++r) {

        int32_t acc = 0;

        const size_t offset =
            static_cast<size_t>(r) * cols;

        for (uint32_t c = 0; c < cols; ++c) {

            acc +=
                static_cast<int32_t>(
                    weights[offset + c]
                )
                *
                static_cast<int32_t>(
                    activations[c]
                );
        }

        checksum += acc;
    }

    return checksum;
}

// ============================================================
// BASELINE B
//
// Packed ternary weights.
// Scalar on-the-fly decoding.
// ============================================================

NOINLINE int64_t baseline_b_unpack(
    const PackedLayer& layer,
    const std::vector<int8_t>& activations
) {

    int64_t checksum = 0;

    for (uint32_t r = 0;
         r < layer.rows;
         ++r) {

        int32_t acc = 0;

        const size_t offset =
            static_cast<size_t>(r)
            * layer.words_per_row;

        for (uint32_t c = 0;
             c < layer.original_cols;
             ++c) {

            const uint32_t word = c / 32;
            const uint32_t bit = c % 32;

            const uint32_t p =
                layer.pos[offset + word];

            const uint32_t n =
                layer.neg[offset + word];

            const int32_t weight =
                static_cast<int32_t>(
                    (p >> bit) & 1U
                )
                -
                static_cast<int32_t>(
                    (n >> bit) & 1U
                );

            acc +=
                weight
                *
                static_cast<int32_t>(
                    activations[c]
                );
        }

        checksum += acc;
    }

    return checksum;
}

// ============================================================
// Allocate activation bitplanes
// ============================================================

void allocate_activation_planes(
    std::array<std::vector<uint32_t>, 8>& planes,
    uint32_t words_per_row
) {

    for (auto& plane : planes) {
        plane.resize(words_per_row);
    }
}

// ============================================================
// Activation packing
//
// INT8 activations -> 8 bitplanes.
//
// No allocation/resizing inside function.
// ============================================================

NOINLINE void pack_activations_reuse(
    const std::vector<int8_t>& activations,
    std::array<std::vector<uint32_t>, 8>& planes
) {

    for (auto& plane : planes) {
        std::fill(
            plane.begin(),
            plane.end(),
            0U
        );
    }

    for (uint32_t c = 0;
         c < activations.size();
         ++c) {

        const uint8_t value =
            static_cast<uint8_t>(
                activations[c]
            );

        const uint32_t word = c / 32;
        const uint32_t bit = c % 32;

        const uint32_t mask =
            1U << bit;

        for (int b = 0; b < 8; ++b) {

            if ((value >> b) & 1U) {
                planes[b][word] |= mask;
            }
        }
    }
}

// ============================================================
// Proposed bit-serial kernel
// ============================================================

NOINLINE int64_t proposed_bitserial(
    const PackedLayer& layer,
    const std::array<std::vector<uint32_t>, 8>& planes
) {

    int64_t checksum = 0;

    for (uint32_t r = 0;
         r < layer.rows;
         ++r) {

        int32_t acc = 0;

        const size_t offset =
            static_cast<size_t>(r)
            * layer.words_per_row;

        for (int b = 0; b < 8; ++b) {

            int32_t plane_acc = 0;

            const int32_t scale =
                (b == 7)
                ? -128
                : (1 << b);

            for (uint32_t w = 0;
                 w < layer.words_per_row;
                 ++w) {

                const uint32_t activation_bits =
                    planes[b][w];

                plane_acc +=
                    POPCOUNT32(
                        layer.pos[offset + w]
                        & activation_bits
                    );

                plane_acc -=
                    POPCOUNT32(
                        layer.neg[offset + w]
                        & activation_bits
                    );
            }

            acc +=
                plane_acc * scale;
        }

        checksum += acc;
    }

    return checksum;
}

// ============================================================
// Activation index helper
//
// Ensures all benchmark paths use the same sequence.
// ============================================================

inline int activation_index(
    int trial,
    int rep,
    int inner_reps,
    int activation_sets
) {

    return (
        trial * inner_reps + rep
    ) % activation_sets;
}

// ============================================================
// Time Baseline A
// ============================================================

double time_baseline_a_us(
    const std::vector<int8_t>& dense_weights,
    const std::vector<std::vector<int8_t>>& activation_pool,
    uint32_t rows,
    uint32_t cols,
    int trial,
    int repetitions
) {

    int64_t local_checksum = 0;

    const int activation_sets =
        static_cast<int>(
            activation_pool.size()
        );

    const auto start =
        std::chrono::steady_clock::now();

    for (int rep = 0;
         rep < repetitions;
         ++rep) {

        const int idx =
            activation_index(
                trial,
                rep,
                repetitions,
                activation_sets
            );

        local_checksum +=
            baseline_a_dense(
                dense_weights,
                activation_pool[idx],
                rows,
                cols
            );
    }

    const auto end =
        std::chrono::steady_clock::now();

    benchmark_sink += local_checksum;

    const double total_us =
        std::chrono::duration<
            double,
            std::micro
        >(end - start).count();

    return total_us / repetitions;
}

// ============================================================
// Time Baseline B
// ============================================================

double time_baseline_b_us(
    const PackedLayer& layer,
    const std::vector<std::vector<int8_t>>& activation_pool,
    int trial,
    int repetitions
) {

    int64_t local_checksum = 0;

    const int activation_sets =
        static_cast<int>(
            activation_pool.size()
        );

    const auto start =
        std::chrono::steady_clock::now();

    for (int rep = 0;
         rep < repetitions;
         ++rep) {

        const int idx =
            activation_index(
                trial,
                rep,
                repetitions,
                activation_sets
            );

        local_checksum +=
            baseline_b_unpack(
                layer,
                activation_pool[idx]
            );
    }

    const auto end =
        std::chrono::steady_clock::now();

    benchmark_sink += local_checksum;

    const double total_us =
        std::chrono::duration<
            double,
            std::micro
        >(end - start).count();

    return total_us / repetitions;
}

// ============================================================
// Time proposed kernel only
//
// Uses pre-packed activation planes.
//
// Critically, each repetition reads a different prepacked
// activation according to the same sequence used by A/B/E2E.
// ============================================================

double time_proposed_kernel_us(
    const PackedLayer& layer,
    const std::vector<
        std::array<
            std::vector<uint32_t>,
            8
        >
    >& prepacked_planes,
    int trial,
    int repetitions
) {

    int64_t local_checksum = 0;

    const int activation_sets =
        static_cast<int>(
            prepacked_planes.size()
        );

    const auto start =
        std::chrono::steady_clock::now();

    for (int rep = 0;
         rep < repetitions;
         ++rep) {

        const int idx =
            activation_index(
                trial,
                rep,
                repetitions,
                activation_sets
            );

        local_checksum +=
            proposed_bitserial(
                layer,
                prepacked_planes[idx]
            );
    }

    const auto end =
        std::chrono::steady_clock::now();

    benchmark_sink += local_checksum;

    const double total_us =
        std::chrono::duration<
            double,
            std::micro
        >(end - start).count();

    return total_us / repetitions;
}

// ============================================================
// Time activation packing only
//
// Same rotating activation sequence.
// ============================================================

double time_pack_us(
    const std::vector<std::vector<int8_t>>& activation_pool,
    std::array<std::vector<uint32_t>, 8>& planes,
    int trial,
    int repetitions
) {

    const int activation_sets =
        static_cast<int>(
            activation_pool.size()
        );

    uint64_t local_touch = 0;

    const auto start =
        std::chrono::steady_clock::now();

    for (int rep = 0;
         rep < repetitions;
         ++rep) {

        const int idx =
            activation_index(
                trial,
                rep,
                repetitions,
                activation_sets
            );

        pack_activations_reuse(
            activation_pool[idx],
            planes
        );

        // Read one packed value each iteration so the
        // generated planes remain observably used.
        local_touch += planes[0][0];
    }

    const auto end =
        std::chrono::steady_clock::now();

    benchmark_sink +=
        static_cast<int64_t>(
            local_touch
        );

    const double total_us =
        std::chrono::duration<
            double,
            std::micro
        >(end - start).count();

    return total_us / repetitions;
}

// ============================================================
// Direct proposed E2E
//
// Measures:
//
//     pack activation
//          +
//     proposed bit-serial kernel
//
// Uses exactly the same rotating activation sequence.
// ============================================================

double time_proposed_e2e_us(
    const PackedLayer& layer,
    const std::vector<std::vector<int8_t>>& activation_pool,
    std::array<std::vector<uint32_t>, 8>& planes,
    int trial,
    int repetitions
) {

    int64_t local_checksum = 0;

    const int activation_sets =
        static_cast<int>(
            activation_pool.size()
        );

    const auto start =
        std::chrono::steady_clock::now();

    for (int rep = 0;
         rep < repetitions;
         ++rep) {

        const int idx =
            activation_index(
                trial,
                rep,
                repetitions,
                activation_sets
            );

        pack_activations_reuse(
            activation_pool[idx],
            planes
        );

        local_checksum +=
            proposed_bitserial(
                layer,
                planes
            );
    }

    const auto end =
        std::chrono::steady_clock::now();

    benchmark_sink += local_checksum;

    const double total_us =
        std::chrono::duration<
            double,
            std::micro
        >(end - start).count();

    return total_us / repetitions;
}

// ============================================================
// Main
// ============================================================

int main(
    int argc,
    char* argv[]
) {

    if (argc != 4) {

        std::cerr
            << "Usage:\n"
            << "./benchmark_v5 "
            << "<layer.bin> "
            << "<layer_name> "
            << "<output.csv>\n";

        return 1;
    }

    const std::string binary_path =
        argv[1];

    const std::string layer_name =
        argv[2];

    const std::string csv_path =
        argv[3];

    // ========================================================
    // Configuration
    // ========================================================

    constexpr int WARMUP_ROUNDS = 30;
    constexpr int TRIALS = 60;
    constexpr int INNER_REPS = 10;

    // Must be greater than INNER_REPS so each timed repetition
    // can operate on changing activation data.
    constexpr int ACTIVATION_SETS = 64;

    const std::array<std::string, 6>
        BASE_ORDERS = {

            "A-B-C",
            "A-C-B",
            "B-A-C",
            "B-C-A",
            "C-A-B",
            "C-B-A"
        };

    // ========================================================
    // Balanced randomized execution order
    // ========================================================

    std::vector<std::string> trial_orders;

    trial_orders.reserve(TRIALS);

    for (int repeat = 0;
         repeat < TRIALS / 6;
         ++repeat) {

        for (const auto& order :
             BASE_ORDERS) {

            trial_orders.push_back(
                order
            );
        }
    }

    std::mt19937 order_rng(2026);

    std::shuffle(
        trial_orders.begin(),
        trial_orders.end(),
        order_rng
    );

    // ========================================================
    // Load layer
    // ========================================================

    PackedLayer layer =
        load_layer(binary_path);

    std::cout
        << "\n========================================\n"
        << "FINAL BENCHMARK V5\n"
        << "========================================\n"
        << "Layer           : "
        << layer_name
        << "\n"
        << "Matrix          : "
        << layer.rows
        << " x "
        << layer.original_cols
        << "\n"
        << "Words/row       : "
        << layer.words_per_row
        << "\n"
        << "Warmups         : "
        << WARMUP_ROUNDS
        << "\n"
        << "Trials          : "
        << TRIALS
        << "\n"
        << "Inner reps      : "
        << INNER_REPS
        << "\n"
        << "Activation sets : "
        << ACTIVATION_SETS
        << "\n"
        << "========================================\n";

    // ========================================================
    // Reconstruct dense weights
    // ========================================================

    std::cout
        << "[1] Reconstructing dense INT8 weights...\n";

    std::vector<int8_t> dense_weights =
        reconstruct_dense_weights(
            layer
        );

    // ========================================================
    // Generate activation pool
    // ========================================================

    std::cout
        << "[2] Generating "
        << ACTIVATION_SETS
        << " shared INT8 activation vectors...\n";

    std::vector<
        std::vector<int8_t>
    > activation_pool(

        ACTIVATION_SETS,

        std::vector<int8_t>(
            layer.original_cols
        )
    );

    std::mt19937 activation_rng(42);

    std::uniform_int_distribution<int>
        distribution(-128, 127);

    for (auto& activations :
         activation_pool) {

        for (auto& value :
             activations) {

            value =
                static_cast<int8_t>(
                    distribution(
                        activation_rng
                    )
                );
        }
    }

    // ========================================================
    // Prepack activation pool
    //
    // Used ONLY by kernel-only C timing.
    // ========================================================

    std::cout
        << "[3] Prepacking activation pool for C-only timing...\n";

    std::vector<
        std::array<
            std::vector<uint32_t>,
            8
        >
    > prepacked_planes(
        ACTIVATION_SETS
    );

    for (int i = 0;
         i < ACTIVATION_SETS;
         ++i) {

        allocate_activation_planes(
            prepacked_planes[i],
            layer.words_per_row
        );

        pack_activations_reuse(
            activation_pool[i],
            prepacked_planes[i]
        );
    }

    // ========================================================
    // Reusable buffers
    // ========================================================

    std::array<
        std::vector<uint32_t>,
        8
    > pack_planes;

    std::array<
        std::vector<uint32_t>,
        8
    > e2e_planes;

    allocate_activation_planes(
        pack_planes,
        layer.words_per_row
    );

    allocate_activation_planes(
        e2e_planes,
        layer.words_per_row
    );

    // ========================================================
    // Multi-input correctness check
    //
    // Check every activation set, not only one vector.
    // ========================================================

    std::cout
        << "[4] Running multi-input three-way correctness check...\n";

    int64_t correctness_sum_a = 0;
    int64_t correctness_sum_b = 0;
    int64_t correctness_sum_c = 0;

    for (int i = 0;
         i < ACTIVATION_SETS;
         ++i) {

        const int64_t output_a =
            baseline_a_dense(
                dense_weights,
                activation_pool[i],
                layer.rows,
                layer.original_cols
            );

        const int64_t output_b =
            baseline_b_unpack(
                layer,
                activation_pool[i]
            );

        const int64_t output_c =
            proposed_bitserial(
                layer,
                prepacked_planes[i]
            );

        if (
            output_a != output_b
            ||
            output_a != output_c
        ) {

            std::cerr
                << "\nFAIL at activation set "
                << i
                << "\n"
                << "A = "
                << output_a
                << "\n"
                << "B = "
                << output_b
                << "\n"
                << "C = "
                << output_c
                << "\n";

            return 2;
        }

        correctness_sum_a += output_a;
        correctness_sum_b += output_b;
        correctness_sum_c += output_c;
    }

    std::cout
        << "    PASS: A == B == C for all "
        << ACTIVATION_SETS
        << " activation sets\n"
        << "    Aggregate A = "
        << correctness_sum_a
        << "\n"
        << "    Aggregate B = "
        << correctness_sum_b
        << "\n"
        << "    Aggregate C = "
        << correctness_sum_c
        << "\n";

    // ========================================================
    // Warm-up
    //
    // Rotate through activation pool.
    // ========================================================

    std::cout
        << "[5] Running "
        << WARMUP_ROUNDS
        << " warm-up rounds...\n";

    for (int i = 0;
         i < WARMUP_ROUNDS;
         ++i) {

        const int idx =
            i % ACTIVATION_SETS;

        benchmark_sink +=
            baseline_a_dense(
                dense_weights,
                activation_pool[idx],
                layer.rows,
                layer.original_cols
            );

        benchmark_sink +=
            baseline_b_unpack(
                layer,
                activation_pool[idx]
            );

        benchmark_sink +=
            proposed_bitserial(
                layer,
                prepacked_planes[idx]
            );

        pack_activations_reuse(
            activation_pool[idx],
            e2e_planes
        );

        benchmark_sink +=
            proposed_bitserial(
                layer,
                e2e_planes
            );
    }

    // ========================================================
    // CSV setup
    // ========================================================

    const bool csv_exists =
        std::ifstream(
            csv_path
        ).good();

    std::ofstream csv(
        csv_path,
        std::ios::app
    );

    if (!csv) {

        std::cerr
            << "Could not open CSV: "
            << csv_path
            << "\n";

        return 3;
    }

    if (!csv_exists) {

        csv
            << "layer,"
            << "trial,"
            << "order,"
            << "baseline_a_us,"
            << "baseline_b_us,"
            << "activation_pack_us,"
            << "proposed_kernel_us,"
            << "proposed_component_sum_us,"
            << "proposed_direct_e2e_us,"
            << "e2e_to_component_ratio\n";
    }

    // ========================================================
    // Measured trials
    // ========================================================

    std::cout
        << "[6] Starting measured trials...\n";

    for (int trial = 0;
         trial < TRIALS;
         ++trial) {

        const std::string& order =
            trial_orders[trial];

        double time_a = 0.0;
        double time_b = 0.0;
        double time_c = 0.0;

        // ====================================================
        // Packing-only measurement
        // ====================================================

        const double activation_pack_us =
            time_pack_us(
                activation_pool,
                pack_planes,
                trial,
                INNER_REPS
            );

        // ====================================================
        // A/B/C helpers
        // ====================================================

        auto run_a = [&]() {

            time_a =
                time_baseline_a_us(
                    dense_weights,
                    activation_pool,
                    layer.rows,
                    layer.original_cols,
                    trial,
                    INNER_REPS
                );
        };

        auto run_b = [&]() {

            time_b =
                time_baseline_b_us(
                    layer,
                    activation_pool,
                    trial,
                    INNER_REPS
                );
        };

        auto run_c = [&]() {

            time_c =
                time_proposed_kernel_us(
                    layer,
                    prepacked_planes,
                    trial,
                    INNER_REPS
                );
        };

        // ====================================================
        // Balanced randomized A/B/C execution
        // ====================================================

        if (order == "A-B-C") {

            run_a();
            run_b();
            run_c();
        }

        else if (order == "A-C-B") {

            run_a();
            run_c();
            run_b();
        }

        else if (order == "B-A-C") {

            run_b();
            run_a();
            run_c();
        }

        else if (order == "B-C-A") {

            run_b();
            run_c();
            run_a();
        }

        else if (order == "C-A-B") {

            run_c();
            run_a();
            run_b();
        }

        else if (order == "C-B-A") {

            run_c();
            run_b();
            run_a();
        }

        // ====================================================
        // Component sum
        //
        // Separately measured pack + C.
        // ====================================================

        const double proposed_component_sum_us =
            activation_pack_us
            +
            time_c;

        // ====================================================
        // Direct E2E
        //
        // Same activation sequence:
        //
        // pack(current activation)
        // +
        // proposed_bitserial(current packed activation)
        // ====================================================

        const double proposed_direct_e2e_us =
            time_proposed_e2e_us(
                layer,
                activation_pool,
                e2e_planes,
                trial,
                INNER_REPS
            );

        const double e2e_to_component_ratio =
            proposed_component_sum_us > 0.0
            ? proposed_direct_e2e_us
                / proposed_component_sum_us
            : 0.0;

        // ====================================================
        // CSV
        // ====================================================

        csv
            << layer_name
            << ","
            << trial
            << ","
            << order
            << ","
            << std::fixed
            << std::setprecision(3)
            << time_a
            << ","
            << time_b
            << ","
            << activation_pack_us
            << ","
            << time_c
            << ","
            << proposed_component_sum_us
            << ","
            << proposed_direct_e2e_us
            << ","
            << std::setprecision(6)
            << e2e_to_component_ratio
            << "\n";

        // ====================================================
        // Console
        // ====================================================

        std::cout
            << "Trial "
            << std::setw(2)
            << trial
            << " ["
            << order
            << "] "
            << std::fixed
            << std::setprecision(2)
            << "A="
            << time_a
            << " us  "
            << "B="
            << time_b
            << " us  "
            << "C="
            << time_c
            << " us  "
            << "Pack="
            << activation_pack_us
            << " us  "
            << "Component="
            << proposed_component_sum_us
            << " us  "
            << "DirectE2E="
            << proposed_direct_e2e_us
            << " us  "
            << "Ratio="
            << std::setprecision(3)
            << e2e_to_component_ratio
            << "\n";
    }

    csv.close();

    // ========================================================
    // Complete
    // ========================================================

    std::cout
        << "\n========================================\n"
        << "BENCHMARK V5 COMPLETE\n"
        << "========================================\n"
        << "Layer: "
        << layer_name
        << "\n"
        << "Checksum sink: "
        << benchmark_sink
        << "\n"
        << "CSV: "
        << csv_path
        << "\n";

    return 0;
}

Writing /content/benchmark_v5.cpp


In [ ]:
!g++ -O3 -std=c++17 -march=native /content/benchmark_v5.cpp -o /content/benchmark_v5

In [ ]:
# Remove old V5 results if rerunning from scratch
!rm -f /content/benchmark_v5_raw.csv

# Run k_proj
!/content/benchmark_v5 \
"/content/drive/MyDrive/BitNetExperiments/k_proj_640x2560.bin" \
"k_proj_640x2560" \
"/content/benchmark_v5_raw.csv"

# Run q_proj
!/content/benchmark_v5 \
"/content/drive/MyDrive/BitNetExperiments/q_proj_2560x2560.bin" \
"q_proj_2560x2560" \
"/content/benchmark_v5_raw.csv"

# Run gate_proj
!/content/benchmark_v5 \
"/content/drive/MyDrive/BitNetExperiments/gate_proj_6912x2560.bin" \
"gate_proj_6912x2560" \
"/content/benchmark_v5_raw.csv"


FINAL BENCHMARK V5
Layer           : k_proj_640x2560
Matrix          : 640 x 2560
Words/row       : 80
Warmups         : 30
Trials          : 60
Inner reps      : 10
Activation sets : 64
[1] Reconstructing dense INT8 weights...
[2] Generating 64 shared INT8 activation vectors...
[3] Prepacking activation pool for C-only timing...
[4] Running multi-input three-way correctness check...
    PASS: A == B == C for all 64 activation sets
    Aggregate A = -708320
    Aggregate B = -708320
    Aggregate C = -708320
[5] Running 30 warm-up rounds...
[6] Starting measured trials...
Trial  0 [A-C-B] A=269.44 us  B=2689.12 us  C=1108.91 us  Pack=103.65 us  Component=1212.56 us  DirectE2E=760.66 us  Ratio=0.627
Trial  1 [A-B-C] A=251.19 us  B=2433.77 us  C=683.84 us  Pack=89.25 us  Component=773.10 us  DirectE2E=771.01 us  Ratio=0.997
Trial  2 [B-A-C] A=249.39 us  B=2432.45 us  C=812.00 us  Pack=96.50 us  Component=908.49 us  DirectE2E=759.40 us  Ratio=0.836
Trial  3 [A-B-C] A=247.89 us  B=2431.89

In [ ]:
import pandas as pd

df = pd.read_csv("/content/benchmark_v5_raw.csv")

print("Total rows:", len(df))

print("\nTrials per layer:")
print(df.groupby("layer").size())

print("\n=== MEDIAN RESULTS ===")
print(
    df.groupby("layer")[
        [
            "baseline_a_us",
            "baseline_b_us",
            "activation_pack_us",
            "proposed_kernel_us",
            "proposed_component_sum_us",
            "proposed_direct_e2e_us",
            "e2e_to_component_ratio"
        ]
    ].median().round(3)
)

print("\n=== E2E / COMPONENT RATIO ===")
print(
    df.groupby("layer")["e2e_to_component_ratio"]
      .agg(["mean", "median", "min", "max"])
      .round(3)
)

Total rows: 180

Trials per layer:
layer
gate_proj_6912x2560    60
k_proj_640x2560        60
q_proj_2560x2560       60
dtype: int64

=== MEDIAN RESULTS ===
                     baseline_a_us  baseline_b_us  activation_pack_us  \
layer                                                                   
gate_proj_6912x2560       3068.414      28156.434              87.222   
k_proj_640x2560            249.014       2453.614              87.420   
q_proj_2560x2560          1025.412      10241.131              87.624   

                     proposed_kernel_us  proposed_component_sum_us  \
layer                                                                
gate_proj_6912x2560            7461.314                   7548.289   
k_proj_640x2560                 677.090                    765.764   
q_proj_2560x2560               2734.138                   2821.469   

                     proposed_direct_e2e_us  e2e_to_component_ratio  
layer                                                    

In [ ]:
%%writefile /content/benchmark_v6.cpp

#include <algorithm>
#include <array>
#include <chrono>
#include <cstdint>
#include <fstream>
#include <iomanip>
#include <immintrin.h>
#include <iostream>
#include <random>
#include <stdexcept>
#include <string>
#include <vector>

#if defined(_MSC_VER)
#include <intrin.h>
#define POPCOUNT32 __popcnt
#define NOINLINE __declspec(noinline)
#elif defined(__GNUC__) || defined(__clang__)
#define POPCOUNT32 __builtin_popcount
#define NOINLINE __attribute__((noinline))
#else
#define POPCOUNT32 __builtin_popcount
#define NOINLINE
#endif

volatile int64_t benchmark_sink = 0;

// ============================================================
// Packed ternary layer
// ============================================================

struct PackedLayer {
    uint32_t rows;
    uint32_t words_per_row;
    uint32_t original_cols;

    std::vector<uint32_t> pos;
    std::vector<uint32_t> neg;
};

// ============================================================
// Load packed layer
// ============================================================

PackedLayer load_layer(const std::string& filename) {

    std::ifstream file(filename, std::ios::binary);

    if (!file) {
        throw std::runtime_error(
            "Could not open: " + filename
        );
    }

    PackedLayer layer;

    file.read(
        reinterpret_cast<char*>(&layer.rows),
        sizeof(uint32_t)
    );

    file.read(
        reinterpret_cast<char*>(&layer.words_per_row),
        sizeof(uint32_t)
    );

    file.read(
        reinterpret_cast<char*>(&layer.original_cols),
        sizeof(uint32_t)
    );

    if (!file) {
        throw std::runtime_error(
            "Failed to read header."
        );
    }

    const size_t total_words =
        static_cast<size_t>(layer.rows)
        * layer.words_per_row;

    layer.pos.resize(total_words);
    layer.neg.resize(total_words);

    file.read(
        reinterpret_cast<char*>(layer.pos.data()),
        total_words * sizeof(uint32_t)
    );

    file.read(
        reinterpret_cast<char*>(layer.neg.data()),
        total_words * sizeof(uint32_t)
    );

    if (!file) {
        throw std::runtime_error(
            "Failed to read packed weight data."
        );
    }

    return layer;
}

// ============================================================
// Reconstruct dense INT8 ternary weights
// ============================================================

std::vector<int8_t> reconstruct_dense_weights(
    const PackedLayer& layer
) {

    const size_t total_elements =
        static_cast<size_t>(layer.rows)
        * layer.original_cols;

    std::vector<int8_t> dense(total_elements);

    for (uint32_t r = 0; r < layer.rows; ++r) {

        const size_t packed_offset =
            static_cast<size_t>(r)
            * layer.words_per_row;

        const size_t dense_offset =
            static_cast<size_t>(r)
            * layer.original_cols;

        for (uint32_t c = 0;
             c < layer.original_cols;
             ++c) {

            const uint32_t word = c / 32;
            const uint32_t bit = c % 32;

            const uint32_t p =
                layer.pos[packed_offset + word];

            const uint32_t n =
                layer.neg[packed_offset + word];

            const int32_t weight =
                static_cast<int32_t>(
                    (p >> bit) & 1U
                )
                -
                static_cast<int32_t>(
                    (n >> bit) & 1U
                );

            dense[dense_offset + c] =
                static_cast<int8_t>(weight);
        }
    }

    return dense;
}

// ============================================================
// BASELINE A
//
// Dense INT8 ternary weights.
// Scalar multiply-accumulate.
// ============================================================

NOINLINE int64_t baseline_a_dense(
    const std::vector<int8_t>& weights,
    const std::vector<int8_t>& activations,
    uint32_t rows,
    uint32_t cols
) {

    int64_t checksum = 0;

    for (uint32_t r = 0; r < rows; ++r) {

        int32_t acc = 0;

        const size_t offset =
            static_cast<size_t>(r) * cols;

        for (uint32_t c = 0; c < cols; ++c) {

            acc +=
                static_cast<int32_t>(
                    weights[offset + c]
                )
                *
                static_cast<int32_t>(
                    activations[c]
                );
        }

        checksum += acc;
    }

    return checksum;
}

// ============================================================
// BASELINE B
//
// Packed ternary weights.
// Scalar on-the-fly decoding.
// ============================================================

NOINLINE int64_t baseline_b_unpack(
    const PackedLayer& layer,
    const std::vector<int8_t>& activations
) {

    int64_t checksum = 0;

    for (uint32_t r = 0;
         r < layer.rows;
         ++r) {

        int32_t acc = 0;

        const size_t offset =
            static_cast<size_t>(r)
            * layer.words_per_row;

        for (uint32_t c = 0;
             c < layer.original_cols;
             ++c) {

            const uint32_t word = c / 32;
            const uint32_t bit = c % 32;

            const uint32_t p =
                layer.pos[offset + word];

            const uint32_t n =
                layer.neg[offset + word];

            const int32_t weight =
                static_cast<int32_t>(
                    (p >> bit) & 1U
                )
                -
                static_cast<int32_t>(
                    (n >> bit) & 1U
                );

            acc +=
                weight
                *
                static_cast<int32_t>(
                    activations[c]
                );
        }

        checksum += acc;
    }

    return checksum;
}

// ============================================================
// Allocate activation bitplanes
// ============================================================

void allocate_activation_planes(
    std::array<std::vector<uint32_t>, 8>& planes,
    uint32_t words_per_row
) {

    for (auto& plane : planes) {
        plane.resize(words_per_row);
    }
}

// ============================================================
// Activation packing
//
// INT8 -> 8 bitplanes
// ============================================================

NOINLINE void pack_activations_reuse(
    const std::vector<int8_t>& activations,
    std::array<std::vector<uint32_t>, 8>& planes
) {

    for (auto& plane : planes) {
        std::fill(
            plane.begin(),
            plane.end(),
            0U
        );
    }

    for (uint32_t c = 0;
         c < activations.size();
         ++c) {

        const uint8_t value =
            static_cast<uint8_t>(
                activations[c]
            );

        const uint32_t word = c / 32;
        const uint32_t bit = c % 32;

        const uint32_t mask =
            1U << bit;

        for (int b = 0; b < 8; ++b) {

            if ((value >> b) & 1U) {
                planes[b][word] |= mask;
            }
        }
    }
}

// ============================================================
// C: Scalar bit-serial reference kernel
// ============================================================

NOINLINE int64_t proposed_bitserial_scalar(
    const PackedLayer& layer,
    const std::array<std::vector<uint32_t>, 8>& planes
) {

    int64_t checksum = 0;

    for (uint32_t r = 0;
         r < layer.rows;
         ++r) {

        int32_t acc = 0;

        const size_t offset =
            static_cast<size_t>(r)
            * layer.words_per_row;

        for (int b = 0; b < 8; ++b) {

            int32_t plane_acc = 0;

            const int32_t scale =
                (b == 7)
                ? -128
                : (1 << b);

            for (uint32_t w = 0;
                 w < layer.words_per_row;
                 ++w) {

                const uint32_t activation_bits =
                    planes[b][w];

                plane_acc +=
                    POPCOUNT32(
                        layer.pos[offset + w]
                        & activation_bits
                    );

                plane_acc -=
                    POPCOUNT32(
                        layer.neg[offset + w]
                        & activation_bits
                    );
            }

            acc +=
                plane_acc * scale;
        }

        checksum += acc;
    }

    return checksum;
}

// ============================================================
// AVX2 popcount helper
//
// AVX2 does not provide native vector POPCNT.
//
// We therefore:
// 1. Compute eight 32-bit AND operations with AVX2.
// 2. Store the eight lanes.
// 3. Apply scalar hardware POPCNT to each lane.
//
// This is our first AVX2 implementation.
// ============================================================

inline int32_t popcount256_epi32_sum(
    __m256i value
) {

    alignas(32) uint32_t lanes[8];

    _mm256_store_si256(
        reinterpret_cast<__m256i*>(lanes),
        value
    );

    return
        POPCOUNT32(lanes[0]) +
        POPCOUNT32(lanes[1]) +
        POPCOUNT32(lanes[2]) +
        POPCOUNT32(lanes[3]) +
        POPCOUNT32(lanes[4]) +
        POPCOUNT32(lanes[5]) +
        POPCOUNT32(lanes[6]) +
        POPCOUNT32(lanes[7]);
}

// ============================================================
// D: AVX2 bit-serial kernel
//
// Processes 8 packed uint32 words = 256 activation/weight
// positions per AVX2 iteration.
//
// Mathematical operation remains identical to C:
//
// popcount(pos & activation_bits)
// -
// popcount(neg & activation_bits)
//
// Tail words are handled by scalar POPCOUNT32.
// ============================================================

NOINLINE int64_t proposed_bitserial_avx2(
    const PackedLayer& layer,
    const std::array<std::vector<uint32_t>, 8>& planes
) {

    int64_t checksum = 0;

    for (uint32_t r = 0;
         r < layer.rows;
         ++r) {

        int32_t acc = 0;

        const size_t offset =
            static_cast<size_t>(r)
            * layer.words_per_row;

        for (int b = 0;
             b < 8;
             ++b) {

            int32_t plane_acc = 0;

            const int32_t scale =
                (b == 7)
                ? -128
                : (1 << b);

            uint32_t w = 0;

            // ------------------------------------------------
            // AVX2 main loop
            //
            // 8 x uint32 words at once.
            // ------------------------------------------------

            for (;
                 w + 8 <= layer.words_per_row;
                 w += 8) {

                const __m256i pos_vec =
                    _mm256_loadu_si256(
                        reinterpret_cast<
                            const __m256i*
                        >(
                            layer.pos.data()
                            + offset
                            + w
                        )
                    );

                const __m256i neg_vec =
                    _mm256_loadu_si256(
                        reinterpret_cast<
                            const __m256i*
                        >(
                            layer.neg.data()
                            + offset
                            + w
                        )
                    );

                const __m256i activation_vec =
                    _mm256_loadu_si256(
                        reinterpret_cast<
                            const __m256i*
                        >(
                            planes[b].data()
                            + w
                        )
                    );

                const __m256i pos_and =
                    _mm256_and_si256(
                        pos_vec,
                        activation_vec
                    );

                const __m256i neg_and =
                    _mm256_and_si256(
                        neg_vec,
                        activation_vec
                    );

                plane_acc +=
                    popcount256_epi32_sum(
                        pos_and
                    );

                plane_acc -=
                    popcount256_epi32_sum(
                        neg_and
                    );
            }

            // ------------------------------------------------
            // Scalar tail
            // ------------------------------------------------

            for (;
                 w < layer.words_per_row;
                 ++w) {

                const uint32_t activation_bits =
                    planes[b][w];

                plane_acc +=
                    POPCOUNT32(
                        layer.pos[offset + w]
                        & activation_bits
                    );

                plane_acc -=
                    POPCOUNT32(
                        layer.neg[offset + w]
                        & activation_bits
                    );
            }

            acc +=
                plane_acc * scale;
        }

        checksum += acc;
    }

    return checksum;
}

// ============================================================
// Activation index
// ============================================================

inline int activation_index(
    int trial,
    int rep,
    int inner_reps,
    int activation_sets
) {

    return (
        trial * inner_reps + rep
    ) % activation_sets;
}

// ============================================================
// Time A
// ============================================================

double time_baseline_a_us(
    const std::vector<int8_t>& dense_weights,
    const std::vector<std::vector<int8_t>>& activation_pool,
    uint32_t rows,
    uint32_t cols,
    int trial,
    int repetitions
) {

    int64_t local_checksum = 0;

    const int activation_sets =
        static_cast<int>(
            activation_pool.size()
        );

    const auto start =
        std::chrono::steady_clock::now();

    for (int rep = 0;
         rep < repetitions;
         ++rep) {

        const int idx =
            activation_index(
                trial,
                rep,
                repetitions,
                activation_sets
            );

        local_checksum +=
            baseline_a_dense(
                dense_weights,
                activation_pool[idx],
                rows,
                cols
            );
    }

    const auto end =
        std::chrono::steady_clock::now();

    benchmark_sink += local_checksum;

    return std::chrono::duration<
        double,
        std::micro
    >(end - start).count() / repetitions;
}

// ============================================================
// Time B
// ============================================================

double time_baseline_b_us(
    const PackedLayer& layer,
    const std::vector<std::vector<int8_t>>& activation_pool,
    int trial,
    int repetitions
) {

    int64_t local_checksum = 0;

    const int activation_sets =
        static_cast<int>(
            activation_pool.size()
        );

    const auto start =
        std::chrono::steady_clock::now();

    for (int rep = 0;
         rep < repetitions;
         ++rep) {

        const int idx =
            activation_index(
                trial,
                rep,
                repetitions,
                activation_sets
            );

        local_checksum +=
            baseline_b_unpack(
                layer,
                activation_pool[idx]
            );
    }

    const auto end =
        std::chrono::steady_clock::now();

    benchmark_sink += local_checksum;

    return std::chrono::duration<
        double,
        std::micro
    >(end - start).count() / repetitions;
}

// ============================================================
// Generic prepacked kernel timer
//
// Used for both C and D.
// ============================================================

template <typename Kernel>
double time_prepacked_kernel_us(
    Kernel kernel,
    const PackedLayer& layer,
    const std::vector<
        std::array<
            std::vector<uint32_t>,
            8
        >
    >& prepacked_planes,
    int trial,
    int repetitions
) {

    int64_t local_checksum = 0;

    const int activation_sets =
        static_cast<int>(
            prepacked_planes.size()
        );

    const auto start =
        std::chrono::steady_clock::now();

    for (int rep = 0;
         rep < repetitions;
         ++rep) {

        const int idx =
            activation_index(
                trial,
                rep,
                repetitions,
                activation_sets
            );

        local_checksum +=
            kernel(
                layer,
                prepacked_planes[idx]
            );
    }

    const auto end =
        std::chrono::steady_clock::now();

    benchmark_sink += local_checksum;

    return std::chrono::duration<
        double,
        std::micro
    >(end - start).count() / repetitions;
}

// ============================================================
// Time packing
// ============================================================

double time_pack_us(
    const std::vector<std::vector<int8_t>>& activation_pool,
    std::array<std::vector<uint32_t>, 8>& planes,
    int trial,
    int repetitions
) {

    const int activation_sets =
        static_cast<int>(
            activation_pool.size()
        );

    uint64_t local_touch = 0;

    const auto start =
        std::chrono::steady_clock::now();

    for (int rep = 0;
         rep < repetitions;
         ++rep) {

        const int idx =
            activation_index(
                trial,
                rep,
                repetitions,
                activation_sets
            );

        pack_activations_reuse(
            activation_pool[idx],
            planes
        );

        local_touch +=
            planes[0][0];
    }

    const auto end =
        std::chrono::steady_clock::now();

    benchmark_sink +=
        static_cast<int64_t>(
            local_touch
        );

    return std::chrono::duration<
        double,
        std::micro
    >(end - start).count() / repetitions;
}

// ============================================================
// Direct D E2E
//
// pack activation + AVX2 kernel
// ============================================================

double time_avx2_e2e_us(
    const PackedLayer& layer,
    const std::vector<std::vector<int8_t>>& activation_pool,
    std::array<std::vector<uint32_t>, 8>& planes,
    int trial,
    int repetitions
) {

    int64_t local_checksum = 0;

    const int activation_sets =
        static_cast<int>(
            activation_pool.size()
        );

    const auto start =
        std::chrono::steady_clock::now();

    for (int rep = 0;
         rep < repetitions;
         ++rep) {

        const int idx =
            activation_index(
                trial,
                rep,
                repetitions,
                activation_sets
            );

        pack_activations_reuse(
            activation_pool[idx],
            planes
        );

        local_checksum +=
            proposed_bitserial_avx2(
                layer,
                planes
            );
    }

    const auto end =
        std::chrono::steady_clock::now();

    benchmark_sink += local_checksum;

    return std::chrono::duration<
        double,
        std::micro
    >(end - start).count() / repetitions;
}

// ============================================================
// Main
// ============================================================

int main(
    int argc,
    char* argv[]
) {

    if (argc != 4) {

        std::cerr
            << "Usage:\n"
            << "./benchmark_v6 "
            << "<layer.bin> "
            << "<layer_name> "
            << "<output.csv>\n";

        return 1;
    }

    const std::string binary_path =
        argv[1];

    const std::string layer_name =
        argv[2];

    const std::string csv_path =
        argv[3];

    constexpr int WARMUP_ROUNDS = 30;
    constexpr int TRIALS = 60;
    constexpr int INNER_REPS = 10;
    constexpr int ACTIVATION_SETS = 64;

    // ========================================================
    // Balanced C/D execution order
    //
    // A and B remain reference baselines.
    // C and D are the primary kernel comparison.
    // ========================================================

    std::vector<bool> d_first(TRIALS);

    for (int i = 0;
         i < TRIALS;
         ++i) {

        d_first[i] =
            (i >= TRIALS / 2);
    }

    std::mt19937 order_rng(2026);

    std::shuffle(
        d_first.begin(),
        d_first.end(),
        order_rng
    );

    // ========================================================
    // Load
    // ========================================================

    PackedLayer layer =
        load_layer(
            binary_path
        );

    std::cout
        << "\n========================================\n"
        << "FINAL BENCHMARK V6 - AVX2\n"
        << "========================================\n"
        << "Layer           : "
        << layer_name
        << "\n"
        << "Matrix          : "
        << layer.rows
        << " x "
        << layer.original_cols
        << "\n"
        << "Words/row       : "
        << layer.words_per_row
        << "\n"
        << "Warmups         : "
        << WARMUP_ROUNDS
        << "\n"
        << "Trials          : "
        << TRIALS
        << "\n"
        << "Inner reps      : "
        << INNER_REPS
        << "\n"
        << "Activation sets : "
        << ACTIVATION_SETS
        << "\n"
        << "SIMD            : AVX2\n"
        << "========================================\n";

    // ========================================================
    // Dense weights
    // ========================================================

    std::cout
        << "[1] Reconstructing dense INT8 weights...\n";

    std::vector<int8_t> dense_weights =
        reconstruct_dense_weights(
            layer
        );

    // ========================================================
    // Activation pool
    // ========================================================

    std::cout
        << "[2] Generating activation pool...\n";

    std::vector<
        std::vector<int8_t>
    > activation_pool(

        ACTIVATION_SETS,

        std::vector<int8_t>(
            layer.original_cols
        )
    );

    std::mt19937 activation_rng(42);

    std::uniform_int_distribution<int>
        distribution(-128, 127);

    for (auto& activations :
         activation_pool) {

        for (auto& value :
             activations) {

            value =
                static_cast<int8_t>(
                    distribution(
                        activation_rng
                    )
                );
        }
    }

    // ========================================================
    // Prepack activation pool
    // ========================================================

    std::cout
        << "[3] Prepacking activation pool...\n";

    std::vector<
        std::array<
            std::vector<uint32_t>,
            8
        >
    > prepacked_planes(
        ACTIVATION_SETS
    );

    for (int i = 0;
         i < ACTIVATION_SETS;
         ++i) {

        allocate_activation_planes(
            prepacked_planes[i],
            layer.words_per_row
        );

        pack_activations_reuse(
            activation_pool[i],
            prepacked_planes[i]
        );
    }

    std::array<
        std::vector<uint32_t>,
        8
    > pack_planes;

    std::array<
        std::vector<uint32_t>,
        8
    > e2e_planes;

    allocate_activation_planes(
        pack_planes,
        layer.words_per_row
    );

    allocate_activation_planes(
        e2e_planes,
        layer.words_per_row
    );

    // ========================================================
    // Four-way correctness
    // ========================================================

    std::cout
        << "[4] Running 64-input A/B/C/D correctness check...\n";

    int64_t aggregate_a = 0;
    int64_t aggregate_b = 0;
    int64_t aggregate_c = 0;
    int64_t aggregate_d = 0;

    for (int i = 0;
         i < ACTIVATION_SETS;
         ++i) {

        const int64_t a =
            baseline_a_dense(
                dense_weights,
                activation_pool[i],
                layer.rows,
                layer.original_cols
            );

        const int64_t b =
            baseline_b_unpack(
                layer,
                activation_pool[i]
            );

        const int64_t c =
            proposed_bitserial_scalar(
                layer,
                prepacked_planes[i]
            );

        const int64_t d =
            proposed_bitserial_avx2(
                layer,
                prepacked_planes[i]
            );

        if (
            a != b
            ||
            a != c
            ||
            a != d
        ) {

            std::cerr
                << "\nFAIL at activation set "
                << i
                << "\n"
                << "A = "
                << a
                << "\n"
                << "B = "
                << b
                << "\n"
                << "C = "
                << c
                << "\n"
                << "D = "
                << d
                << "\n";

            return 2;
        }

        aggregate_a += a;
        aggregate_b += b;
        aggregate_c += c;
        aggregate_d += d;
    }

    std::cout
        << "    PASS: A == B == C == D for all "
        << ACTIVATION_SETS
        << " activation sets\n"
        << "    Aggregate A = "
        << aggregate_a
        << "\n"
        << "    Aggregate B = "
        << aggregate_b
        << "\n"
        << "    Aggregate C = "
        << aggregate_c
        << "\n"
        << "    Aggregate D = "
        << aggregate_d
        << "\n";

    // ========================================================
    // Warmup
    // ========================================================

    std::cout
        << "[5] Running warm-up rounds...\n";

    for (int i = 0;
         i < WARMUP_ROUNDS;
         ++i) {

        const int idx =
            i % ACTIVATION_SETS;

        benchmark_sink +=
            baseline_a_dense(
                dense_weights,
                activation_pool[idx],
                layer.rows,
                layer.original_cols
            );

        benchmark_sink +=
            baseline_b_unpack(
                layer,
                activation_pool[idx]
            );

        benchmark_sink +=
            proposed_bitserial_scalar(
                layer,
                prepacked_planes[idx]
            );

        benchmark_sink +=
            proposed_bitserial_avx2(
                layer,
                prepacked_planes[idx]
            );

        pack_activations_reuse(
            activation_pool[idx],
            e2e_planes
        );

        benchmark_sink +=
            proposed_bitserial_avx2(
                layer,
                e2e_planes
            );
    }

    // ========================================================
    // CSV
    // ========================================================

    const bool csv_exists =
        std::ifstream(
            csv_path
        ).good();

    std::ofstream csv(
        csv_path,
        std::ios::app
    );

    if (!csv) {

        std::cerr
            << "Could not open CSV: "
            << csv_path
            << "\n";

        return 3;
    }

    if (!csv_exists) {

        csv
            << "layer,"
            << "trial,"
            << "cd_order,"
            << "baseline_a_us,"
            << "baseline_b_us,"
            << "activation_pack_us,"
            << "scalar_c_kernel_us,"
            << "avx2_d_kernel_us,"
            << "avx2_component_sum_us,"
            << "avx2_direct_e2e_us,"
            << "avx2_vs_scalar_speedup,"
            << "dense_vs_avx2_kernel_ratio,"
            << "dense_vs_avx2_e2e_ratio,"
            << "e2e_to_component_ratio\n";
    }

    // ========================================================
    // Trials
    // ========================================================

    std::cout
        << "[6] Starting measured trials...\n";

    for (int trial = 0;
         trial < TRIALS;
         ++trial) {

        const double time_a =
            time_baseline_a_us(
                dense_weights,
                activation_pool,
                layer.rows,
                layer.original_cols,
                trial,
                INNER_REPS
            );

        const double time_b =
            time_baseline_b_us(
                layer,
                activation_pool,
                trial,
                INNER_REPS
            );

        const double activation_pack_us =
            time_pack_us(
                activation_pool,
                pack_planes,
                trial,
                INNER_REPS
            );

        double time_c = 0.0;
        double time_d = 0.0;

        std::string cd_order;

        if (d_first[trial]) {

            cd_order = "D-C";

            time_d =
                time_prepacked_kernel_us(
                    proposed_bitserial_avx2,
                    layer,
                    prepacked_planes,
                    trial,
                    INNER_REPS
                );

            time_c =
                time_prepacked_kernel_us(
                    proposed_bitserial_scalar,
                    layer,
                    prepacked_planes,
                    trial,
                    INNER_REPS
                );
        }

        else {

            cd_order = "C-D";

            time_c =
                time_prepacked_kernel_us(
                    proposed_bitserial_scalar,
                    layer,
                    prepacked_planes,
                    trial,
                    INNER_REPS
                );

            time_d =
                time_prepacked_kernel_us(
                    proposed_bitserial_avx2,
                    layer,
                    prepacked_planes,
                    trial,
                    INNER_REPS
                );
        }

        const double avx2_component_sum_us =
            activation_pack_us
            +
            time_d;

        const double avx2_direct_e2e_us =
            time_avx2_e2e_us(
                layer,
                activation_pool,
                e2e_planes,
                trial,
                INNER_REPS
            );

        // C / D:
        // >1 means AVX2 is faster than scalar bit-serial.
        const double avx2_vs_scalar_speedup =
            time_d > 0.0
            ? time_c / time_d
            : 0.0;

        // A / D:
        // >1 means AVX2 D is faster than dense A.
        // <1 means dense A remains faster.
        const double dense_vs_avx2_kernel_ratio =
            time_d > 0.0
            ? time_a / time_d
            : 0.0;

        // A / D-E2E:
        // >1 means proposed AVX2 E2E beats dense A.
        const double dense_vs_avx2_e2e_ratio =
            avx2_direct_e2e_us > 0.0
            ? time_a / avx2_direct_e2e_us
            : 0.0;

        const double e2e_to_component_ratio =
            avx2_component_sum_us > 0.0
            ? avx2_direct_e2e_us
                / avx2_component_sum_us
            : 0.0;

        csv
            << layer_name
            << ","
            << trial
            << ","
            << cd_order
            << ","
            << std::fixed
            << std::setprecision(3)
            << time_a
            << ","
            << time_b
            << ","
            << activation_pack_us
            << ","
            << time_c
            << ","
            << time_d
            << ","
            << avx2_component_sum_us
            << ","
            << avx2_direct_e2e_us
            << ","
            << std::setprecision(6)
            << avx2_vs_scalar_speedup
            << ","
            << dense_vs_avx2_kernel_ratio
            << ","
            << dense_vs_avx2_e2e_ratio
            << ","
            << e2e_to_component_ratio
            << "\n";

        std::cout
            << "Trial "
            << std::setw(2)
            << trial
            << " ["
            << cd_order
            << "] "
            << std::fixed
            << std::setprecision(2)
            << "A="
            << time_a
            << " us  "
            << "B="
            << time_b
            << " us  "
            << "C="
            << time_c
            << " us  "
            << "D="
            << time_d
            << " us  "
            << "Pack="
            << activation_pack_us
            << " us  "
            << "D-E2E="
            << avx2_direct_e2e_us
            << " us  "
            << "C/D="
            << std::setprecision(3)
            << avx2_vs_scalar_speedup
            << "x  "
            << "A/D="
            << dense_vs_avx2_kernel_ratio
            << "x  "
            << "A/D-E2E="
            << dense_vs_avx2_e2e_ratio
            << "x\n";
    }

    csv.close();

    std::cout
        << "\n========================================\n"
        << "BENCHMARK V6 COMPLETE\n"
        << "========================================\n"
        << "Layer: "
        << layer_name
        << "\n"
        << "Checksum sink: "
        << benchmark_sink
        << "\n"
        << "CSV: "
        << csv_path
        << "\n";

    return 0;
}

Writing /content/benchmark_v6.cpp


In [ ]:
!g++ -O3 -std=c++17 -mavx2 -mpopcnt \
    /content/benchmark_v6.cpp \
    -o /content/benchmark_v6

In [ ]:
!/content/benchmark_v6 \
"/content/drive/MyDrive/BitNetExperiments/k_proj_640x2560.bin" \
"k_proj_640x2560" \
"/content/benchmark_v6_raw.csv"


FINAL BENCHMARK V6 - AVX2
Layer           : k_proj_640x2560
Matrix          : 640 x 2560
Words/row       : 80
Warmups         : 30
Trials          : 60
Inner reps      : 10
Activation sets : 64
SIMD            : AVX2
[1] Reconstructing dense INT8 weights...
[2] Generating activation pool...
[3] Prepacking activation pool...
[4] Running 64-input A/B/C/D correctness check...
    PASS: A == B == C == D for all 64 activation sets
    Aggregate A = -708320
    Aggregate B = -708320
    Aggregate C = -708320
    Aggregate D = -708320
[5] Running warm-up rounds...
[6] Starting measured trials...
Trial  0 [D-C] A=245.85 us  B=3217.02 us  C=493.86 us  D=584.77 us  Pack=88.15 us  D-E2E=676.01 us  C/D=0.845x  A/D=0.420x  A/D-E2E=0.364x
Trial  1 [C-D] A=249.38 us  B=3236.62 us  C=503.25 us  D=586.30 us  Pack=86.03 us  D-E2E=690.07 us  C/D=0.858x  A/D=0.425x  A/D-E2E=0.361x
Trial  2 [D-C] A=249.15 us  B=3205.58 us  C=493.31 us  D=579.16 us  Pack=87.70 us  D-E2E=683.92 us  C/D=0.852x  A/D=0.430x  A

In [ ]:
!/content/benchmark_v6 \
"/content/drive/MyDrive/BitNetExperiments/q_proj_2560x2560.bin" \
"q_proj_2560x2560" \
"/content/benchmark_v6_raw.csv"

!/content/benchmark_v6 \
"/content/drive/MyDrive/BitNetExperiments/gate_proj_6912x2560.bin" \
"gate_proj_6912x2560" \
"/content/benchmark_v6_raw.csv"


FINAL BENCHMARK V6 - AVX2
Layer           : q_proj_2560x2560
Matrix          : 2560 x 2560
Words/row       : 80
Warmups         : 30
Trials          : 60
Inner reps      : 10
Activation sets : 64
SIMD            : AVX2
[1] Reconstructing dense INT8 weights...
[2] Generating activation pool...
[3] Prepacking activation pool...
[4] Running 64-input A/B/C/D correctness check...
    PASS: A == B == C == D for all 64 activation sets
    Aggregate A = 840906
    Aggregate B = 840906
    Aggregate C = 840906
    Aggregate D = 840906
[5] Running warm-up rounds...
[6] Starting measured trials...
Trial  0 [D-C] A=1058.45 us  B=12952.89 us  C=2041.16 us  D=2348.42 us  Pack=85.18 us  D-E2E=2435.15 us  C/D=0.869x  A/D=0.451x  A/D-E2E=0.435x
Trial  1 [C-D] A=1017.35 us  B=13204.93 us  C=2117.05 us  D=2344.76 us  Pack=86.77 us  D-E2E=2462.21 us  C/D=0.903x  A/D=0.434x  A/D-E2E=0.413x
Trial  2 [D-C] A=1003.50 us  B=14278.70 us  C=2017.30 us  D=2383.49 us  Pack=86.82 us  D-E2E=2633.88 us  C/D=0.846x  